# Modelo final y su entrenamiento — V48 / V56 (AGE, `tweet-age-2026-puj`)

**Competencia:** `tweet-age-2026-puj` (Kaggle, `NLP_Course_PUJ`)  
**Programa:** Maestría en Inteligencia Artificial — PLN, Pontificia Universidad Javeriana  
**Equipo:** Grupo 2 - PLN (`carlosgarnica`)  
**Modelo entregado:** **V48 Geometric Product-of-Experts + regla plug-in** — leaderboard público **0.63724**  
**Mejor validación local:** **V56** — CV **0.6189929** (9 modelos)

---

## Alcance de este cuaderno

Este cuaderno contiene **el modelo final y su entrenamiento**: construye la matriz de 279 columnas,
reconstruye el troncal V30 y entrena los componentes neuronales V38–V43 y V53, arma los consensos
V48/V53/V54/V56, aplica la regla *plug-in* y genera la *submission*.

No repite el análisis exploratorio, la historia experimental ni las falsaciones: eso vive en
`kaggle_final_pipeline.ipynb` (reconstrucción sin el modelo final) y en
`99_entrega_consolidado.ipynb` (entrega consolidada). Aquí solo está lo necesario para reproducir el
modelo entregado.

**Hardware:** se recomienda GPU (los seis componentes neuronales son el cuello de botella; en CPU el
cuaderno completo supera el límite de 12 h de Kaggle).


## Protocolo de reproducibilidad


Este notebook es **independiente y determinista**:

1. **Corpus autónomo.** La primera celda localiza los archivos de la competencia (datos adjuntos de Kaggle,
   copias locales o descarga directa vía API oficial). No se clona ningún repositorio.
2. **Cadena embebida.** Todo el código de reconstrucción (representaciones, 14 miembros, features,
   meta-modelo, V20/V30, regla plug-in y redes neuronales) viaja dentro del notebook; cada módulo es un
   porte fiel del script original del proyecto, con su referencia en el docstring.
3. **Protocolo reproducible.** La partición canónica de validación (5 folds agrupados por clústeres de
   near-duplicados) se genera en la propia ejecución con la regla documentada —`StratifiedGroupKFold`,
   `shuffle=True`, semilla 42— y se verifica por propiedades; los pesos de las reglas de decisión son
   constantes legibles del protocolo. Ejecutar dos veces produce exactamente los mismos valores.
4. **Trazabilidad.** Cada experimento reporta su resultado con la fuente documental del proyecto
   (registro de corridas, actas de decisión e informes), y el notebook verifica que lo reconstruido
   coincide con lo documentado.

**Tiempo de ejecución completo:** del orden de 4 horas (entre 3 y 5) en CPU de 8 hilos + GPU (los
componentes neuronales usan GPU si está disponible; sin ella corren en CPU con diferencias del orden
de la precisión de máquina, documentadas en la sección 6.2).

## 0.1 Entorno de ejecución

In [ ]:
# Versiones exactas del protocolo experimental: la particion canonica y los reajustes de
# arboles se generaron con scikit-learn 1.8.0 y LightGBM 4.6.0. Se fijan al arranque para
# que la ejecucion en Kaggle reproduzca las cifras documentadas (no las del entorno local).
import subprocess as _subprocess
import sys as _sys

_subprocess.run([_sys.executable, "-m", "pip", "install", "-q",
                 "scikit-learn==1.8.0", "lightgbm==4.6.0"], check=True)

# Entorno: semillas fijas, estilo de figuras y utilidades basicas del notebook.
import hashlib
import json
import os
import random
import sys
import time
import warnings
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

SEMILLA = 42
os.environ["PYTHONHASHSEED"] = str(SEMILLA)
random.seed(SEMILLA)
np.random.seed(SEMILLA)

plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25

CLASSES = ["13-17", "18-24", "25-34", "35-49", "50-64", "65-xx"]

import sklearn  # noqa: E402  (se importa aqui solo para reportar la version)

print("Entorno listo | Python:", sys.version.split()[0],
      "| numpy:", np.__version__, "| pandas:", pd.__version__,
      "| scikit-learn:", sklearn.__version__)
try:
    import torch
    print("torch:", torch.__version__, "| GPU disponible:", torch.cuda.is_available())
except ImportError:
    print("torch no disponible: los componentes neuronales correran en CPU numpy equivalente")

## 0.2 Resolución autónoma del corpus

Esta sección localiza los tres archivos oficiales (`age_train.csv`, `age_test.csv` y la plantilla de envío).
El orden de búsqueda es: datos adjuntos de la competencia en Kaggle, copias locales habituales y, si no
aparecen, descarga directa con la API oficial (requiere credenciales de Kaggle en el entorno). Todo ocurre
dentro del notebook, sin dependencias externas.

In [ ]:
# Resolucion autonoma del corpus: Kaggle adjunto -> copias locales -> API oficial.
def _buscar_en(base, nombres):
    """Devuelve la primera ruta existente de una lista de nombres dentro de una base."""
    for nombre in nombres:
        ruta = base / nombre
        if ruta.is_file():
            return ruta
    return None


def _descarga_api(destino):
    """Descarga y descomprime el corpus oficial via la API de Kaggle (solo stdlib)."""
    import urllib.request
    import zipfile

    token = os.environ.get("KAGGLE_API_TOKEN", "").strip()
    if not token:
        credencial = Path.home() / ".kaggle" / "kaggle.json"
        if credencial.is_file():
            datos = json.loads(credencial.read_text(encoding="utf-8"))
            token = str(datos.get("token", "")).strip()
    if not token:
        raise RuntimeError(
            "No hay corpus local ni credenciales de Kaggle (KAGGLE_API_TOKEN o ~/.kaggle/kaggle.json)")
    url = "https://www.kaggle.com/api/v1/competitions/data/download-all/tweet-age-2026-puj"
    peticion = urllib.request.Request(url, headers={"Authorization": "Bearer " + token})
    destino.mkdir(parents=True, exist_ok=True)
    archivo = destino / "tweet-age-2026-puj.zip"
    with urllib.request.urlopen(peticion, timeout=300) as respuesta:
        archivo.write_bytes(respuesta.read())
    with zipfile.ZipFile(archivo) as comprimido:
        comprimido.extractall(destino)
    return destino


def resolver_corpus():
    """Devuelve (train, test, submission) localizando o descargando el corpus oficial."""
    bases = []
    if Path("/kaggle/input").exists():
        bases.extend(sorted(p for p in Path("/kaggle/input").iterdir() if p.is_dir()))
    # Kaggle monta las competiciones en /kaggle/input/competitions/<slug>/
    if Path("/kaggle/input/competitions").exists():
        bases.extend(sorted(p for p in Path("/kaggle/input/competitions").iterdir()
                             if p.is_dir()))
    # Kaggle monta las competiciones en /kaggle/input/competitions/<slug>/
    if Path("/kaggle/input/competitions").exists():
        bases.extend(sorted(p for p in Path("/kaggle/input/competitions").iterdir()
                             if p.is_dir()))
    bases.extend([Path("data/raw"), Path("../data/raw"), Path.cwd()])
    for base in bases:
        train = _buscar_en(base, ("age_train.csv", "Age_train.csv"))
        test = _buscar_en(base, ("age_test.csv", "Age_test.csv"))
        envio = _buscar_en(base, ("age_submission.csv", "sample_submission.csv"))
        if train and test:
            return train, test, envio
    print("Corpus no encontrado localmente: descargando via API de Kaggle...")
    base = _descarga_api(Path("data/raw"))
    train = _buscar_en(base, ("age_train.csv",))
    test = _buscar_en(base, ("age_test.csv",))
    envio = _buscar_en(base, ("age_submission.csv",))
    if not (train and test):
        raise FileNotFoundError("La descarga no produjo age_train.csv / age_test.csv")
    return train, test, envio


RUTA_TRAIN, RUTA_TEST, RUTA_ENVIO = resolver_corpus()
print("train :", RUTA_TRAIN)
print("test  :", RUTA_TEST)
print("envio :", RUTA_ENVIO if RUTA_ENVIO else "(se generara desde los ids de test)")

In [ ]:
# Carga del corpus con la convencion exacta del proyecto (separador ';', utf-8-sig, texto limpio).
train = pd.read_csv(RUTA_TRAIN, sep=";", encoding="utf-8-sig")
test = pd.read_csv(RUTA_TEST, sep=";", encoding="utf-8-sig")
train["text"] = train["text"].fillna("").astype(str).str.strip()
test["text"] = test["text"].fillna("").astype(str).str.strip()

assert train.shape == (169336, 3), f"train inesperado: {train.shape}"
assert test.shape == (112892, 2), f"test inesperado: {test.shape}"
assert list(train.columns) == ["id", "text", "age_range"], train.columns

y = train["age_range"].astype(str).to_numpy()
clase = np.array([CLASSES.index(valor) for valor in y])

print(f"train: {len(train):,} filas | test: {len(test):,} filas")
print("distribucion de clases (train):")
for etiqueta, cuenta in train["age_range"].value_counts().reindex(CLASSES).items():
    print(f"  {etiqueta}: {cuenta:,} ({cuenta / len(train) * 100:.2f}%)")

## 0.3 La cadena de reconstrucción (código embebido)

Todo el código de reconstrucción viaja dentro de este notebook: cada módulo es un porte
fiel del script original del proyecto (la referencia exacta aparece en su docstring).
No hay dependencias externas: con el corpus descargado, la cadena se reconstruye entera.
Los módulos se presentan en orden de dependencias.

### Módulo `comun.py`

In [ ]:
"""Utilidades s de la cadena AGE (porte fiel del proyecto KAGGLE_AGE_2026_PUJ).

Regla de oro: cada funcion es un PORTE LITERAL del original citado; no se mejora ni se
reinterpreta nada. ASCII puro. Sin efectos en import.
"""

from __future__ import annotations

import re
from pathlib import Path

import numpy as np
import pandas as pd

CLASSES = ["13-17", "18-24", "25-34", "35-49", "50-64", "65-xx"]
SEED = 42

CHAR_300K = {"analyzer": "char_wb", "ngram_range": (3, 6), "min_df": 5,
             "sublinear_tf": True, "max_features": 300_000, "lowercase": True}
CHAR_700K = {"analyzer": "char_wb", "ngram_range": (3, 6), "min_df": 3,
             "sublinear_tf": True, "max_features": 700_000, "lowercase": True}
MENTION = {"analyzer": "word", "ngram_range": (1, 1), "min_df": 2,
           "sublinear_tf": True, "lowercase": True}
HASHTAG = {"analyzer": "word", "ngram_range": (1, 1), "min_df": 2,
           "sublinear_tf": True, "lowercase": True}
WORD = {"analyzer": "word", "ngram_range": (1, 2), "min_df": 3,
        "sublinear_tf": True, "max_features": 300_000, "lowercase": True}

CLASS_MIDPOINTS = {"13-17": 15.0, "18-24": 21.0, "25-34": 29.5, "35-49": 42.0,
                   "50-64": 57.0, "65-xx": 70.0}

ADJACENT_PAIRS = [("13-17", "18-24"), ("18-24", "25-34"), ("25-34", "35-49"),
                  ("35-49", "50-64"), ("50-64", "65-xx")]

SPACES_RE = re.compile(r"\s+")


def collapse_spaces(series: pd.Series) -> pd.Series:
    """Colapsa espacios repetidos y recorta extremos (src/preprocessing.py:98-100)."""
    return series.fillna("").astype(str).str.replace(SPACES_RE, " ", regex=True).str.strip()


def tokens_of(series: pd.Series, pattern: str) -> pd.Series:
    """Extrae tokens por patron y los une en minusculas (src/pipeline.py:75-78)."""
    return collapse_spaces(series.str.findall(pattern).apply(lambda values: " ".join(values).lower()))


def cargar_train(raiz: Path) -> pd.DataFrame:
    """Lee age_train.csv como data.load_train(): sep ';', utf-8-sig, texto sin padding."""
    marco = pd.read_csv(raiz / "data" / "raw" / "age_train.csv", sep=";", encoding="utf-8-sig")
    marco["text"] = marco["text"].fillna("").astype(str).str.strip()
    return marco


def cargar_test(raiz: Path) -> pd.DataFrame:
    """Lee age_test.csv con la misma convencion."""
    marco = pd.read_csv(raiz / "data" / "raw" / "age_test.csv", sep=";", encoding="utf-8-sig")
    marco["text"] = marco["text"].fillna("").astype(str).str.strip()
    return marco


def cargar_folds(raiz: Path) -> np.ndarray:
    """fold_idx canonico (protocolo congelado): 169.336 valores 0-4.

    En los spikes se lee del carril de artefactos; el notebook embebe este vector.
    """
    return np.load(raiz / "artifacts" / "p33_scores" / "todos" / "fold_idx.npy")


def to_scores(raw: np.ndarray, classes, probabilistic: bool) -> np.ndarray:
    """Reordena columnas al orden canonico y aplica log a las probabilidades (src/pipeline.py:89-94)."""
    orden = [list(classes).index(etiqueta) for etiqueta in CLASSES]
    scores = raw[:, orden]
    return np.log(np.clip(scores, 1e-12, None)) if probabilistic else scores


def softmax_ordinal(continuous: np.ndarray) -> np.ndarray:
    """Convierte la prediccion ordinal continua en distribucion suave (src/pipeline.py:81-86)."""
    midpoints = np.array([CLASS_MIDPOINTS[etiqueta] for etiqueta in CLASSES])
    distances = np.abs(continuous[:, None] - midpoints)
    soft = np.exp(-distances / 5.0)
    return soft / soft.sum(axis=1, keepdims=True)

### Módulo `clusters.py`

In [ ]:
"""Clusters de near-duplicados (porte fiel de src/cv.py:33-229).

Construye los clusters de duplicado exacto o near-duplicado sobre el conjunto completo de textos
(train + test, sin usar etiquetas) con *blocking* por tokens raros y union-find. Es el soporte del
protocolo congelado de folds (D-003). ASCII puro. Sin efectos en import.
"""

from __future__ import annotations

import re
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

#: Longitud minima del texto normalizado para participar en el agrupamiento.
MIN_LEN = 20
#: Umbral de Jaccard sobre conjuntos de tokens para considerar near-duplicado.
JACCARD_THRESHOLD = 0.8
#: Numero de tokens raros usados como clave de bloqueo.
BLOCK_TOKENS = 3
#: Frecuencia documental maxima para que un token sirva como clave de bloqueo.
MAX_DF_FOR_KEY = 200
#: Tope de documentos comparados dentro de un mismo bloque (evita costo cuadratico).
MAX_BLOCK_SIZE = 400

_URL_RE = re.compile(r"https?://\S+|www\.\S+")
_MENTION_RE = re.compile(r"@\w+")
_RT_RE = re.compile(r"^\s*rt\s*:?\s*", flags=re.IGNORECASE)
_NOISE_RE = re.compile("[^0-9a-z\\u00e1\\u00e9\\u00ed\\u00f3\\u00fa\\u00fc\\u00f1 ]+")
_SPACES_RE = re.compile(r"\s+")


def normalize_for_dedup(series: pd.Series) -> pd.Series:
    """Normaliza el texto para detectar duplicados y near-duplicados (src/cv.py:51-73)."""
    text = series.fillna("").astype(str).str.lower()
    text = text.str.replace(_URL_RE, " ", regex=True)
    text = text.str.replace(_MENTION_RE, " ", regex=True)
    text = text.str.replace(_RT_RE, "", regex=True)
    text = text.str.replace(_NOISE_RE, " ", regex=True)
    return text.str.replace(_SPACES_RE, " ", regex=True).str.strip()


def dedup_signature(normalized: pd.Series, chars: int = 60) -> pd.Series:
    """Firma informativa de prefijo (solo metadato, no agrupa; src/cv.py:76-78)."""
    return normalized.str.slice(0, chars)


def jaccard(left: set, right: set) -> float:
    """Similitud de Jaccard entre dos conjuntos de tokens (src/cv.py:81-93)."""
    if not left or not right:
        return 0.0
    if len(left) > len(right):
        left, right = right, left
    if len(left) / len(right) < JACCARD_THRESHOLD and len(left) < JACCARD_THRESHOLD * len(right):
        return 0.0
    intersection = len(left & right)
    if intersection == 0:
        return 0.0
    return intersection / (len(left) + len(right) - intersection)


class _UnionFind:
    """Union-find con compresion de caminos (src/cv.py:96-115)."""

    def __init__(self, size: int) -> None:
        self.parent = list(range(size))

    def find(self, item: int) -> int:
        """Raiz del conjunto de ``item``."""
        root = item
        while self.parent[root] != root:
            root = self.parent[root]
        while self.parent[item] != root:
            self.parent[item], item = root, self.parent[item]
        return root

    def union(self, left: int, right: int) -> None:
        """Une los conjuntos de ``left`` y ``right``."""
        root_left, root_right = self.find(left), self.find(right)
        if root_left != root_right:
            self.parent[root_right] = root_left


def construir_clusters(textos: pd.Series, min_len: int = MIN_LEN,
                       threshold: float = JACCARD_THRESHOLD,
                       block_tokens: int = BLOCK_TOKENS,
                       max_df_for_key: int = MAX_DF_FOR_KEY,
                       max_block_size: int = MAX_BLOCK_SIZE,
                       pair_budget: int = 20_000_000,
                       stats: dict | None = None) -> pd.DataFrame:
    """Agrupa textos en clusters de duplicado exacto o near-duplicado (src/cv.py:118-229).

    Dos filas caen en el mismo cluster si su texto normalizado es identico o si la similitud de
    Jaccard entre sus conjuntos de tokens es >= ``threshold``. El blocking por los ``block_tokens``
    tokens mas raros evita el costo cuadratico.
    """
    normalized = normalize_for_dedup(textos)
    signature = dedup_signature(normalized)
    normalized_array = normalized.to_numpy()
    valid = normalized.str.len().to_numpy() >= min_len
    union = _UnionFind(len(normalized))

    exact_groups: dict = defaultdict(list)
    for position, key in enumerate(normalized_array):
        if valid[position]:
            exact_groups[key].append(position)
    for positions in exact_groups.values():
        anchor = positions[0]
        for position in positions[1:]:
            union.union(anchor, position)

    token_sets = [frozenset(tokens.split()) for tokens in normalized_array]
    document_frequency: Counter = Counter()
    for tokens in token_sets:
        document_frequency.update(tokens)

    blocks: dict = defaultdict(list)
    for position, tokens in enumerate(token_sets):
        if not valid[position]:
            continue
        rare = sorted((document_frequency[token], token) for token in tokens
                      if document_frequency[token] <= max_df_for_key)
        for _, token in rare[:block_tokens]:
            blocks[token].append(position)

    comparisons = 0
    budget_exhausted = False
    for documents in sorted(blocks.values(), key=len):
        if len(documents) < 2:
            continue
        if len(documents) > max_block_size:
            documents = documents[:max_block_size]
        for left_index in range(len(documents)):
            left = documents[left_index]
            left_tokens = token_sets[left]
            if len(left_tokens) < 2:
                continue
            for right_index in range(left_index + 1, len(documents)):
                right = documents[right_index]
                if union.find(left) == union.find(right):
                    continue
                comparisons += 1
                if comparisons > pair_budget:
                    budget_exhausted = True
                    break
                if jaccard(left_tokens, token_sets[right]) >= threshold:
                    union.union(left, right)
            if budget_exhausted:
                break
        if budget_exhausted:
            break

    roots = np.array([union.find(index) for index in range(len(normalized))])
    _, cluster = np.unique(roots, return_inverse=True)

    if stats is not None:
        stats.update({"blocks": len(blocks), "comparisons": comparisons,
                      "pair_budget_exhausted": budget_exhausted, "threshold": threshold,
                      "max_df_for_key": max_df_for_key, "block_tokens": block_tokens})
    return pd.DataFrame({"norm": normalized_array, "sig": signature.to_numpy(),
                         "cluster": cluster})

### Módulo `dense.py`

In [ ]:
"""Textometria y features estilometricas a nivel de documento (porte literal).

Porte de ``src/textometry.py`` (lineas 1-173) a ``repro/cadena/dense.py``. La funcion
publica ``extraer_textometria`` reproduce exactamente el original: 38 columnas en el
orden ``TEXTOMETRY_FEATURES + STYLE_FEATURES``. Las regex ``_EMOJI_RE`` y ``_URL_RE``
se copian de ``src/preprocessing.py:18,25-32`` como constantes locales del modulo.

Todas las features son deterministas y locales al documento: no aprenden parametros del
corpus, por lo que pueden calcularse sobre train y test sin riesgo de leakage.

Sin dependencias del proyecto (solo stdlib + numpy/pandas), ASCII puro y sin efectos en
import.
"""

from __future__ import annotations

import re

import numpy as np
import pandas as pd

# Emoticonos textuales clasicos (con fronteras de palabra para evitar falsos positivos).
# src/textometry.py:24-26
EMOTICON_RE = re.compile(
    r"(?<![\w:/])(?:[:;=8][-o*']?[\)\]\(\[dDpP/:}{@|]|x[dDpP](?!\w)|XD(?!\w)|<3)(?![\w])")
# Palabras con elongacion (3 o mas repeticiones de un caracter). src/textometry.py:27-28
ELONGATED_RE = re.compile(r"\b\w*(\w)\1{2,}\w*\b", flags=re.UNICODE)
# Secuencias de puntuacion repetida (!!, ??, ..., !?, etc.). src/textometry.py:29-30
MULTI_PUNCT_RE = re.compile(r"[!?.,;:]{2,}")
# Signos de apertura del espanol. src/textometry.py:31-32
OPEN_PUNCT_RE = re.compile(r"[\u00a1\u00bf]")
# Palabras completamente en mayusculas (2+ letras). src/textometry.py:33-34
ALL_CAPS_RE = re.compile(r"\b[A-Z\u00c1\u00c9\u00cd\u00d3\u00da\u00dc\u00d1]{2,}\b")
# Separador de frases. src/textometry.py:35-36
SENTENCE_SPLIT_RE = re.compile(r"[.!?\u2026]+|\n+")

# Lista fija de stopwords en espanol (recurso lexico cerrado, sin parametros aprendidos).
# src/textometry.py:38-48
SPANISH_STOPWORDS = frozenset(["a", "al", "algo", "algunas", "alguno", "algunos", "ambos", "ante", "antes", "aquel", "aquella", "aquellas", "aquellos", "aqui", "asi", "aun", "aunque", "bien", "cada", "casi", "como", "con", "contra", "cual", "cuales", "cuando", "cuanto", "cuenta", "da", "de", "del", "desde", "donde", "dos", "el", "ella", "ellas", "ello", "ellos", "en", "entonces", "entre", "era", "eran", "es", "esa", "esas", "ese", "eso", "esos", "esta", "estaba", "estado", "estan", "estar", "estas", "este", "esto", "estos", "fue", "fueron", "ha", "habia", "han", "hasta", "hay", "la", "las", "le", "les", "lo", "los", "mas", "me", "mi", "mientras", "mis", "mucho", "muchos", "muy", "nada", "ni", "no", "nos", "nosotros", "nuestra", "nuestro", "nuestros", "o", "os", "otra", "otras", "otro", "otros", "para", "pero", "poco", "por", "porque", "que", "quien", "quienes", "se", "sea", "segun", "ser", "si", "sin", "sobre", "solo", "son", "su", "sus", "tal", "tambien", "tanto", "te", "tener", "tiene", "tienen", "todo", "todos", "tu", "tus", "un", "una", "uno", "unos", "usted", "ustedes", "va", "vamos", "van", "varias", "varios", "ver", "y", "ya", "yo"])

# Familias de features (orden exacto del original). src/textometry.py:50-62
TEXTOMETRY_FEATURES = [
    "n_chars", "n_words", "n_unique_words", "ttr", "cttr", "hapax_ratio", "lexical_entropy",
    "avg_word_len", "max_word_len", "std_word_len", "n_sentences", "avg_sentence_len_words",
    "stopword_ratio", "content_ratio", "long_word_ratio",
]
STYLE_FEATURES = [
    "upper_ratio", "digit_ratio", "punct_ratio", "n_excl", "n_quest", "n_open_punct",
    "multi_punct_count", "elongated_count", "all_caps_count", "has_all_caps",
    "n_emoji", "emoji_diversity", "has_emoji", "n_emoticon", "has_emoticon",
    "n_hashtag", "n_mention", "n_url", "is_rt", "starts_mention", "n_repeated_char_runs",
    "punct_per_word", "unique_char_ratio",
]

# Patrones compartidos copiados de src/preprocessing.py (_URL_RE en la linea 18;
# _EMOJI_RE en las lineas 25-32).
_URL_RE = re.compile(r"https?://\S+|www\.\S+")
_EMOJI_RE = re.compile(
    "["
    "\U0001F300-\U0001F5FF\U0001F600-\U0001F64F\U0001F680-\U0001F6FF"
    "\U0001F700-\U0001F77F\U0001F780-\U0001F7FF\U0001F800-\U0001F8FF"
    "\U0001F900-\U0001F9FF\U0001FA00-\U0001FAFF\U0001F1E6-\U0001F1FF"
    "\U00002600-\U000027BF\U00002B00-\U00002BFF\U0000FE0F"
    "]"
)


def _entropy(counts: np.ndarray) -> float:
    """Entropia de Shannon (base 2) de un vector de conteos (src/textometry.py:65-71)."""
    total = counts.sum()
    if total <= 0:
        return 0.0
    probabilities = counts[counts > 0] / total
    return float(-(probabilities * np.log2(probabilities)).sum())


def extraer_textometria(df: pd.DataFrame) -> pd.DataFrame:
    """Calcula las features textometricas y estilometricas de cada documento.

    Porte literal de ``src/textometry.py:83-173`` (helper ``_entropy`` en 65-71). El
    original acepta ``text_col: str = "text"``; en este porte la columna es fija
    ``"text"``, segun la interfaz de ``repro/cadena``. La entrada debe traer el texto
    ya limpio (strip) y la salida conserva el indice de ``df``, con 38 columnas en el
    orden ``TEXTOMETRY_FEATURES + STYLE_FEATURES``.

    Parameters
    ----------
    df : pandas.DataFrame
        DataFrame con la columna ``text`` (ya sin padding).

    Returns
    -------
    pandas.DataFrame
        Una fila por documento, con las columnas de ``TEXTOMETRY_FEATURES`` y
        ``STYLE_FEATURES``.
    """
    text = df["text"].fillna("").astype(str)
    lowered = text.str.lower()
    words = lowered.str.split()
    # Los emoticonos se cuentan sobre el texto sin URLs: "http://" contiene ":/" y
    # contaminaba el conteo (detectado en el EDA de la Fase 2).
    text_without_urls = text.str.replace(_URL_RE, " ", regex=True)

    n_chars = text.str.len()
    n_words = words.str.len()
    n_unique = words.apply(lambda tokens: len(set(tokens)) if tokens else 0)
    n_hapax = words.apply(
        lambda tokens: int(sum(1 for count in pd.Series(tokens).value_counts() if count == 1))
        if tokens else 0)
    entropy = words.apply(lambda tokens: _entropy(np.unique(tokens, return_counts=True)[1])
                          if tokens else 0.0)
    word_len_mean = words.apply(lambda tokens: float(np.mean([len(t) for t in tokens]))
                                if tokens else 0.0)
    word_len_max = words.apply(lambda tokens: int(max((len(t) for t in tokens), default=0)))
    word_len_std = words.apply(lambda tokens: float(np.std([len(t) for t in tokens]))
                               if tokens else 0.0)

    letters = text.str.count(r"[A-Za-z\u00c1\u00c9\u00cd\u00d3\u00da\u00dc\u00d1\u00e1\u00e9\u00ed\u00f3\u00fa\u00fc\u00f1]")
    upper = text.str.count(r"[A-Z\u00c1\u00c9\u00cd\u00d3\u00da\u00dc\u00d1]")
    digits = text.str.count(r"\d")
    punct = text.str.count(r"[^\w\s]")
    sentences = text.apply(lambda value: len([s for s in SENTENCE_SPLIT_RE.split(value) if s.strip()]))
    stopword_hits = words.apply(lambda tokens: sum(1 for token in tokens
                                                   if token.strip(".,;:!?\u00a1\u00bf\"'()") in SPANISH_STOPWORDS))
    long_words = words.apply(lambda tokens: sum(1 for token in tokens if len(token) >= 8))

    features = pd.DataFrame({
        # --- textometria ---
        "n_chars": n_chars,
        "n_words": n_words,
        "n_unique_words": n_unique,
        "ttr": (n_unique / n_words.replace(0, np.nan)).fillna(0.0),
        "cttr": (n_unique / np.sqrt(2 * n_words.replace(0, np.nan))).fillna(0.0),
        "hapax_ratio": (n_hapax / n_words.replace(0, np.nan)).fillna(0.0),
        "lexical_entropy": entropy,
        "avg_word_len": word_len_mean,
        "max_word_len": word_len_max,
        "std_word_len": word_len_std,
        "n_sentences": sentences,
        "avg_sentence_len_words": (n_words / sentences.replace(0, np.nan)).fillna(0.0),
        "stopword_ratio": (stopword_hits / n_words.replace(0, np.nan)).fillna(0.0),
        "content_ratio": (1 - (stopword_hits / n_words.replace(0, np.nan))).fillna(0.0),
        "long_word_ratio": (long_words / n_words.replace(0, np.nan)).fillna(0.0),
        # --- estilo ---
        "upper_ratio": (upper / letters.replace(0, np.nan)).fillna(0.0),
        "digit_ratio": (digits / n_chars.replace(0, np.nan)).fillna(0.0),
        "punct_ratio": (punct / n_chars.replace(0, np.nan)).fillna(0.0),
        "n_excl": text.str.count("!"),
        "n_quest": text.str.count(r"\?"),
        "n_open_punct": text.str.count(OPEN_PUNCT_RE.pattern),
        "multi_punct_count": text.str.count(MULTI_PUNCT_RE.pattern),
        "elongated_count": text.str.count(ELONGATED_RE.pattern),
        "all_caps_count": text.str.count(ALL_CAPS_RE.pattern),
        "has_all_caps": text.str.contains(ALL_CAPS_RE.pattern).astype(int),
        "n_emoji": text.str.count(_EMOJI_RE.pattern),
        "emoji_diversity": text.apply(
            lambda value: len(set(_EMOJI_RE.findall(value))) if value else 0),
        "has_emoji": text.str.contains(_EMOJI_RE.pattern).astype(int),
        "n_emoticon": text_without_urls.str.count(EMOTICON_RE.pattern),
        "has_emoticon": text_without_urls.str.contains(EMOTICON_RE.pattern).astype(int),
        "n_hashtag": text.str.count("#"),
        "n_mention": text.str.count("@"),
        "n_url": text.str.count(r"https?://|www\."),
        "is_rt": text.str.match(r"^\s*RT\b", case=False).astype(int),
        "starts_mention": text.str.match(r"^\s*@").astype(int),
        "n_repeated_char_runs": text.str.count(r"(\w)\1{1,}"),
        "punct_per_word": (punct / n_words.replace(0, np.nan)).fillna(0.0),
        "unique_char_ratio": (text.apply(lambda value: len(set(value)))
                              / n_chars.replace(0, np.nan)).fillna(0.0),
    })
    return features

### Módulo `structured.py`

In [ ]:
"""Features estructuradas de segunda generacion (87 columnas) - porte literal.

Porte fiel de las seis familias locales de ``src/advanced_features.py``, en el orden de
ensamblado de ``scripts/advanced_02_features.py:38-45,94-104`` (``LOCAL_BUILDERS`` /
``build_local_features``) y de ``scripts/phase3b_recorregido.py:54-69`` (``build_local``):

    1. ``distribution_features``       (src/advanced_features.py:75-111)  -> 21 cols wlen_*/prop_*
    2. ``char_distribution_features``  (src/advanced_features.py:117-153) -> 10 cols char_*
    3. ``lexical_richness_features``   (src/advanced_features.py:159-189) -> 9 cols de riqueza lexica
    4. ``word_shape_features``         (src/advanced_features.py:195-226) -> 10 cols shape_*/repeat_*
    5. ``position_features``           (src/advanced_features.py:232-267) -> 20 cols pos_*/first_*/last_*
    6. ``structure_entropy_features``  (src/advanced_features.py:273-306) -> 17 cols punct_*/entropia

mas la correccion vigente de ``scripts/phase3b_recorregido.py:42-51`` (``ALNUM_RE`` y recalculo
de ``shape_alphanumeric_ratio``), que es la version que produjo el artefacto
``features/age_structured_features.parquet``.

Orden y conteo verificados contra ese parquet: 21 + 10 + 9 + 10 + 20 + 17 = 87 columnas
(73 float64 + 14 int64). Nota: el plan cita 19/18 para las familias 5/6, pero el codigo literal
y el artefacto confirman 20/17 con el mismo total de 87.

Entrada: DataFrame con columna ``text`` limpia (se procesa en orden de fila). Salida: DataFrame
de 87 columnas con el mismo indice que la entrada. Determinista, sin RNG. Solo
numpy/pandas/scipy/stdlib; sin efectos en import. ASCII puro: los caracteres acentuados de las
regex se escriben con escapes unicode (``\\uXXXX``) para mantener el fuente en ASCII sin
cambiar el patron compilado.
"""

from __future__ import annotations

import re
import zlib
from collections import Counter

import numpy as np
import pandas as pd
from scipy import stats

# --- Patrones (src/advanced_features.py:28-37) ------------------------------------------
MENTION_RE = re.compile(r"@(\w+)")
HASHTAG_RE = re.compile(r"#(\w+)")
URL_RE = re.compile(r"https?://\S+|www\.\S+")
#: Alternativa de palabra con vocales acentuadas (mismo patron que el original; el fuente usa
#: escapes unicode para quedar en ASCII).
WORD_RE = re.compile(r"[A-Za-z"
                     "\u00c1\u00c9\u00cd\u00d3\u00da\u00dc\u00d1"
                     "\u00e1\u00e9\u00ed\u00f3\u00fa\u00fc\u00f1"
                     r"0-9]+")
#: Tokenizador propio de tweets. La alternativa de palabra **debe** aceptar mayusculas: si no,
#: las palabras capitalizadas o en mayusculas se fragmentan y las features de forma quedan en
#: cero (bug detectado en la auditoria de alineacion de la Fase 3).
TOKEN_RE = re.compile(r"@\w+|#\w+|https?://\S+|[A-Za-z"
                      "\u00c1\u00c9\u00cd\u00d3\u00da\u00dc\u00d1"
                      "\u00e1\u00e9\u00ed\u00f3\u00fa\u00fc\u00f1"
                      r"]+|\d+|[^\s\w]", flags=re.UNICODE)
#: Tokens que contienen letras **y** digitos (p. ej. "abc123"), que el tokenizador separa
#: (``scripts/phase3b_recorregido.py:42``).
ALNUM_RE = re.compile(r"\b(?=\w*\d)(?=\w*[A-Za-z])\w+\b")


def _safe_entropy(counts: np.ndarray) -> float:
    """Entropia de Shannon (base 2) segura para vectores vacios (src/advanced_features.py:56-62)."""
    total = counts.sum()
    if total <= 0:
        return 0.0
    probabilities = counts[counts > 0] / total
    return float(-(probabilities * np.log2(probabilities)).sum())


def _percentiles(values: np.ndarray, points: tuple[int, ...] = (10, 25, 50, 75, 90)) -> dict:
    """Percentiles de un vector (0 si esta vacio) (src/advanced_features.py:65-69)."""
    if values.size == 0:
        return {f"p{p}": 0.0 for p in points}
    return {f"p{p}": float(np.percentile(values, p)) for p in points}


# --------------------------------------------------------------------------------------
# Familias A, C - distribucion de la longitud de palabra (src/advanced_features.py:75-111)
# --------------------------------------------------------------------------------------
def distribution_features(texts: pd.Series) -> pd.DataFrame:
    """Familia A/C: distribucion interna de la longitud de palabra y de los tokens."""
    tokens = texts.fillna("").str.lower().str.findall(WORD_RE.pattern)
    rows: list[dict] = []
    for token_list in tokens:
        lengths = np.array([len(token) for token in token_list], dtype=float)
        if lengths.size == 0:
            rows.append({"wlen_mean": 0.0, "wlen_median": 0.0, "wlen_std": 0.0, "wlen_cv": 0.0,
                         "wlen_skew": 0.0, "wlen_kurtosis": 0.0, "wlen_iqr": 0.0, "wlen_mad": 0.0,
                         "wlen_range": 0.0, "wlen_p10": 0.0, "wlen_p25": 0.0, "wlen_p50": 0.0,
                         "wlen_p75": 0.0, "wlen_p90": 0.0, "wlen_tail_ratio": 0.0,
                         "wlen_head_ratio": 0.0, "prop_len1": 0.0, "prop_len2": 0.0,
                         "prop_len3": 0.0, "prop_len_ge8": 0.0, "prop_len_ge10": 0.0})
            continue
        percentiles = {f"wlen_{key}": value for key, value in _percentiles(lengths).items()}
        mean = float(lengths.mean())
        std = float(lengths.std())
        has_variance = lengths.std() > 0
        rows.append({
            "wlen_mean": mean, "wlen_median": float(np.median(lengths)), "wlen_std": std,
            "wlen_cv": std / mean if mean > 0 else 0.0,
            # Asimetria y curtosis requieren varianza no nula (si todos los tokens miden igual,
            # la forma no esta definida y se reporta 0 en lugar de NaN).
            "wlen_skew": float(stats.skew(lengths)) if lengths.size > 2 and has_variance else 0.0,
            "wlen_kurtosis": (float(stats.kurtosis(lengths))
                              if lengths.size > 3 and has_variance else 0.0),
            "wlen_iqr": float(np.percentile(lengths, 75) - np.percentile(lengths, 25)),
            "wlen_mad": float(np.mean(np.abs(lengths - mean))),
            "wlen_range": float(lengths.max() - lengths.min()),
            **percentiles,
            "wlen_tail_ratio": float((lengths >= np.percentile(lengths, 90)).mean()),
            "wlen_head_ratio": float((lengths <= np.percentile(lengths, 10)).mean()),
            "prop_len1": float((lengths == 1).mean()), "prop_len2": float((lengths == 2).mean()),
            "prop_len3": float((lengths == 3).mean()), "prop_len_ge8": float((lengths >= 8).mean()),
            "prop_len_ge10": float((lengths >= 10).mean()),
        })
    return pd.DataFrame(rows)


# --------------------------------------------------------------------------------------
# Familia B, R - distribucion de caracteres y entropias (src/advanced_features.py:117-153)
# --------------------------------------------------------------------------------------
def char_distribution_features(texts: pd.Series) -> pd.DataFrame:
    """Familia B/R: composicion y concentracion del alfabeto usado en el tweet."""
    clean = texts.fillna("").astype(str)
    lowered = clean.str.lower()
    total = clean.str.len().replace(0, np.nan)
    vowels = lowered.str.count(r"[aeiou" "\u00e1\u00e9\u00ed\u00f3\u00fa\u00fc" r"]")
    consonants = lowered.str.count(r"[bcdfghjklmnpqrstvwxyz" "\u00f1" r"]")
    letters = vowels + consonants
    rows = pd.DataFrame({
        "char_vowel_ratio": (vowels / letters.replace(0, np.nan)).fillna(0.0),
        "char_consonant_ratio": (consonants / letters.replace(0, np.nan)).fillna(0.0),
        "char_alpha_ratio": (letters / total).fillna(0.0),
        "char_space_ratio": (clean.str.count(r"\s") / total).fillna(0.0),
        "char_symbol_ratio": (clean.str.count(r"[^\w\s]") / total).fillna(0.0),
    })
    character_entropy, simpson, hhi, dominant, unique_ratio = [], [], [], [], []
    for value in clean:
        if not value:
            character_entropy.append(0.0)
            simpson.append(0.0)
            hhi.append(0.0)
            dominant.append(0.0)
            unique_ratio.append(0.0)
            continue
        counts = np.array(list(Counter(value).values()), dtype=float)
        probabilities = counts / counts.sum()
        character_entropy.append(_safe_entropy(counts))
        simpson.append(float(1 - (probabilities ** 2).sum()))
        hhi.append(float((probabilities ** 2).sum()))
        dominant.append(float(probabilities.max()))
        unique_ratio.append(len(set(value)) / len(value))
    rows["char_entropy"] = character_entropy
    rows["char_simpson"] = simpson
    rows["char_hhi"] = hhi
    rows["char_dominant_ratio"] = dominant
    rows["char_unique_ratio"] = unique_ratio
    return rows


# --------------------------------------------------------------------------------------
# Familia D - riqueza lexica avanzada (src/advanced_features.py:159-189)
# --------------------------------------------------------------------------------------
def lexical_richness_features(texts: pd.Series) -> pd.DataFrame:
    """Familia D: diversidad lexica estable para textos cortos (MATTR, Yule's K, Simpson)."""
    tokens = texts.fillna("").str.lower().str.findall(WORD_RE.pattern)
    rows: list[dict] = []
    for token_list in tokens:
        n = len(token_list)
        if n == 0:
            rows.append({"mattr_25": 0.0, "root_ttr": 0.0, "log_ttr": 0.0, "maas": 0.0,
                         "yule_k": 0.0, "simpson_d": 0.0, "dis_legomena_ratio": 0.0,
                         "repeated_token_ratio": 0.0, "top_token_concentration": 0.0})
            continue
        counts = Counter(token_list)
        window = min(25, n)
        windows = [token_list[start:start + window] for start in range(0, max(1, n - window + 1))]
        mattr = float(np.mean([len(set(w)) / len(w) for w in windows if w]))
        types = len(counts)
        frequencies = np.array(list(counts.values()), dtype=float)
        yule_k = float(1e4 * ((frequencies ** 2).sum() - n) / (n ** 2)) if n > 1 else 0.0
        rows.append({
            "mattr_25": mattr,
            "root_ttr": types / np.sqrt(n),
            "log_ttr": np.log(types) / np.log(n) if n > 1 and types > 1 else 0.0,
            "maas": (np.log(n) - np.log(types)) / (np.log(n) ** 2) if n > 1 and types > 1 else 0.0,
            "yule_k": yule_k,
            "simpson_d": float((frequencies * (frequencies - 1)).sum() / (n * (n - 1)))
            if n > 1 else 0.0,
            "dis_legomena_ratio": float((frequencies == 2).sum() / types) if types else 0.0,
            "repeated_token_ratio": float((n - types) / n),
            "top_token_concentration": float(frequencies.max() / n),
        })
    return pd.DataFrame(rows)


# --------------------------------------------------------------------------------------
# Familia H, T - forma de palabra y repeticion (src/advanced_features.py:195-226)
# --------------------------------------------------------------------------------------
def word_shape_features(texts: pd.Series) -> pd.DataFrame:
    """Familia H/T: patrones de escritura de las palabras y repeticion interna."""
    tokens = texts.fillna("").str.findall(TOKEN_RE.pattern)
    rows: list[dict] = []
    for token_list in tokens:
        # ``WORD_RE`` acepta mayusculas: si no, las features de forma de palabra quedarian en
        # cero (bug detectado en la auditoria de alineacion de la Fase 3).
        words = [token for token in token_list if WORD_RE.fullmatch(token)]
        n = len(words) or 1
        all_caps = sum(1 for token in words if token.isupper() and len(token) > 1)
        capitalized = sum(1 for token in words if token[:1].isupper() and token[1:].islower())
        numeric = sum(1 for token in words if token.isdigit())
        alphanumeric = sum(1 for token in words if any(c.isdigit() for c in token)
                           and any(c.isalpha() for c in token))
        repeated = sum(1 for token in words if re.search(r"(\w)\1{1,}", token))
        bigrams = list(zip(token_list, token_list[1:])) if len(token_list) > 1 else []
        trigrams = (list(zip(token_list, token_list[1:], token_list[2:]))
                    if len(token_list) > 2 else [])
        rows.append({
            "shape_allcaps_ratio": all_caps / n,
            "shape_capitalized_ratio": capitalized / n,
            "shape_lower_ratio": sum(1 for token in words if token.islower()) / n,
            "shape_numeric_ratio": numeric / n,
            "shape_alphanumeric_ratio": alphanumeric / n,
            "shape_repeated_letter_token_ratio": repeated / n,
            "repeat_token_ratio": float(1 - len(set(token_list)) / len(token_list))
            if token_list else 0.0,
            "repeat_bigram_ratio": float(1 - len(set(bigrams)) / len(bigrams)) if bigrams else 0.0,
            "repeat_trigram_ratio": float(1 - len(set(trigrams)) / len(trigrams)) if trigrams else 0.0,
            "adjacent_repeat_ratio": float(np.mean([a == b for a, b in bigrams])) if bigrams else 0.0,
        })
    return pd.DataFrame(rows)


# --------------------------------------------------------------------------------------
# Familia I, M, U - posicion de elementos, URLs y posicion lexica
# (src/advanced_features.py:232-267)
# --------------------------------------------------------------------------------------
def position_features(texts: pd.Series) -> pd.DataFrame:
    """Familias I/M/U: posicion normalizada de entidades y de la primera/ultima palabra."""
    clean = texts.fillna("").astype(str)
    rows: list[dict] = []
    for value in clean:
        length = max(len(value), 1)
        tokens = TOKEN_RE.findall(value)

        def positions(pattern: re.Pattern) -> tuple[float, float, float]:
            matches = [m.start() / length for m in pattern.finditer(value)]
            if not matches:
                return 0.0, 0.0, 0.0
            return matches[0], matches[-1], float(np.mean(matches))

        mention_first, mention_last, mention_mean = positions(MENTION_RE)
        hashtag_first, hashtag_last, hashtag_mean = positions(HASHTAG_RE)
        url_first, url_last, url_mean = positions(URL_RE)
        first_word = tokens[0] if tokens else ""
        last_word = tokens[-1] if tokens else ""
        rows.append({
            "pos_mention_first": mention_first, "pos_mention_last": mention_last,
            "pos_mention_mean": mention_mean, "pos_hashtag_first": hashtag_first,
            "pos_hashtag_last": hashtag_last, "pos_hashtag_mean": hashtag_mean,
            "pos_url_first": url_first, "pos_url_last": url_last, "pos_url_mean": url_mean,
            "first_token_is_mention": int(bool(tokens) and tokens[0].startswith("@")),
            "last_token_is_mention": int(bool(tokens) and tokens[-1].startswith("@")),
            "first_token_is_hashtag": int(bool(tokens) and tokens[0].startswith("#")),
            "last_token_is_hashtag": int(bool(tokens) and tokens[-1].startswith("#")),
            "first_token_is_url": int(bool(tokens) and tokens[0].startswith("http")),
            "last_token_is_url": int(bool(tokens) and tokens[-1].startswith("http")),
            "first_word_len": len(first_word), "last_word_len": len(last_word),
            "first_word_is_word": int(bool(WORD_RE.fullmatch(first_word))),
            "last_word_is_word": int(bool(WORD_RE.fullmatch(last_word))),
            "first_word_is_upper": int(first_word[:1].isupper() and len(first_word) > 1),
        })
    return pd.DataFrame(rows)


# --------------------------------------------------------------------------------------
# Familia N, R, S - estructura, entropia y compresibilidad (src/advanced_features.py:273-306)
# --------------------------------------------------------------------------------------
def structure_entropy_features(texts: pd.Series) -> pd.DataFrame:
    """Familias N/R/S: puntuacion, entropias por tipo y ratio de compresion."""
    clean = texts.fillna("").astype(str)
    total = clean.str.len().replace(0, np.nan)
    punctuation = {name: clean.str.count(pattern) for name, pattern in
                   (("comma", ","), ("period", r"\."), ("colon", ":"), ("semicolon", ";"),
                    ("parenthesis", r"[()]"), ("quote", r"[\"']"), ("dash", "[-\u2013\u2014]"),
                    ("excl", "!"), ("quest", r"\?"))}
    rows = pd.DataFrame({
        f"punct_{name}_density": (count / total).fillna(0.0)
        for name, count in punctuation.items()
    })
    rows["punct_type_diversity"] = clean.apply(
        lambda value: len(set(re.findall(r"[^\w\s]", value))))
    rows["punct_transitions"] = clean.apply(
        lambda value: len(re.findall(r"[^\w\s]{2,}", value)))
    rows["token_entropy"] = clean.apply(
        lambda value: _safe_entropy(np.array(list(Counter(TOKEN_RE.findall(value)).values()),
                                             dtype=float)) if value else 0.0)
    rows["punct_entropy"] = clean.apply(
        lambda value: _safe_entropy(np.array(list(Counter(re.findall(r"[^\w\s]", value)).values()),
                                             dtype=float)) if value else 0.0)
    rows["entity_count"] = clean.apply(lambda value: len(MENTION_RE.findall(value))
                                       + len(HASHTAG_RE.findall(value)))
    rows["entity_entropy"] = clean.apply(
        lambda value: _safe_entropy(np.array(list(Counter(
            [m.lower() for m in MENTION_RE.findall(value)]
            + [h.lower() for h in HASHTAG_RE.findall(value)]).values()), dtype=float))
        if value else 0.0)
    rows["compression_ratio"] = clean.apply(
        lambda value: len(zlib.compress(value.encode("utf-8"), 6)) / max(len(value), 1))
    rows["compression_gain"] = clean.apply(
        lambda value: 1 - len(zlib.compress(value.encode("utf-8"), 6)) / max(len(value), 1))
    return rows


# --------------------------------------------------------------------------------------
# Correccion vigente (scripts/phase3b_recorregido.py:42-51)
# --------------------------------------------------------------------------------------
def add_alphanumeric_ratio(frame: pd.DataFrame, texts: pd.Series) -> pd.DataFrame:
    """Recalcula ``shape_alphanumeric_ratio`` sobre el texto crudo (phase3b:45-51).

    El tokenizador separa letras y digitos, asi que el conteo interno de ``word_shape_features``
    es siempre cero; aqui se cuentan los tokens mixtos con ``ALNUM_RE`` y se dividen por el
    numero de tokens de ``TOKEN_RE`` (denominador con piso 1).
    """
    counts = texts.fillna("").astype(str).apply(lambda value: len(ALNUM_RE.findall(value)))
    words = texts.fillna("").astype(str).str.findall(TOKEN_RE.pattern).apply(len).replace(0, 1)
    frame = frame.copy()
    frame["shape_alphanumeric_ratio"] = (counts / words).to_numpy()
    return frame


# --------------------------------------------------------------------------------------
# Ensamblado (scripts/advanced_02_features.py:94-104 / scripts/phase3b_recorregido.py:54-69)
# --------------------------------------------------------------------------------------
def extraer_estructuradas(df: pd.DataFrame) -> pd.DataFrame:
    """Calcula las 87 features estructuradas de segunda generacion.

    Ensambla las seis familias en el orden original (``scripts/advanced_02_features.py:38-45,
    94-104``; ``scripts/phase3b_recorregido.py:54-69``) y aplica la correccion vigente de
    ``scripts/phase3b_recorregido.py:42-51`` (recalculo de ``shape_alphanumeric_ratio`` con
    ``ALNUM_RE`` sobre el texto crudo).

    Parameters
    ----------
    df : pd.DataFrame
        Marco con columna ``text`` (texto limpio, sin padding).

    Returns
    -------
    pd.DataFrame
        87 columnas (73 float64 + 14 int64) con el mismo indice que ``df``.
    """
    # Los textos se procesan en orden de fila con indice neutro (los builders construyen sus
    # marcos de forma posicional); al final se restituye el indice de la entrada.
    textos = df["text"].reset_index(drop=True)
    builders = (("A_distribucion", distribution_features),
                ("B_char", char_distribution_features),
                ("D_riqueza", lexical_richness_features),
                ("HT_forma", word_shape_features),
                ("IMU_posicion", position_features),
                ("NRS_estructura", structure_entropy_features))
    frames = [builder(textos) for _name, builder in builders]
    combined = pd.concat(frames, axis=1)
    combined = add_alphanumeric_ratio(combined, textos)
    combined.index = df.index
    return combined

### Módulo `supervised.py`

In [ ]:
"""Features supervisadas in-fold de la cadena AGE (porte fiel del proyecto original).

Reproduce los 5 bloques supervisados con el orden EXACTO de columnas (73):
mention 17 + hashtag 17 + rt_source 17 + rarity 12 + lexicon 10.

- Perfil de clase por entidad: P(clase|entidad) con suavizado ``smoothing=20.0`` sobre el
  prior del train recibido; sin entidad conocida ``[prior, prior, 0, 0, 0, 0, 0]``
  (src/advanced_features.py:326-393, linea 377).
- Rareza lexica: frecuencias solo del train recibido; OOV con log10 centinela ``-6.0``
  (src/advanced_features.py:396-434, linea 419).
- Lexico por clase: log-odds de Monroe et al. 2008 con ``alpha0=100.0, min_count=5,
  top_k=3000`` (src/advanced_features.py:437-490).
- Regexes ``MENTION_RE``/``HASHTAG_RE``/``RT_SOURCE_RE``/``WORD_RE`` de
  src/advanced_features.py:28-32.

Lado OOF (LOFO estricto): scripts/advanced_02_features.py:205-234; cada fold ajusta los
5 bloques con el train de los otros 4 y transforma su validacion, reensamblando por
indice. Lado test: train completo como fuente
(scripts/phase25_submission_fresca.py:69-75).

ASCII puro. Sin efectos en import. Solo numpy/pandas/stdlib (+ cadena..
"""

from __future__ import annotations

import re
from collections import Counter

import numpy as np
import pandas as pd


# --------------------------------------------------------------------------------------
# Patrones de entidades (src/advanced_features.py:28-32)
# --------------------------------------------------------------------------------------
#: En WORD_RE los caracteres acentuados van como escapes \uXXXX para mantener el fichero
#: en ASCII puro; el patron compilado es identico al literal del original.
MENTION_RE = re.compile(r"@(\w+)")
HASHTAG_RE = re.compile(r"#(\w+)")
RT_SOURCE_RE = re.compile(r"^\s*RT\s+@(\w+)", flags=re.IGNORECASE)
WORD_RE = re.compile("[A-Za-z\u00c1\u00c9\u00cd\u00d3\u00da\u00dc\u00d1\u00e1\u00e9\u00ed\u00f3\u00fa\u00fc\u00f10-9]+")


def _safe_entropy(counts: np.ndarray) -> float:
    """Entropia de Shannon (base 2) segura para vectores vacios (src/advanced_features.py:56-62)."""
    total = counts.sum()
    if total <= 0:
        return 0.0
    probabilities = counts[counts > 0] / total
    return float(-(probabilities * np.log2(probabilities)).sum())


def _entity_documents(texts: pd.Series, kind: str) -> list[list[str]]:
    """Lista de entidades por documento segun el tipo pedido (src/advanced_features.py:312-323)."""
    if kind == "mention":
        return [[m.lower() for m in MENTION_RE.findall(value)]
                for value in texts.fillna("").astype(str)]
    if kind == "hashtag":
        return [[h.lower() for h in HASHTAG_RE.findall(value)]
                for value in texts.fillna("").astype(str)]
    if kind == "rt_source":
        return [[m.group(1).lower()] if (m := RT_SOURCE_RE.match(value)) else []
                for value in texts.fillna("").astype(str)]
    raise ValueError(f"tipo de entidad desconocido: {kind}")


def supervised_entity_features(train_texts: pd.Series, y_train: np.ndarray,
                               target_texts: pd.Series, kind: str = "mention",
                               smoothing: float = 20.0) -> pd.DataFrame:
    """Perfil de clase supervisado de las entidades (src/advanced_features.py:326-393).

    Aprende P(clase|entidad) con suavizado empirico-bayesiano solo con el train recibido
    (por tanto es seguro dentro de un fold) y resume cada documento con el perfil medio,
    el perfil de su entidad mas informativa, el margen, la entropia y la cobertura. Sin
    entidad conocida devuelve ``[prior, prior, 0, 0, 0, 0, 0]`` (17 columnas).

    Parameters
    ----------
    train_texts : pandas.Series
        Textos de train del fold.
    y_train : numpy.ndarray
        Etiquetas de train del fold.
    target_texts : pandas.Series
        Textos a transformar (validacion o test).
    kind : {"mention", "hashtag", "rt_source"}
        Tipo de entidad.
    smoothing : float
        Peso del prior de clase (cuanto mayor, mas se acerca a la distribucion global).

    Returns
    -------
    pandas.DataFrame
        17 columnas: ``{kind}_mean_c0..c5``, ``{kind}_best_c0..c5``, ``{kind}_margin``,
        ``{kind}_entropy``, ``{kind}_known_count``, ``{kind}_unknown_ratio`` y
        ``{kind}_dominance``.
    """
    classes = CLASSES
    class_index = {label: index for index, label in enumerate(classes)}
    train_entities = _entity_documents(train_texts, kind)
    target_entities = _entity_documents(target_texts, kind)

    prior = np.bincount([class_index[label] for label in y_train],
                        minlength=len(classes)).astype(float)
    prior /= prior.sum()

    counts: dict[str, np.ndarray] = {}
    for entities, label in zip(train_entities, y_train, strict=True):
        for entity in set(entities):
            counts.setdefault(entity, np.zeros(len(classes)))[class_index[label]] += 1

    profiles: dict[str, np.ndarray] = {}
    for entity, vector in counts.items():
        total = vector.sum()
        profiles[entity] = (vector + smoothing * prior) / (total + smoothing)

    rows: list[np.ndarray] = []
    for entities in target_entities:
        known = [profiles[entity] for entity in entities if entity in profiles]
        if not known:
            rows.append(np.concatenate([prior, prior, [0.0, 0.0, 0.0, 0.0, 0.0]]))
            continue
        stack = np.vstack(known)
        mean_profile = stack.mean(axis=0)
        best = stack[np.argmax(stack.max(axis=1))]
        ordered = np.sort(best)[::-1]
        margin = float(ordered[0] - ordered[1])
        entropy = _safe_entropy(best)
        rows.append(np.concatenate([mean_profile, best,
                                    [margin, entropy, float(len(stack)),
                                     float(1 - len(stack) / max(len(entities), 1)),
                                     float((np.array(entities) == Counter(entities).most_common(1)[0][0]).mean())]]))
    columns = ([f"{kind}_mean_c{i}" for i in range(len(classes))]
               + [f"{kind}_best_c{i}" for i in range(len(classes))]
               + [f"{kind}_margin", f"{kind}_entropy", f"{kind}_known_count",
                  f"{kind}_unknown_ratio", f"{kind}_dominance"])
    return pd.DataFrame(np.vstack(rows), columns=columns)


def lexical_rarity_features(train_texts: pd.Series, target_texts: pd.Series) -> pd.DataFrame:
    """Rareza y novedad lexica respecto del vocabulario del train recibido
    (src/advanced_features.py:396-434).

    Nunca usa estadisticas del conjunto objetivo (regla 74 del brief): las frecuencias se
    aprenden solo de ``train_texts``. Los OOV reciben el log10 centinela ``-6.0``
    (linea 419). Devuelve 12 columnas: ``rarity_mean_logfreq``, ``rarity_median_logfreq``,
    ``rarity_min_logfreq``, ``rarity_max_logfreq``, ``rarity_mean_freq``,
    ``rare_token_ratio``, ``very_rare_token_ratio``, ``oov_ratio``, ``hapax_token_ratio``,
    ``top_frequency_ratio``, ``lexical_novelty`` y ``lexical_familiarity``.
    """
    frequency: Counter = Counter()
    for tokens in train_texts.fillna("").str.lower().str.findall(WORD_RE.pattern):
        frequency.update(tokens)
    total = sum(frequency.values()) or 1
    log_frequencies = {token: np.log10(count / total) for token, count in frequency.items()}

    rows: list[dict] = []
    for tokens in target_texts.fillna("").str.lower().str.findall(WORD_RE.pattern):
        if not tokens:
            rows.append({"rarity_mean_logfreq": 0.0, "rarity_median_logfreq": 0.0,
                         "rarity_min_logfreq": 0.0, "rarity_max_logfreq": 0.0,
                         "rarity_mean_freq": 0.0, "rare_token_ratio": 0.0,
                         "very_rare_token_ratio": 0.0, "oov_ratio": 0.0,
                         "hapax_token_ratio": 0.0, "top_frequency_ratio": 0.0,
                         "lexical_novelty": 0.0, "lexical_familiarity": 0.0})
            continue
        frequencies = np.array([frequency.get(token, 0) for token in tokens], dtype=float)
        log_freqs = np.array([log_frequencies.get(token, -6.0) for token in tokens])
        rows.append({
            "rarity_mean_logfreq": float(log_freqs.mean()),
            "rarity_median_logfreq": float(np.median(log_freqs)),
            "rarity_min_logfreq": float(log_freqs.min()),
            "rarity_max_logfreq": float(log_freqs.max()),
            "rarity_mean_freq": float(frequencies.mean()),
            "rare_token_ratio": float((frequencies <= 5).mean()),
            "very_rare_token_ratio": float((frequencies <= 2).mean()),
            "oov_ratio": float((frequencies == 0).mean()),
            "hapax_token_ratio": float((frequencies == 1).mean()),
            "top_frequency_ratio": float(np.mean(log_freqs >= np.percentile(log_freqs, 90))),
            "lexical_novelty": float((frequencies <= 2).mean()),
            "lexical_familiarity": float(np.mean(log_freqs)),
        })
    return pd.DataFrame(rows)


def class_lexicon_scores(train_texts: pd.Series, y_train: np.ndarray,
                         target_texts: pd.Series, alpha0: float = 100.0,
                         min_count: int = 5, top_k: int = 3000) -> pd.DataFrame:
    """Distancia al lexico de cada clase mediante log-odds (Monroe et al. 2008)
    (src/advanced_features.py:437-490).

    Aprende, solo con el train recibido, los ``top_k`` terminos mas distintivos de cada
    clase (frecuencia minima ``min_count``, prior ``alpha0``) y resume cada documento con
    la puntuacion media de sus tokens frente a cada lexico, mas el margen, la entropia, la
    clase ganadora y la desviacion. Devuelve 10 columnas: ``lexicon_c0..c5``,
    ``lexicon_margin``, ``lexicon_entropy``, ``lexicon_best_class`` y ``lexicon_std``.
    """
    classes = CLASSES
    class_index = {label: index for index, label in enumerate(classes)}
    train_tokens = train_texts.fillna("").str.lower().str.findall(WORD_RE.pattern)
    counter_global: Counter = Counter()
    counters_by_class = [Counter() for _ in classes]
    for tokens, label in zip(train_tokens, y_train, strict=True):
        position = class_index[label]
        counter_global.update(tokens)
        counters_by_class[position].update(tokens)

    total = sum(counter_global.values()) or 1
    prior = {token: alpha0 * count / total for token, count in counter_global.items()}
    class_scores: list[dict[str, float]] = []
    totals_by_class = [sum(counter.values()) for counter in counters_by_class]
    for position in range(len(classes)):
        scores: dict[str, float] = {}
        other_total = total - totals_by_class[position]
        for token, count in counter_global.items():
            if count < min_count:
                continue
            in_class = counters_by_class[position].get(token, 0)
            out_class = count - in_class
            if in_class == 0:
                continue
            log_odds = (np.log((in_class + prior[token])
                               / (totals_by_class[position] + alpha0 - in_class - prior[token]))
                        - np.log((out_class + prior[token])
                                 / (other_total + alpha0 - out_class - prior[token])))
            scores[token] = float(log_odds)
        top_items = sorted(scores.items(), key=lambda item: -item[1])[:top_k]
        class_scores.append({token: value for token, value in top_items})

    rows: list[dict] = []
    for tokens in target_texts.fillna("").str.lower().str.findall(WORD_RE.pattern):
        vector = np.zeros(len(classes))
        for position, scores in enumerate(class_scores):
            if tokens:
                vector[position] = float(np.mean([scores.get(token, 0.0) for token in tokens]))
        ordered = np.sort(vector)[::-1]
        rows.append({**{f"lexicon_c{i}": vector[i] for i in range(len(classes))},
                     "lexicon_margin": float(ordered[0] - ordered[1]),
                     "lexicon_entropy": _safe_entropy(vector - vector.min() + 1e-9),
                     "lexicon_best_class": float(np.argmax(vector)),
                     "lexicon_std": float(vector.std())})
    return pd.DataFrame(rows)


def supervisadas_infold(textos: pd.Series, y: np.ndarray, fold_idx: np.ndarray) -> np.ndarray:
    """Features supervisadas LOFO estricto, (n, 73) float64.

    Porte de scripts/advanced_02_features.py:205-234. Para cada fold se ajustan los 5
    bloques con el train de los otros 4 folds y se transforma la validacion; el resultado
    se reensambla por indice (posicional) en el orden original. Orden de columnas:
    mention 17 + hashtag 17 + rt_source 17 + rarity 12 + lexicon 10.

    Parameters
    ----------
    textos : pandas.Series
        Textos del train completo (columna de texto, ya limpia).
    y : numpy.ndarray
        Etiquetas del train completo (strings de CLASSES).
    fold_idx : numpy.ndarray
        Protocolo de folds congelado: valor 0-4 por fila.

    Returns
    -------
    numpy.ndarray
        Matriz (n, 73) float64 en el orden de filas de ``textos``.
    """
    partes: list[pd.DataFrame] = []
    for fold in sorted(np.unique(fold_idx)):
        train_idx = np.flatnonzero(fold_idx != fold)
        valid_idx = np.flatnonzero(fold_idx == fold)
        train_texts = textos.iloc[train_idx]
        valid_texts = textos.iloc[valid_idx]
        y_train = y[train_idx]
        # Mismo orden que el dict `blocks` del original: mencion, hashtag, fuente_rt,
        # rareza, lexico (scripts/advanced_02_features.py:215-221).
        block = pd.concat([
            supervised_entity_features(train_texts, y_train, valid_texts, "mention"),
            supervised_entity_features(train_texts, y_train, valid_texts, "hashtag"),
            supervised_entity_features(train_texts, y_train, valid_texts, "rt_source"),
            lexical_rarity_features(train_texts, valid_texts),
            class_lexicon_scores(train_texts, y_train, valid_texts),
        ], axis=1)
        block.index = valid_idx
        partes.append(block)
    matriz = pd.concat(partes).sort_index().to_numpy(float)
    assert matriz.shape[1] == 73, matriz.shape
    return matriz


def supervisadas_test(textos_train: pd.Series, y: np.ndarray,
                      textos_test: pd.Series) -> np.ndarray:
    """Features supervisadas con el train completo como fuente, (m, 73) float64.

    Porte de scripts/phase25_submission_fresca.py:69-75: misma funcion que in-fold pero
    ajustando los 5 bloques con todo el train y transformando el test. Orden de columnas:
    mention 17 + hashtag 17 + rt_source 17 + rarity 12 + lexicon 10.

    Parameters
    ----------
    textos_train : pandas.Series
        Textos del train completo.
    y : numpy.ndarray
        Etiquetas del train completo (strings de CLASSES).
    textos_test : pandas.Series
        Textos del test a transformar.

    Returns
    -------
    numpy.ndarray
        Matriz (m, 73) float64 en el orden de filas de ``textos_test``.
    """
    matriz = pd.concat([
        supervised_entity_features(textos_train, y, textos_test, "mention"),
        supervised_entity_features(textos_train, y, textos_test, "hashtag"),
        supervised_entity_features(textos_train, y, textos_test, "rt_source"),
        lexical_rarity_features(textos_train, textos_test),
        class_lexicon_scores(textos_train, y, textos_test),
    ], axis=1).to_numpy(float)
    assert matriz.shape[1] == 73, matriz.shape
    return matriz

### Módulo `nb_svm.py`

In [ ]:
r"""NB-SVM (Wang & Manning 2012) uno-vs-rest: log-count ratio + SVM lineal por clase.

Porte literal de los originales del proyecto KAGGLE_AGE_2026_PUJ:
- scripts/advanced_05_nbsvm_seleccion.py:66-94 (ADV05: char_wb 3-6 300k + menciones).
- scripts/phase7_nbsvm_canal_meta.py:57-78 (ADV07: solo n-gramas de palabra).
- Lado test (mismo calculo reajustado con todo el train): scripts/phase3_holdout_v11.py:115-127
  y scripts/phase25_submission_fresca.py:77-89.

Reglas del paquete: ASCII puro; sin efectos en import; solo numpy/pandas/scipy/sklearn y
`cadena.. Interfaz en plan/kaggle_notebook/port_interface.md.
"""

from __future__ import annotations

import numpy as np
import pandas as pd
import scipy.sparse as sp
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC



def nb_log_count_ratio(x_train: sp.csr_matrix, binary_target: np.ndarray) -> np.ndarray:
    r"""Log-count ratio de la clase positiva frente al resto (Wang & Manning 2012).

    Porte literal de scripts/advanced_05_nbsvm_seleccion.py:66-70 y
    scripts/phase7_nbsvm_canal_meta.py:57-61:
    log((pos / pos.sum()) / (neg / neg.sum())) con +1.0 de suavizado en los conteos.
    """
    positive = np.asarray(x_train[binary_target == 1].sum(axis=0)).ravel() + 1.0
    negative = np.asarray(x_train[binary_target == 0].sum(axis=0)).ravel() + 1.0
    return np.log((positive / positive.sum()) / (negative / negative.sum()))


def nb_svm_margins(textos: pd.Series, y: np.ndarray, train_idx: np.ndarray,
                   valid_idx: np.ndarray, config: dict) -> np.ndarray:
    r"""Margenes NB-SVM uno-vs-rest del fold retenido (float32, columnas en CLASSES).

    Porte de scripts/advanced_05_nbsvm_seleccion.py:73-94 (ADV05, char300 + menciones) y
    scripts/phase7_nbsvm_canal_meta.py:64-78 (ADV07, solo palabras): vectorizador(es)
    TF-IDF ajustados SOLO con textos.iloc[train_idx]; por clase (en orden CLASSES)
    log-count ratio sobre el train del fold y LinearSVC(C=0.2, max_iter=3000) entrenado
    sobre x_train.multiply(ratio).tocsr(); decision_function sobre el fold retenido
    multiplicado por el mismo ratio. Las columnas ya quedan en el orden de CLASSES
    porque el bucle entrena una clase por posicion.

    config: parametros de TfidfVectorizer del canal principal. Clave opcional
    "mentions" = {"config": {...}, "pattern": r"@\w+"} para anadir un segundo canal
    TF-IDF sobre tokens_of(textos, pattern). ADV05 la usa con CHAR_300K + MENTION y
    patron r"@\w+"; ADV07 pasa WORD sin canal de menciones (el texto tal cual).

    train_idx/valid_idx: indices enteros (p. ej. np.flatnonzero) de las filas de
    entrenamiento y validacion, como fold["train_idx"]/fold["val_idx"] del protocolo
    congelado. Salida (len(valid_idx), 6) float32.
    """
    config = dict(config)
    mentions = config.pop("mentions", None)
    y = np.asarray(y)
    vectorizer = TfidfVectorizer(**config)
    if mentions is None:
        x_train = vectorizer.fit_transform(textos.iloc[train_idx])
        x_valid = vectorizer.transform(textos.iloc[valid_idx])
    else:
        mention_vectorizer = TfidfVectorizer(**mentions["config"])
        pattern = mentions["pattern"]
        x_train = sp.hstack([
            vectorizer.fit_transform(textos.iloc[train_idx]),
            mention_vectorizer.fit_transform(tokens_of(textos.iloc[train_idx], pattern)),
        ]).tocsr()
        x_valid = sp.hstack([
            vectorizer.transform(textos.iloc[valid_idx]),
            mention_vectorizer.transform(tokens_of(textos.iloc[valid_idx], pattern)),
        ]).tocsr()
    y_train = y[train_idx]

    margins = np.zeros((len(valid_idx), len(CLASSES)), dtype=np.float32)
    for position, label in enumerate(CLASSES):
        binary_target = (y_train == label).astype(int)
        ratio = nb_log_count_ratio(x_train, binary_target)
        model = LinearSVC(C=0.2, max_iter=3000)
        model.fit(x_train.multiply(ratio).tocsr(), binary_target)
        margins[:, position] = model.decision_function(x_valid.multiply(ratio).tocsr())
    return margins

### Módulo `miembros_oof_a.py`

In [ ]:
"""Miembros OOF del bloque A de la cadena AGE (porte fiel del proyecto original).

Cada funcion reproduce EXACTAMENTE el script citado: mismos modelos, hiperparametros,
representaciones y orden de columnas. Protocolo  entrenar con ``fold_idx != fold``,
predecir ``fold_idx == fold`` y reensamblar por indice.

Fuentes:
- E-013: scripts/experimento_e013_barrido_c.py:80-92,112
- E-016: scripts/experimento_e016_e017.py:76-87,126
- E-009 / E-010 / E-011: scripts/experimento_e009_modelos_clasicos.py:55-74,147-151,171-175
- E-014 / E-015: scripts/experimento_e014_e015_hipotesis_error.py:101-112,122-132,151-153

Salidas: margenes y probabilidades (n, 6) float32 reordenados a ``CLASSES``;
E-015 ordinal (n,) float32. ASCII puro. Sin efectos en import.
"""

from __future__ import annotations

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, Ridge, SGDClassifier
from sklearn.multiclass import OneVsRestClassifier
from sklearn.naive_bayes import ComplementNB
from sklearn.svm import LinearSVC


# Helpers compartidos (representacion y orden de columnas).


def _orden_canonico(classes: np.ndarray) -> list[int]:
    """Indices para reordenar columnas al orden de ``CLASSES``.

    Patron literal de los originales:
    ``[list(model.classes_).index(label) for label in CLASSES]``
    (scripts/experimento_e013_barrido_c.py:91-92).
    """
    return [list(classes).index(etiqueta) for etiqueta in CLASSES]


def _representacion_char_menciones(train_texts: pd.Series, valid_texts: pd.Series,
                                   char_config: dict) -> tuple[csr_matrix, csr_matrix]:
    """Representacion char + menciones ajustada solo con el train del fold.

    Fuente: scripts/experimento_e009_modelos_clasicos.py:90-98 (``build_representation``);
    bloque identico al de E-013/E-016/E-014/E-015
    (scripts/experimento_e013_barrido_c.py:71-76).
    """
    char = TfidfVectorizer(**char_config)
    menciones = TfidfVectorizer(**MENTION)
    x_train = hstack([char.fit_transform(train_texts),
                      menciones.fit_transform(tokens_of(train_texts, r"@\w+"))]).tocsr()
    x_valid = hstack([char.transform(valid_texts),
                      menciones.transform(tokens_of(valid_texts, r"@\w+"))]).tocsr()
    return x_train, x_valid


# Miembros OOF (uno por artefacto del bloque A).


def oof_e013(textos: pd.Series, y: np.ndarray, fold_idx: np.ndarray) -> np.ndarray:
    """Margenes OOF de E-013.

    Fuente: scripts/experimento_e013_barrido_c.py:80-92,112.
    LinearSVC(C=0.2, max_iter=3000) sobre char_wb 3-6 (min_df=5, max_features=300k) +
    menciones word 1-1 (min_df=2); vectorizadores ajustados in-fold. Devuelve (n, 6)
    float32 con ``decision_function`` reordenado a ``CLASSES``.
    """
    salida = np.zeros((len(textos), len(CLASSES)), dtype=np.float32)
    for fold in np.unique(fold_idx):
        train_idx = fold_idx != fold
        valid_idx = fold_idx == fold
        x_train, x_valid = _representacion_char_menciones(textos.iloc[train_idx],
                                                          textos.iloc[valid_idx],
                                                          CHAR_300K)
        modelo = LinearSVC(C=0.2, max_iter=3000)
        modelo.fit(x_train, y[train_idx])
        salida[valid_idx] = modelo.decision_function(x_valid)[:, _orden_canonico(modelo.classes_)]
    return salida


def oof_e016(textos: pd.Series, y: np.ndarray, fold_idx: np.ndarray) -> np.ndarray:
    """Margenes OOF de E-016.

    Fuente: scripts/experimento_e016_e017.py:76-87,126.
    LinearSVC(C=0.2, max_iter=3000) sobre char_wb 3-6 (min_df=3, max_features=700k) +
    menciones word 1-1 (min_df=2); vectorizadores ajustados in-fold. Devuelve (n, 6)
    float32 con ``decision_function`` reordenado a ``CLASSES``.
    """
    salida = np.zeros((len(textos), len(CLASSES)), dtype=np.float32)
    for fold in np.unique(fold_idx):
        train_idx = fold_idx != fold
        valid_idx = fold_idx == fold
        x_train, x_valid = _representacion_char_menciones(textos.iloc[train_idx],
                                                          textos.iloc[valid_idx],
                                                          CHAR_700K)
        modelo = LinearSVC(C=0.2, max_iter=3000)
        modelo.fit(x_train, y[train_idx])
        salida[valid_idx] = modelo.decision_function(x_valid)[:, _orden_canonico(modelo.classes_)]
    return salida


def oof_e009(textos: pd.Series, y: np.ndarray, fold_idx: np.ndarray) -> np.ndarray:
    """Probabilidades OOF de E-009.

    Fuente: scripts/experimento_e009_modelos_clasicos.py:55-61,147-151,171-175.
    OneVsRestClassifier(LogisticRegression(C=4.0, max_iter=1000, solver="liblinear",
    random_state=42)) sobre char_wb 3-6 (300k) + menciones. Devuelve (n, 6) float32 con
    ``predict_proba`` reordenada a ``CLASSES``.
    """
    salida = np.zeros((len(textos), len(CLASSES)), dtype=np.float32)
    for fold in np.unique(fold_idx):
        train_idx = fold_idx != fold
        valid_idx = fold_idx == fold
        x_train, x_valid = _representacion_char_menciones(textos.iloc[train_idx],
                                                          textos.iloc[valid_idx],
                                                          CHAR_300K)
        modelo = OneVsRestClassifier(LogisticRegression(C=4.0, max_iter=1000,
                                                        solver="liblinear",
                                                        random_state=SEED))
        modelo.fit(x_train, y[train_idx])
        salida[valid_idx] = modelo.predict_proba(x_valid)[:, _orden_canonico(modelo.classes_)]
    return salida


def oof_e011(textos: pd.Series, y: np.ndarray, fold_idx: np.ndarray) -> np.ndarray:
    """Probabilidades OOF de E-011.

    Fuente: scripts/experimento_e009_modelos_clasicos.py:68-74,147-151,171-175.
    SGDClassifier(loss="log_loss", alpha=1e-6, max_iter=30, tol=1e-4, random_state=42,
    n_jobs=32) sobre char_wb 3-6 (300k) + menciones. Devuelve (n, 6) float32 con
    ``predict_proba`` reordenada a ``CLASSES``.
    """
    salida = np.zeros((len(textos), len(CLASSES)), dtype=np.float32)
    for fold in np.unique(fold_idx):
        train_idx = fold_idx != fold
        valid_idx = fold_idx == fold
        x_train, x_valid = _representacion_char_menciones(textos.iloc[train_idx],
                                                          textos.iloc[valid_idx],
                                                          CHAR_300K)
        modelo = SGDClassifier(loss="log_loss", alpha=1e-6, max_iter=30, tol=1e-4,
                               random_state=SEED, n_jobs=32)
        modelo.fit(x_train, y[train_idx])
        salida[valid_idx] = modelo.predict_proba(x_valid)[:, _orden_canonico(modelo.classes_)]
    return salida


def oof_e010(textos: pd.Series, y: np.ndarray, fold_idx: np.ndarray) -> np.ndarray:
    """Probabilidades OOF de E-010.

    Fuente: scripts/experimento_e009_modelos_clasicos.py:62-67,147-151,171-175.
    ComplementNB(alpha=0.3) sobre char_wb 3-6 (300k) + menciones. Devuelve (n, 6) float32
    con ``predict_proba`` reordenada a ``CLASSES``.
    """
    salida = np.zeros((len(textos), len(CLASSES)), dtype=np.float32)
    for fold in np.unique(fold_idx):
        train_idx = fold_idx != fold
        valid_idx = fold_idx == fold
        x_train, x_valid = _representacion_char_menciones(textos.iloc[train_idx],
                                                          textos.iloc[valid_idx],
                                                          CHAR_300K)
        modelo = ComplementNB(alpha=0.3)
        modelo.fit(x_train, y[train_idx])
        salida[valid_idx] = modelo.predict_proba(x_valid)[:, _orden_canonico(modelo.classes_)]
    return salida


def oof_e014(textos: pd.Series, y: np.ndarray, fold_idx: np.ndarray) -> np.ndarray:
    """Margenes OOF de E-014.

    Fuente: scripts/experimento_e014_e015_hipotesis_error.py:101-112,151.
    LinearSVC(C=0.2, max_iter=3000) sobre char_wb 3-6 (300k) + menciones + hashtags
    word 1-1 (min_df=2); vectorizadores ajustados in-fold. Devuelve (n, 6) float32 con
    ``decision_function`` reordenado a ``CLASSES``.
    """
    salida = np.zeros((len(textos), len(CLASSES)), dtype=np.float32)
    for fold in np.unique(fold_idx):
        train_idx = fold_idx != fold
        valid_idx = fold_idx == fold
        x_train, x_valid = _representacion_char_menciones(textos.iloc[train_idx],
                                                          textos.iloc[valid_idx],
                                                          CHAR_300K)
        hashtags = TfidfVectorizer(**HASHTAG)
        x_train_h = hstack([x_train, hashtags.fit_transform(
            tokens_of(textos.iloc[train_idx], r"#\w+"))]).tocsr()
        x_valid_h = hstack([x_valid, hashtags.transform(
            tokens_of(textos.iloc[valid_idx], r"#\w+"))]).tocsr()
        modelo = LinearSVC(C=0.2, max_iter=3000)
        modelo.fit(x_train_h, y[train_idx])
        salida[valid_idx] = modelo.decision_function(x_valid_h)[:, _orden_canonico(modelo.classes_)]
    return salida


def oof_e015_ordinal(textos: pd.Series, y: np.ndarray, fold_idx: np.ndarray) -> np.ndarray:
    """Prediccion ordinal continua OOF de E-015.

    Fuente: scripts/experimento_e014_e015_hipotesis_error.py:122-128,152.
    Ridge(alpha=1.0, solver="sparse_cg", max_iter=2000, random_state=42) sobre
    char_wb 3-6 (300k) + menciones, con objetivo ``CLASS_MIDPOINTS`` de cada
    etiqueta (como el ``ordinal_target`` original). Devuelve (n,) float32.
    """
    objetivo = np.array([CLASS_MIDPOINTS[etiqueta] for etiqueta in y])
    salida = np.zeros(len(textos), dtype=np.float32)
    for fold in np.unique(fold_idx):
        train_idx = fold_idx != fold
        valid_idx = fold_idx == fold
        x_train, x_valid = _representacion_char_menciones(textos.iloc[train_idx],
                                                          textos.iloc[valid_idx],
                                                          CHAR_300K)
        modelo = Ridge(alpha=1.0, solver="sparse_cg", max_iter=2000, random_state=SEED)
        modelo.fit(x_train, objetivo[train_idx])
        salida[valid_idx] = modelo.predict(x_valid)
    return salida


def oof_e015_softmax(textos: pd.Series, y: np.ndarray, fold_idx: np.ndarray) -> np.ndarray:
    """Distribucion suave OOF de E-015 (softmax ordinal).

    Fuente: scripts/experimento_e014_e015_hipotesis_error.py:129-132,153.
    Aplica ``softmax_ordinal`` a la salida de ``oof_e015_ordinal``. Devuelve (n, 6)
    float32. El original calculaba el softmax sobre la prediccion continua float64 previa
    al casteo; aqui se aplica sobre la salida float32 (diferencia numerica esperada del
    orden de 1e-6).
    """
    return softmax_ordinal(oof_e015_ordinal(textos, y, fold_idx)).astype(np.float32)

### Módulo `miembros_oof_b.py`

In [ ]:
r"""Miembros OOF del paquete B de la cadena AGE: E-017, E-019b, E-021a, E-021b, E-028a y ADV05.

Firma `(textos, y, fold_idx)`: por fold del protocolo congelado (5 folds) se entrena
con `fold_idx != fold` y se predice `fold_idx == fold`, reensamblando por indice. Salidas
float32 con columnas en CLASSES, salvo E-028a (columnas = ADJACENT_PAIRS).

Fuentes (porte literal, sin mejorar ni reinterpretar):
- E-017: scripts/experimento_e016_e017.py:97-108,127.
- E-019b: scripts/experimento_e019_modelos_densos.py:65-80,98-99,113-131,157.
- E-021a: scripts/experimento_e021_ordinal_centroides.py:57-75,110-122,150.
- E-021b: scripts/experimento_e021_ordinal_centroides.py:78-91,110-127,151.
- E-028a: scripts/experimento_e028_ordinal_fino.py:92-106,147.
- ADV05: scripts/advanced_05_nbsvm_seleccion.py:73-94,138-146.

Reglas del paquete: ASCII puro; sin efectos en import; solo numpy/pandas/scipy/sklearn y
`cadena` (nb_svm). Interfaz en plan/kaggle_notebook/port_interface.md.
"""

from __future__ import annotations

import numpy as np
import pandas as pd
from scipy.sparse import hstack
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import LinearSVC



def _folds_estandar(fold_idx: np.ndarray):
    """Genera (fold, train_idx, valid_idx) del protocolo congelado (5 folds).

    Entrena con fold_idx != fold y predice fold_idx == fold; devuelve indices enteros,
    como fold["train_idx"]/fold["val_idx"] de src/cv.py:332-374 (make_folds agrupado).
    """
    fold_idx = np.asarray(fold_idx)
    for fold in range(5):
        yield fold, np.flatnonzero(fold_idx != fold), np.flatnonzero(fold_idx == fold)


def _tfidf_char_menciones(textos: pd.Series, train_idx: np.ndarray, valid_idx: np.ndarray):
    r"""TF-IDF in-fold char_wb 3-6 (CHAR_300K) + canal de menciones (MENTION, r"@\w+").

    Porte de scripts/experimento_e021_ordinal_centroides.py:112-117; identico en
    scripts/experimento_e019_modelos_densos.py:114-119 y
    scripts/experimento_e028_ordinal_fino.py:84-89.
    """
    char = TfidfVectorizer(**CHAR_300K)
    menciones = TfidfVectorizer(**MENTION)
    x_train = hstack([char.fit_transform(textos.iloc[train_idx]),
                      menciones.fit_transform(tokens_of(textos.iloc[train_idx],
                                                        r"@\w+"))]).tocsr()
    x_valid = hstack([char.transform(textos.iloc[valid_idx]),
                      menciones.transform(tokens_of(textos.iloc[valid_idx],
                                                    r"@\w+"))]).tocsr()
    return x_train, x_valid


def _ordinal_probabilidades(x_train, y_index: np.ndarray, x_valid) -> np.ndarray:
    r"""Distribucion de clase por modelo ordinal acumulativo (Frank & Hall 2001).

    Porte de scripts/experimento_e021_ordinal_centroides.py:57-75: K-1 clasificadores
    binarios P(y > k) con LogisticRegression(C=1.0, max_iter=1000, solver="liblinear",
    random_state=SEED); acumuladas monotonizadas con minimum.accumulate y clip [0, 1];
    P(y = k) por diferencias (-diff + ultima columna), clip 1e-12 y normalizacion.
    """
    thresholds = np.arange(1, len(CLASSES))
    cumulative = np.ones((x_valid.shape[0], len(thresholds) + 1), dtype=np.float64)
    for position, threshold in enumerate(thresholds):
        binary_target = (y_index >= threshold).astype(int)
        model = LogisticRegression(C=1.0, max_iter=1000, solver="liblinear",
                                   random_state=SEED)
        model.fit(x_train, binary_target)
        cumulative[:, position + 1] = model.predict_proba(x_valid)[:, 1]
    cumulative = np.clip(np.minimum.accumulate(cumulative, axis=1), 0.0, 1.0)
    # P(y = k) = P(y > k-1) - P(y > k); la ultima clase es P(y > K-1) - 0.
    probabilities = np.column_stack([-np.diff(cumulative, axis=1), cumulative[:, -1]])
    probabilities = np.clip(probabilities, 1e-12, None)
    return probabilities / probabilities.sum(axis=1, keepdims=True)


def _rocchio_probabilidades(x_train, y: np.ndarray, x_valid) -> np.ndarray:
    r"""Clasificador de centroides (Rocchio) con similitud coseno y softmax T=8.

    Porte de scripts/experimento_e021_ordinal_centroides.py:78-91: centroide medio por
    clase normalizado L2 (epsilon 1e-12); norma de cada fila valida (sqrt de la suma de
    cuadrados, epsilon 1e-12); similitud coseno; softmax desplazada por el maximo por
    fila con temperatura 8.0.
    """
    centroids = []
    for label in CLASSES:
        mask = y == label
        centroid = np.asarray(x_train[mask].mean(axis=0)).ravel()
        centroids.append(centroid / (np.linalg.norm(centroid) + 1e-12))
    centroid_matrix = np.vstack(centroids)

    valid_norms = np.sqrt(np.asarray(x_valid.multiply(x_valid).sum(axis=1)).ravel()) + 1e-12
    similarity = np.asarray(x_valid @ centroid_matrix.T) / valid_norms[:, None]
    shifted = similarity - similarity.max(axis=1, keepdims=True)
    exponentials = np.exp(shifted * 8.0)  # temperatura: afina la distribucion
    return exponentials / exponentials.sum(axis=1, keepdims=True)


def oof_e017(textos: pd.Series, y: np.ndarray, fold_idx: np.ndarray) -> np.ndarray:
    r"""Probabilidades OOF de E-017: LogReg OvR C=4.0 sobre word 1-2 (min_df=3, 300k).

    Porte de scripts/experimento_e016_e017.py:97-108 y 127: TfidfVectorizer(**WORD)
    ajustado in-fold; OneVsRestClassifier(LogisticRegression(C=4.0, max_iter=1000,
    solver="liblinear", random_state=SEED)); predict_proba del fold retenido reordenada
    a CLASSES con logreg.classes_. Salida (n, 6) float32.
    """
    y = np.asarray(y)
    probabilidades = np.zeros((len(textos), len(CLASSES)), dtype=np.float32)
    for _fold, train_idx, valid_idx in _folds_estandar(fold_idx):
        word = TfidfVectorizer(**WORD)
        x_train = word.fit_transform(textos.iloc[train_idx])
        x_valid = word.transform(textos.iloc[valid_idx])
        logreg = OneVsRestClassifier(LogisticRegression(C=4.0, max_iter=1000,
                                                        solver="liblinear",
                                                        random_state=SEED))
        logreg.fit(x_train, y[train_idx])
        orden = [list(logreg.classes_).index(etiqueta) for etiqueta in CLASSES]
        probabilidades[valid_idx] = logreg.predict_proba(x_valid)[:, orden]
    return probabilidades


def oof_e019b(textos: pd.Series, y: np.ndarray, fold_idx: np.ndarray,
              dense: pd.DataFrame) -> np.ndarray:
    r"""Probabilidades OOF de E-019b: MLP(256,128) sobre [35 densas + SVD(64)].

    Porte de scripts/experimento_e019_modelos_densos.py:113-131 y 157 (variante B) mas
    fit_mlp (:65-80) y el LabelEncoder con assert de orden CLASSES (:98-99): TF-IDF
    in-fold char_wb 3-6 (CHAR_300K) + menciones (MENTION, r"@\w+"); TruncatedSVD(
    n_components=64, random_state=SEED) ajustada SOLO con el train del fold; entrada
    [densas | z] escalada con StandardScaler (ajustado in-fold); MLPClassifier(
    hidden_layer_sizes=(256, 128), alpha=1e-3, batch_size=512, learning_rate_init=1e-3,
    max_iter=60, early_stopping=True, n_iter_no_change=6, random_state=SEED) con etiquetas
    codificadas como enteros (scikit-learn 1.8 falla en early stopping con etiquetas
    cadena). Las 35 densas = columnas de `dense` sin {n_emoji, emoji_diversity, has_emoji}.
    Salida (n, 6) float32.
    """
    y = np.asarray(y)
    columnas = [col for col in dense.columns
                if col not in {"n_emoji", "emoji_diversity", "has_emoji"}]
    assert len(columnas) == 35, f"se esperaban 35 densas y hay {len(columnas)}"
    densas = dense[columnas].to_numpy(float)
    encoder = LabelEncoder().fit(y)
    assert list(encoder.classes_) == list(CLASSES), "orden de clases inesperado"
    probabilidades = np.zeros((len(textos), len(CLASSES)), dtype=np.float32)
    for _fold, train_idx, valid_idx in _folds_estandar(fold_idx):
        x_train, x_valid = _tfidf_char_menciones(textos, train_idx, valid_idx)
        svd = TruncatedSVD(n_components=64, random_state=SEED)
        z_train = svd.fit_transform(x_train)
        z_valid = svd.transform(x_valid)
        combined_train = np.hstack([densas[train_idx], z_train])
        combined_valid = np.hstack([densas[valid_idx], z_valid])
        scaler = StandardScaler().fit(combined_train)
        model = MLPClassifier(hidden_layer_sizes=(256, 128), alpha=1e-3, batch_size=512,
                              learning_rate_init=1e-3, max_iter=60, early_stopping=True,
                              n_iter_no_change=6, random_state=SEED)
        model.fit(scaler.transform(combined_train), encoder.transform(y[train_idx]))
        probabilidades[valid_idx] = model.predict_proba(scaler.transform(combined_valid))
    return probabilidades


def oof_e021a(textos: pd.Series, y: np.ndarray, fold_idx: np.ndarray) -> np.ndarray:
    r"""Probabilidades OOF de E-021a: ordinal acumulativo Frank & Hall (5 binarios).

    Porte de scripts/experimento_e021_ordinal_centroides.py:110-122 y 150: por fold,
    representacion in-fold char_wb 3-6 (CHAR_300K) + menciones; y_index = posicion de la
    clase en CLASSES; modelo ordinal acumulativo :57-75. Salida (n, 6) float32.
    """
    y = np.asarray(y)
    y_index = np.array([CLASSES.index(etiqueta) for etiqueta in y])
    probabilidades = np.zeros((len(textos), len(CLASSES)), dtype=np.float32)
    for _fold, train_idx, valid_idx in _folds_estandar(fold_idx):
        x_train, x_valid = _tfidf_char_menciones(textos, train_idx, valid_idx)
        probabilidades[valid_idx] = _ordinal_probabilidades(x_train, y_index[train_idx],
                                                            x_valid)
    return probabilidades


def oof_e021b(textos: pd.Series, y: np.ndarray, fold_idx: np.ndarray) -> np.ndarray:
    r"""Probabilidades OOF de E-021b: centroides Rocchio + softmax temperatura 8.0.

    Porte de scripts/experimento_e021_ordinal_centroides.py:110-127 y 151: por fold,
    representacion in-fold char_wb 3-6 (CHAR_300K) + menciones; centroides Rocchio :78-91
    sobre el train del fold. Salida (n, 6) float32.
    """
    y = np.asarray(y)
    probabilidades = np.zeros((len(textos), len(CLASSES)), dtype=np.float32)
    for _fold, train_idx, valid_idx in _folds_estandar(fold_idx):
        x_train, x_valid = _tfidf_char_menciones(textos, train_idx, valid_idx)
        probabilidades[valid_idx] = _rocchio_probabilidades(x_train, y[train_idx], x_valid)
    return probabilidades


def oof_e028a(textos: pd.Series, y: np.ndarray, fold_idx: np.ndarray) -> np.ndarray:
    r"""Margenes OOF de E-028a: 5 SVM entrenados SOLO con cada par adyacente.

    Porte de scripts/experimento_e028_ordinal_fino.py:92-106 y 147: representacion
    in-fold char_wb 3-6 (CHAR_300K) + menciones; por par (lower, upper) de ADJACENT_PAIRS
    mascara con las dos clases, objetivo (y_train == upper) y LinearSVC(C=0.2,
    max_iter=3000); decision_function del fold retenido (margen positivo -> clase mayor).
    Columnas = ADJACENT_PAIRS. Salida (n, 5) float32.
    """
    y = np.asarray(y)
    margenes = np.zeros((len(textos), len(ADJACENT_PAIRS)), dtype=np.float32)
    for _fold, train_idx, valid_idx in _folds_estandar(fold_idx):
        x_train, x_valid = _tfidf_char_menciones(textos, train_idx, valid_idx)
        y_train = y[train_idx]
        pair_margins = np.zeros((len(valid_idx), len(ADJACENT_PAIRS)), dtype=np.float32)
        for position, (lower, upper) in enumerate(ADJACENT_PAIRS):
            mask = (y_train == lower) | (y_train == upper)
            binary_target = (y_train[mask] == upper).astype(int)
            model = LinearSVC(C=0.2, max_iter=3000)
            model.fit(x_train[mask], binary_target)
            margin = model.decision_function(x_valid)
            pair_margins[:, position] = margin
        margenes[valid_idx] = pair_margins
    return margenes


def oof_adv05(textos: pd.Series, y: np.ndarray, fold_idx: np.ndarray) -> np.ndarray:
    r"""Margenes OOF de ADV05: NB-SVM uno-vs-rest sobre char300 + menciones.

    Porte de scripts/advanced_05_nbsvm_seleccion.py:73-94 (nb_svm_fold) y 138-146 (bucle
    OOF y guardado): por fold, TfidfVectorizer(**CHAR_300K) + TfidfVectorizer(**MENTION)
    sobre tokens_of(textos, r"@\w+"), log-count ratio por clase y LinearSVC(C=0.2,
    max_iter=3000) uno-vs-rest via nb_svm_margins. Salida (n, 6) float32.
    """
    y = np.asarray(y)
    config = dict(CHAR_300K)
    config["mentions"] = {"config": dict(MENTION), "pattern": r"@\w+"}
    margenes = np.zeros((len(textos), len(CLASSES)), dtype=np.float32)
    for _fold, train_idx, valid_idx in _folds_estandar(fold_idx):
        margenes[valid_idx] = nb_svm_margins(textos, y, train_idx, valid_idx, config)
    return margenes

### Módulo `meta.py`

In [ ]:
"""Meta-modelo y matrices del meta (porte fiel de phase2/phase3/phase7/phase25 + v19).

Ordenes EXACTOS:
- MEMBER_FILES (14 miembros, 78 columnas) + densas (35) + estructuradas (87) + supervisadas (73) = 273.
- matriz 279 (train) = 273 + ADV07 (6).  ASCII puro. Sin efectos en import.
"""

from __future__ import annotations

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler


#: Miembros y transformacion aplicada al construir el meta (scripts/phase2_ponderacion.py:49-55).
MEMBER_FILES = [("e013", "raw"), ("e016", "raw"), ("e009", "log"), ("e011", "log"),
                ("e010", "log"), ("e014", "raw"), ("e015_ordinal", "raw"),
                ("e015_softmax", "log"), ("e017", "log"), ("e019b", "log"),
                ("e021a", "log"), ("e021b", "log"), ("e028a", "raw"), ("adv05", "raw")]

#: Columnas densas usadas en el meta (35 de 38): se excluyen las 3 de emoji.
DENSE_EXCLUIDAS = {"n_emoji", "emoji_diversity", "has_emoji"}


class EncodedMLP:
    """MLP con etiquetas codificadas a enteros (scripts/phase2_ponderacion.py:106-125)."""

    def __init__(self) -> None:
        self.encoder = LabelEncoder()
        self.scaler = StandardScaler()
        self.model = MLPClassifier(hidden_layer_sizes=(64,), alpha=1e-3, batch_size=512,
                                   learning_rate_init=1e-3, max_iter=40, early_stopping=True,
                                   n_iter_no_change=5, random_state=SEED)

    def fit(self, features: np.ndarray, labels: np.ndarray) -> EncodedMLP:
        self.encoder.fit(labels)
        self.model.fit(self.scaler.fit_transform(features), self.encoder.transform(labels))
        return self

    def predict(self, features: np.ndarray) -> np.ndarray:
        return self.encoder.inverse_transform(self.model.predict(self.scaler.transform(features)))

    def predict_proba(self, features: np.ndarray) -> np.ndarray:
        crudo = self.model.predict_proba(self.scaler.transform(features))
        completa = np.zeros((len(features), len(CLASSES)))
        for posicion, codigo in enumerate(self.model.classes_):
            completa[:, int(codigo)] = crudo[:, posicion]
        return completa


def logreg() -> Pipeline:
    """LogReg del proyecto (scripts/v13_p1b_fusion_tokens.py:79-83)."""
    return Pipeline([("scaler", StandardScaler()),
                     ("clf", LogisticRegression(C=1.0, max_iter=3000, solver="lbfgs",
                                                random_state=SEED))])


def construir_miembros_oof(textos: pd.Series, y: np.ndarray, fold_idx: np.ndarray,
                           dense: pd.DataFrame) -> dict[str, np.ndarray]:
    """Reconstruye los 14 OOF de miembros (formas y dtypes de los artefactos guardados)."""
    return {
        "e013": oof_e013(textos, y, fold_idx),
        "e016": oof_e016(textos, y, fold_idx),
        "e009": oof_e009(textos, y, fold_idx),
        "e011": oof_e011(textos, y, fold_idx),
        "e010": oof_e010(textos, y, fold_idx),
        "e014": oof_e014(textos, y, fold_idx),
        "e015_ordinal": oof_e015_ordinal(textos, y, fold_idx),
        "e015_softmax": oof_e015_softmax(textos, y, fold_idx),
        "e017": oof_e017(textos, y, fold_idx),
        "e019b": oof_e019b(textos, y, fold_idx, dense),
        "e021a": oof_e021a(textos, y, fold_idx),
        "e021b": oof_e021b(textos, y, fold_idx),
        "e028a": oof_e028a(textos, y, fold_idx),
        "adv05": oof_adv05(textos, y, fold_idx),
    }


def build_features_273(oofs: dict[str, np.ndarray], densas35: np.ndarray,
                       estructuradas: np.ndarray, supervisadas: np.ndarray) -> np.ndarray:
    """Concatenacion exacta del meta (scripts/phase2_ponderacion.py:61-81)."""
    partes = []
    for nombre, transform in MEMBER_FILES:
        matriz = oofs[nombre]
        if matriz.ndim == 1:
            matriz = matriz.reshape(-1, 1)
        partes.append(np.log(np.clip(matriz, 1e-12, None)) if transform == "log" else matriz)
    features = np.concatenate([np.concatenate(partes, axis=1),
                               densas35, estructuradas, supervisadas], axis=1)
    assert features.shape[1] == 273, features.shape
    return features


def matriz_279(meta273: np.ndarray, adv07: np.ndarray) -> np.ndarray:
    """Matriz 279 del campeon (scripts/v19_b1_boosting.py:97-108)."""
    matriz = np.concatenate([meta273.astype(np.float32), adv07.astype(np.float32)], axis=1)
    assert matriz.shape[1] == 279, matriz.shape
    return matriz


def meta_v12_oof(meta273: np.ndarray, y: np.ndarray, fold_idx: np.ndarray) -> np.ndarray:
    """OOF del combinador V12: 0.5*(LogReg + EncodedMLP) sobre 273 (scripts/phase8:80-107)."""
    n = len(y)
    salida = np.zeros((n, len(CLASSES)))
    for fold in range(5):
        tr = fold_idx != fold
        va = fold_idx == fold
        lr = logreg().fit(meta273[tr], y[tr])
        red = EncodedMLP().fit(meta273[tr], y[tr])
        p_lr = np.zeros((int(va.sum()), len(CLASSES)))
        for posicion, etiqueta in enumerate(lr.classes_):
            p_lr[:, CLASSES.index(str(etiqueta))] = lr.predict_proba(meta273[va])[:, posicion]
        p_red = red.predict_proba(meta273[va])
        salida[va] = 0.5 * (p_lr + p_red)
    return salida


def densas35(dense: pd.DataFrame) -> np.ndarray:
    """Las 35 columnas densas usadas en el meta (excluye las 3 de emoji)."""
    columnas = [c for c in dense.columns if c not in DENSE_EXCLUIDAS]
    return dense[columnas].to_numpy(float)

### Módulo `v30.py`

In [ ]:
"""V19/V20 (cuatro cuartos), ensemble V30 y regla plug-in (porte fiel de v19_b1/v20_f3/v30_03/age_plugin).

Configuraciones congeladas embebidas como constantes del protocolo (no se leen de ficheros):
- Pesos del ensemble V30: media de los vectores Optuna por fold, renormalizada (v30_02_ensemble.json).
- Configs V29 (reports/v29-01_trials.csv) y V30 (v30_01_optuna_refinamiento.json), en orden de fold.
- Pesos plug-in congelados (age_umbrales_validacion.json, commit bd08aba).
ASCII puro. Sin efectos en import.
"""

from __future__ import annotations

import lightgbm as lgb
import numpy as np


#: Arbol pre-registrado del proyecto (scripts/v19_b1_boosting.py:83-86).
HIERBA = {"objective": "multiclass", "num_class": len(CLASSES), "n_estimators": 300,
          "learning_rate": 0.05, "num_leaves": 8, "max_depth": 3, "min_child_samples": 200,
          "colsample_bytree": 0.5, "subsample": 0.8, "subsample_freq": 1, "reg_lambda": 1.0,
          "n_jobs": 32, "random_state": SEED, "verbose": -1}

#: Pesos del ensemble V30 (v30_02_ensemble.json -> manifest): v20, v29, v30.
PESOS_ENSEMBLE = (0.3548, 0.3354, 0.3098)

#: Pesos plug-in congelados (age_plugin_submission.py:79).
PESOS_PLUG_IN = np.array([1.0000, 1.1749, 1.2721, 1.1630, 1.0010, 0.8128])

#: Mejor trial por fold de V29 (reports/v29-01_trials.csv), en orden de fold.
V29_CONFIGS = [
    {"num_leaves": 87, "max_depth": 5, "learning_rate": 0.04301669500288421,
     "min_child_samples": 76, "subsample": 0.6581570543882859,
     "colsample_bytree": 0.70766450002428, "reg_alpha": 0.6147385859030499,
     "reg_lambda": 9.792145504680492e-05, "n_estimators": 816},
    {"num_leaves": 55, "max_depth": 6, "learning_rate": 0.03352698655336303,
     "min_child_samples": 54, "subsample": 0.8941120537508466,
     "colsample_bytree": 0.6299160105578635, "reg_alpha": 4.026902217415037e-07,
     "reg_lambda": 6.6850377528309e-07, "n_estimators": 1372},
    {"num_leaves": 57, "max_depth": 8, "learning_rate": 0.022124342973521553,
     "min_child_samples": 94, "subsample": 0.9225205673898446,
     "colsample_bytree": 0.52058407816027, "reg_alpha": 1.8772169842126896,
     "reg_lambda": 0.018194045769254683, "n_estimators": 912},
    {"num_leaves": 43, "max_depth": 9, "learning_rate": 0.03000286836410436,
     "min_child_samples": 22, "subsample": 0.6998254389554196,
     "colsample_bytree": 0.5250455092128341, "reg_alpha": 1.0090819656249398e-05,
     "reg_lambda": 7.87982213180386e-06, "n_estimators": 863},
    {"num_leaves": 56, "max_depth": 7, "learning_rate": 0.020811077189914432,
     "min_child_samples": 29, "subsample": 0.7762118181212365,
     "colsample_bytree": 0.6242271327287489, "reg_alpha": 1.3833851148235308e-05,
     "reg_lambda": 0.0004885693686069363, "n_estimators": 1084},
]

#: Mejor trial por fold de V30 (v30_01_optuna_refinamiento.json), en orden de fold.
V30_CONFIGS = [
    {"num_leaves": 41, "max_depth": 10, "learning_rate": 0.015612823179143214,
     "min_child_samples": 79, "subsample": 0.775913545060191,
     "colsample_bytree": 0.5969526009433974, "reg_alpha": 1.0941449983677585e-05,
     "reg_lambda": 0.00036896192791107643, "n_estimators": 1011},
    {"num_leaves": 32, "max_depth": 7, "learning_rate": 0.017387326275059966,
     "min_child_samples": 23, "subsample": 0.8215355497227395,
     "colsample_bytree": 0.8291548707315466, "reg_alpha": 0.8747434883808647,
     "reg_lambda": 2.053077365511584e-05, "n_estimators": 1486},
    {"num_leaves": 63, "max_depth": 7, "learning_rate": 0.02867292541012883,
     "min_child_samples": 110, "subsample": 0.7807106452264457,
     "colsample_bytree": 0.4021908926823267, "reg_alpha": 1.1486360263326448e-05,
     "reg_lambda": 0.00031823134119350724, "n_estimators": 1023},
    {"num_leaves": 42, "max_depth": 7, "learning_rate": 0.015729268043126833,
     "min_child_samples": 41, "subsample": 0.999685646683389,
     "colsample_bytree": 0.596195315949217, "reg_alpha": 0.004978399116554409,
     "reg_lambda": 1.8436108646384413e-08, "n_estimators": 1477},
    {"num_leaves": 64, "max_depth": 6, "learning_rate": 0.03818854692388361,
     "min_child_samples": 52, "subsample": 0.8731483359860639,
     "colsample_bytree": 0.46934472778342573, "reg_alpha": 1.927060050589197e-08,
     "reg_lambda": 0.004339321331656464, "n_estimators": 1000},
]

PARAMETROS_ENTEROS = ("num_leaves", "max_depth", "min_child_samples", "subsample_freq",
                      "n_estimators")


def a_clases(probabilidad: np.ndarray, clases, filas: int) -> np.ndarray:
    """Reordena columnas al orden canonico (v30_03_submission.py:72-90)."""
    salida = np.zeros((filas, len(CLASSES)))
    existentes = list(clases)
    son_enteros = all(isinstance(v, (int, np.integer)) for v in existentes)
    for destino, etiqueta in enumerate(CLASSES):
        buscar = destino if son_enteros else etiqueta
        salida[:, destino] = probabilidad[:, existentes.index(buscar)]
    return salida


def _normaliza(params: dict) -> dict:
    """Convierte a int los parametros enteros de LightGBM (v30_03_submission.py:139-146)."""
    salida = dict(params)
    for clave in PARAMETROS_ENTEROS:
        if clave in salida and salida[clave] is not None:
            salida[clave] = int(round(float(salida[clave])))
    return salida


def _arbol(semilla: int = SEED) -> lgb.LGBMClassifier:
    """Arbol HIERBA con semilla (scripts/v19_b1_boosting.py:110-115)."""
    return lgb.LGBMClassifier(**{**HIERBA, "random_state": semilla})


def v19_oofs(matriz: np.ndarray, y: np.ndarray, clase: np.ndarray, fold_idx: np.ndarray
             ) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """OOF de los tres miembros de V19 (logreg, mlp, arbol) sobre la matriz 279."""
    n = len(y)
    p_lr = np.zeros((n, 6))
    p_mlp = np.zeros((n, 6))
    p_arbol = np.zeros((n, 6))
    for fold in range(5):
        tr = fold_idx != fold
        va = fold_idx == fold
        lr = logreg().fit(matriz[tr], y[tr])
        completa = np.zeros((int(va.sum()), 6))
        for posicion, nombre in enumerate(lr.classes_):
            completa[:, CLASSES.index(str(nombre))] = lr.predict_proba(matriz[va])[:, posicion]
        p_lr[va] = completa
        red = EncodedMLP().fit(matriz[tr], y[tr])
        p_mlp[va] = red.predict_proba(matriz[va])
        modelo = _arbol(SEED).fit(matriz[tr], y[tr])
        p_arbol[va] = a_clases(modelo.predict_proba(matriz[va]), modelo.classes_, int(va.sum()))
    return p_lr, p_mlp, p_arbol


def v20_cadena_oof(matriz: np.ndarray, clase: np.ndarray, fold_idx: np.ndarray) -> np.ndarray:
    """Cadena de 5 umbrales anidados, OOF (scripts/v20_f3_umbrales.py:145-172 + 196-206)."""
    n = len(clase)
    n_clases = len(CLASSES)
    acumuladas = np.zeros((n, n_clases - 1))
    for fold in range(5):
        tr = fold_idx != fold
        va = fold_idx == fold
        for k in range(1, n_clases):
            objetivo = (clase <= k - 1).astype(int)
            modelo = lgb.LGBMClassifier(**HIERBA)
            modelo.fit(matriz[tr], objetivo[tr])
            acumuladas[va, k - 1] = modelo.predict_proba(matriz[va])[:, 1]
    for k in range(1, n_clases - 1):
        acumuladas[:, k] = np.maximum(acumuladas[:, k], acumuladas[:, k - 1])
    seis = np.zeros((n, n_clases))
    seis[:, 0] = acumuladas[:, 0]
    for k in range(1, n_clases - 1):
        seis[:, k] = acumuladas[:, k] - acumuladas[:, k - 1]
    seis[:, n_clases - 1] = 1.0 - acumuladas[:, n_clases - 2]
    seis = np.clip(seis, 0.0, None)
    return seis / np.clip(seis.sum(axis=1, keepdims=True), 1e-9, None)


def v20_oof(matriz: np.ndarray, y: np.ndarray, clase: np.ndarray, fold_idx: np.ndarray
            ) -> np.ndarray:
    """OOF del candidato V20: 0.25 de cada cuarto (v20_f3_umbrales.py:207-213)."""
    p_lr, p_mlp, p_arbol = v19_oofs(matriz, y, clase, fold_idx)
    seis = v20_cadena_oof(matriz, clase, fold_idx)
    return 0.25 * p_lr + 0.25 * p_mlp + 0.25 * p_arbol + 0.25 * seis


def v20_test(matriz: np.ndarray, matriz_test: np.ndarray, y: np.ndarray, clase: np.ndarray
             ) -> np.ndarray:
    """Receta exacta de V20 sobre el train completo (v30_03_submission.py:93-117)."""
    n_test = matriz_test.shape[0]
    modelo_lr = logreg().fit(matriz, y)
    prob_lr = a_clases(modelo_lr.predict_proba(matriz_test), modelo_lr.classes_, n_test)
    red = EncodedMLP().fit(matriz, y)
    prob_mlp = a_clases(red.model.predict_proba(red.scaler.transform(matriz_test)),
                        red.encoder.classes_, n_test)
    modelo_arbol = lgb.LGBMClassifier(**HIERBA).fit(matriz, y)
    prob_arbol = a_clases(modelo_arbol.predict_proba(matriz_test), modelo_arbol.classes_, n_test)
    acumuladas = np.zeros((n_test, len(CLASSES) - 1))
    for k in range(1, len(CLASSES)):
        objetivo = (clase <= k - 1).astype(int)
        modelo = lgb.LGBMClassifier(**HIERBA).fit(matriz, objetivo)
        acumuladas[:, k - 1] = modelo.predict_proba(matriz_test)[:, 1]
    for k in range(1, len(CLASSES) - 1):
        acumuladas[:, k] = np.maximum(acumuladas[:, k], acumuladas[:, k - 1])
    seis = np.zeros((n_test, len(CLASSES)))
    seis[:, 0] = acumuladas[:, 0]
    for k in range(1, len(CLASSES) - 1):
        seis[:, k] = acumuladas[:, k] - acumuladas[:, k - 1]
    seis[:, len(CLASSES) - 1] = 1.0 - acumuladas[:, len(CLASSES) - 2]
    seis = np.clip(seis, 0.0, None)
    seis = seis / np.clip(seis.sum(axis=1, keepdims=True), 1e-9, None)
    return 0.25 * prob_lr + 0.25 * prob_mlp + 0.25 * prob_arbol + 0.25 * seis


def lgbm_refit_oof(matriz: np.ndarray, clase: np.ndarray, fold_idx: np.ndarray,
                   configs: list[dict], base: dict | None = None) -> np.ndarray:
    """OOF de un LightGBM con la configuracion congelada por fold y early stopping del original.

    Porte de ``scripts/v29_04_optuna_individual.py:243-266`` (``_entrena``, rama lightgbm; la
    base incluye ``random_state=SEED``) y ``scripts/v30_01_optuna_refinamiento.py:164-176``
    (``_entrena``; la base NO incluye ``random_state``). El early stopping (60 rondas de
    ``multi_logloss`` sobre el fold retenido) es parte del protocolo original y el corte
    ocurrio de hecho en los reajustes documentados.

    Nota de idempotencia: se anaden ``deterministic=True`` y ``force_col_wise=True`` porque la
    construccion multihilo de histogramas de LightGBM no es estable entre procesos con la
    configuracion original (medido con un probe de dos procesos: el OOF reconstruido cambiaba);
    con ambos parametros dos corridas completas reproducen el mismo resultado bit a bit.
    """
    n = len(clase)
    oof = np.zeros((n, len(CLASSES)))
    for fold in range(5):
        tr = fold_idx != fold
        va = fold_idx == fold
        params = {"objective": "multiclass", "num_class": len(CLASSES), "verbose": -1,
                  "n_jobs": 32, "subsample_freq": 1, "deterministic": True,
                  "force_col_wise": True, **(base or {}),
                  **_normaliza(configs[fold])}
        techo = int(params.pop("n_estimators"))
        modelo = lgb.LGBMClassifier(**params, n_estimators=techo)
        modelo.fit(matriz[tr], clase[tr],
                   eval_set=[(matriz[va], clase[va])], eval_metric="multi_logloss",
                   callbacks=[lgb.early_stopping(60, verbose=False)])
        oof[va] = a_clases(modelo.predict_proba(matriz[va]), modelo.classes_, int(va.sum()))
    return oof


def lgbm_refit_test(matriz: np.ndarray, clase: np.ndarray, matriz_test: np.ndarray,
                    configs: list[dict]) -> np.ndarray:
    """Cinco reajustes (uno por configuracion congelada) sobre todo el train, promediados.

    Incluye ``deterministic=True`` y ``force_col_wise=True`` por la misma razon que
    ``lgbm_refit_oof`` (estabilidad entre procesos de la construccion de histogramas).
    """
    n_test = matriz_test.shape[0]
    predicciones = []
    for params in configs:
        p = {"objective": "multiclass", "num_class": len(CLASSES), "verbose": -1,
             "n_jobs": 28, "subsample_freq": 1, "deterministic": True,
             "force_col_wise": True, **_normaliza(params)}
        techo = int(p.pop("n_estimators"))
        modelo = lgb.LGBMClassifier(**p, n_estimators=techo)
        modelo.fit(matriz, clase)
        predicciones.append(a_clases(modelo.predict_proba(matriz_test), modelo.classes_, n_test))
    return np.mean(predicciones, axis=0)


def mezcla(p20: np.ndarray, p29: np.ndarray, p30: np.ndarray) -> np.ndarray:
    """Mezcla del ensemble V30 con los pesos congelados (v30_03_submission.py:232-240)."""
    vector = np.array(PESOS_ENSEMBLE)
    vector = vector / vector.sum()
    return np.tensordot(vector, np.stack([p20, p29, p30]), axes=(0, 0))


def regla_plug_in(prob: np.ndarray) -> np.ndarray:
    """Decision plug-in por clase: argmax(prob * PESOS_PLUG_IN) (age_plugin_submission.py:258-261)."""
    return np.argmax(prob * PESOS_PLUG_IN, axis=1)


def macro_f1(clase: np.ndarray, pred: np.ndarray) -> float:
    """Macro-F1 sobre indices 0-5 (sin dependencia de sklearn para el chequeo rapido)."""
    from sklearn.metrics import f1_score
    return float(f1_score(clase, pred, average="macro", labels=list(range(len(CLASSES)))))


def _cuartos_v20_fold(matriz: np.ndarray, clase: np.ndarray, y: np.ndarray,
                      tren: np.ndarray, val: np.ndarray) -> np.ndarray:
    """Los cuatro cuartos de V20 sobre `tren`, medidos en `val` (v30_05:146-175)."""
    prob_lr = a_clases(logreg().fit(matriz[tren], y[tren]).predict_proba(matriz[val]),
                       np.array(CLASSES), len(val))
    red = EncodedMLP().fit(matriz[tren], y[tren])
    prob_mlp = a_clases(red.model.predict_proba(red.scaler.transform(matriz[val])),
                        red.encoder.classes_, len(val))
    prob_arbol = a_clases(lgb.LGBMClassifier(**HIERBA).fit(matriz[tren], y[tren])
                          .predict_proba(matriz[val]), np.array(CLASSES), len(val))
    acumuladas = np.zeros((len(val), len(CLASSES) - 1))
    for k in range(1, len(CLASSES)):
        objetivo = (clase <= k - 1).astype(int)
        modelo = lgb.LGBMClassifier(**HIERBA).fit(matriz[tren], objetivo[tren])
        acumuladas[:, k - 1] = modelo.predict_proba(matriz[val])[:, 1]
    for k in range(1, len(CLASSES) - 1):
        acumuladas[:, k] = np.maximum(acumuladas[:, k], acumuladas[:, k - 1])
    seis = np.zeros((len(val), len(CLASSES)))
    seis[:, 0] = acumuladas[:, 0]
    for k in range(1, len(CLASSES) - 1):
        seis[:, k] = acumuladas[:, k] - acumuladas[:, k - 1]
    seis[:, len(CLASSES) - 1] = 1.0 - acumuladas[:, len(CLASSES) - 2]
    seis = np.clip(seis, 0.0, None)
    seis = seis / np.clip(seis.sum(axis=1, keepdims=True), 1e-9, None)
    return 0.25 * prob_lr + 0.25 * prob_mlp + 0.25 * prob_arbol + 0.25 * seis


def cuartos_v20_oof(matriz: np.ndarray, y: np.ndarray, clase: np.ndarray,
                    fold_idx: np.ndarray) -> np.ndarray:
    """OOF de los cuatro cuartos por fold (protocolo de la segunda particion, v30_05:146-175)."""
    n = len(clase)
    oof = np.zeros((n, len(CLASSES)))
    for fold in range(5):
        tren = np.flatnonzero(fold_idx != fold)
        val = np.flatnonzero(fold_idx == fold)
        oof[val] = _cuartos_v20_fold(matriz, clase, y, tren, val)
    return oof


def lgbm_refit_oof_sin_es(matriz: np.ndarray, clase: np.ndarray, y: np.ndarray,
                          fold_idx: np.ndarray, configs: list[dict]) -> np.ndarray:
    """Refit del techo congelado SIN early stopping (age_plugin_confirmacion.py:141-147).

    Incluye ``deterministic=True`` y ``force_col_wise=True``: sin ellos este reajuste NO era
    estable entre procesos (medido con el probe de determinismo), y es el miembro V30 de la
    segunda particion y de la evaluacion publicada del campeon.
    """
    n = len(clase)
    oof = np.zeros((n, len(CLASSES)))
    for fold in range(5):
        tr = fold_idx != fold
        va = fold_idx == fold
        params = {"objective": "multiclass", "num_class": len(CLASSES), "verbose": -1,
                  "n_jobs": 28, "subsample_freq": 1, "deterministic": True,
                  "force_col_wise": True, **_normaliza(configs[fold])}
        techo = int(params.pop("n_estimators"))
        modelo = lgb.LGBMClassifier(**params, n_estimators=techo).fit(matriz[tr], y[tr])
        oof[va] = a_clases(modelo.predict_proba(matriz[va]), modelo.classes_, int(va.sum()))
    return oof


def v30_oof_segunda(matriz: np.ndarray, y: np.ndarray, clase: np.ndarray,
                    fold_idx2: np.ndarray) -> np.ndarray:
    """V30 OOF en la particion alternativa (age_plugin_confirmacion.py:126-156).

    Misma receta y pesos congelados, con los reajustes del techo SIN early stopping (protocolo
    propio de la segunda particion) y los cuatro cuartos por fold.
    """
    p20 = cuartos_v20_oof(matriz, y, clase, fold_idx2)
    p29 = lgbm_refit_oof_sin_es(matriz, clase, y, fold_idx2, V29_CONFIGS)
    p30 = lgbm_refit_oof_sin_es(matriz, clase, y, fold_idx2, V30_CONFIGS)
    return mezcla(p20, p29, p30)

### Módulo `neuronales.py`

In [ ]:
"""Componentes neuronales V38-V53 + HGB (porte fiel de scripts/v38_01..v53_01).

Arquitecturas, perdidas, semillas, SWA y estrategias de test EXACTAS de cada script original:
- V38/V39: TabularResNet, seeds 42+fold, 14 epocas, Cosine(T_max=14), sin SWA; test = modelo full-train.
- V40: SETabularResNet, OrdinalDistanceLoss, seeds 42+fold*100, 12 epocas, sin scheduler, sin SWA; test = media/5.
- V41: SETabularResNet, OrdinalDistanceLoss, 16 epocas, sin scheduler, SWA swa_start=8; test = media/5.
- V42: SETabularResNet, SmoothOrdinalDistanceLoss, 16 epocas, Cosine(T_max=16, eta_min=1e-4), SWA swa_start=6; test = media/5.
- V43: SETabularResNet, CramerDistanceLoss, 16 epocas, Cosine(T_max=16, eta_min=1e-4), SWA swa_start=6; test = media/5.
- V47: SETabularResNet, WeightedCramerLoss (gamma por frontera), recorte Z [-6,+6], seeds 42+fold*100,
  16 epocas, Cosine(T_max=16, eta_min=1e-4), SWA swa_start=6 con update_bn en loader 512 sin barajar; test = media/5.
- V51: SETabularResNet, ConsistentSmoothedCramerLoss, 16 epocas, Cosine(T_max=16, eta_min=1e-4), SWA swa_start=6; test = media/5.
- V53: TabularDenseNet (init 192, growth 128) con SEChannelBlock, ConsistentSmoothedCramerLoss, recorte Z [-6,+6],
  16 epocas, Cosine(T_max=16, eta_min=1e-4), SWA swa_start=6; test = media/5.
- V39-HGB: HistGradientBoostingClassifier por fold; test = modelo full-train.
ASCII puro. Sin efectos en import.
"""

from __future__ import annotations

import numpy as np
import torch
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from torch import nn, optim
from torch.optim.swa_utils import AveragedModel, update_bn
from torch.utils.data import DataLoader, TensorDataset


#: Configuracion exacta por componente (pack 04; los scripts originales mandan).
CONFIGS: dict[str, dict] = {
    "v38": {"modelo": "tabular", "loss": "ce", "epocas": 14, "batch": 256, "lr": 1e-3,
            "wd": 1e-4, "coseno": True, "eta_min": 0.0, "swa_start": None,
            "seed": "42+fold", "test": "full"},
    "v39_ordinal": {"modelo": "tabular", "loss": "ordinal_ce", "epocas": 14, "batch": 256,
                    "lr": 1e-3, "wd": 1e-4, "coseno": True, "eta_min": 0.0, "swa_start": None,
                    "seed": "42+fold", "test": "full"},
    "v40": {"modelo": "se", "loss": "ordinal_dist", "epocas": 12, "batch": 256, "lr": 1e-3,
            "wd": 1e-4, "coseno": False, "eta_min": 0.0, "swa_start": None,
            "seed": "42+fold*100", "test": "media"},
    "v41": {"modelo": "se", "loss": "ordinal_dist", "epocas": 16, "batch": 256, "lr": 1e-3,
            "wd": 1e-4, "coseno": False, "eta_min": 0.0, "swa_start": 8,
            "seed": "42+fold*100", "test": "media"},
    "v42": {"modelo": "se", "loss": "smooth_ordinal", "epocas": 16, "batch": 256, "lr": 1e-3,
            "wd": 1e-4, "coseno": True, "eta_min": 1e-4, "swa_start": 6,
            "seed": "42+fold*100", "test": "media"},
    "v43": {"modelo": "se", "loss": "cramer", "epocas": 16, "batch": 256, "lr": 1e-3,
            "wd": 1e-4, "coseno": True, "eta_min": 1e-4, "swa_start": 6,
            "seed": "42+fold*100", "test": "media"},
    "v47": {"modelo": "se", "loss": "weighted_cramer", "epocas": 16, "batch": 256, "lr": 1e-3,
            "wd": 1e-4, "coseno": True, "eta_min": 1e-4, "swa_start": 6,
            "seed": "42+fold*100", "test": "media", "clip": 6.0, "bn_batch": 512},
    "v51": {"modelo": "se", "loss": "consistent_cramer", "epocas": 16, "batch": 256, "lr": 1e-3,
            "wd": 1e-4, "coseno": True, "eta_min": 1e-4, "swa_start": 6,
            "seed": "42+fold*100", "test": "media"},
    "v53": {"modelo": "dense_se", "loss": "consistent_cramer", "epocas": 16, "batch": 256,
            "lr": 1e-3, "wd": 1e-4, "coseno": True, "eta_min": 1e-4, "swa_start": 6,
            "seed": "42+fold*100", "test": "media", "clip": 6.0},
}


class TabularResNet(nn.Module):
    """Red residual tabular (scripts/v38_01_tabularnet_5folds.py:44-71)."""

    def __init__(self, in_features: int = 279, hidden: int = 256, num_classes: int = 6,
                 drop: float = 0.2):
        super().__init__()
        self.in_proj = nn.Linear(in_features, hidden)
        self.bn0 = nn.BatchNorm1d(hidden)
        self.act0 = nn.SiLU()
        self.fc1 = nn.Linear(hidden, hidden)
        self.bn1 = nn.BatchNorm1d(hidden)
        self.act1 = nn.SiLU()
        self.drop1 = nn.Dropout(drop)
        self.fc2 = nn.Linear(hidden, hidden)
        self.bn2 = nn.BatchNorm1d(hidden)
        self.act2 = nn.SiLU()
        self.drop2 = nn.Dropout(drop)
        self.head = nn.Linear(hidden, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h0 = self.act0(self.bn0(self.in_proj(x)))
        h1 = h0 + self.drop1(self.act1(self.bn1(self.fc1(h0))))
        h2 = h1 + self.drop2(self.act2(self.bn2(self.fc2(h1))))
        return self.head(h2)


class SETabularResNet(nn.Module):
    """Red residual con compuertas Squeeze-and-Excitation (scripts/v40_01..v43_01)."""

    def __init__(self, in_features: int = 279, hidden: int = 256, num_classes: int = 6,
                 drop: float = 0.2, reduction: int = 4):
        super().__init__()
        self.in_proj = nn.Linear(in_features, hidden)
        self.bn0 = nn.BatchNorm1d(hidden)
        self.act0 = nn.SiLU()
        self.fc1 = nn.Linear(hidden, hidden)
        self.bn1 = nn.BatchNorm1d(hidden)
        self.act1 = nn.SiLU()
        self.drop1 = nn.Dropout(drop)
        self.se1_fc1 = nn.Linear(hidden, hidden // reduction)
        self.se1_act = nn.SiLU()
        self.se1_fc2 = nn.Linear(hidden // reduction, hidden)
        self.se1_sig = nn.Sigmoid()
        self.fc2 = nn.Linear(hidden, hidden)
        self.bn2 = nn.BatchNorm1d(hidden)
        self.act2 = nn.SiLU()
        self.drop2 = nn.Dropout(drop)
        self.se2_fc1 = nn.Linear(hidden, hidden // reduction)
        self.se2_act = nn.SiLU()
        self.se2_fc2 = nn.Linear(hidden // reduction, hidden)
        self.se2_sig = nn.Sigmoid()
        self.head = nn.Linear(hidden, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h0 = self.act0(self.bn0(self.in_proj(x)))
        res1 = self.drop1(self.act1(self.bn1(self.fc1(h0))))
        se1 = self.se1_sig(self.se1_fc2(self.se1_act(self.se1_fc1(res1))))
        h1 = h0 + res1 * se1
        res2 = self.drop2(self.act2(self.bn2(self.fc2(h1))))
        se2 = self.se2_sig(self.se2_fc2(self.se2_act(self.se2_fc1(res2))))
        h2 = h1 + res2 * se2
        return self.head(h2)


class SEChannelBlock(nn.Module):
    """Compuerta Squeeze-and-Excitation por canal (scripts/v53_01_dense_se_tabularnet_5folds.py:65-77)."""

    def __init__(self, channels: int, reduction: int = 4):
        super().__init__()
        self.fc1 = nn.Linear(channels, channels // reduction)
        self.act = nn.SiLU()
        self.fc2 = nn.Linear(channels // reduction, channels)
        self.sig = nn.Sigmoid()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        w = self.sig(self.fc2(self.act(self.fc1(x))))
        return x * w


class TabularDenseNet(nn.Module):
    """Red densa tabular con concatenacion multi-escala y compuertas SE (scripts/v53_01:80-142)."""

    def __init__(self, in_features: int = 279, init_dim: int = 192, growth_rate: int = 128,
                 num_classes: int = 6, drop: float = 0.20):
        super().__init__()
        self.in_proj = nn.Linear(in_features, init_dim)
        self.bn0 = nn.BatchNorm1d(init_dim)
        self.act0 = nn.SiLU()
        self.bn1 = nn.BatchNorm1d(init_dim)
        self.fc1 = nn.Linear(init_dim, growth_rate)
        self.act1 = nn.SiLU()
        self.drop1 = nn.Dropout(drop)
        self.se1 = SEChannelBlock(growth_rate, reduction=4)
        in_dim2 = init_dim + growth_rate
        self.bn2 = nn.BatchNorm1d(in_dim2)
        self.fc2 = nn.Linear(in_dim2, growth_rate)
        self.act2 = nn.SiLU()
        self.drop2 = nn.Dropout(drop)
        self.se2 = SEChannelBlock(growth_rate, reduction=4)
        in_dim3 = in_dim2 + growth_rate
        self.bn3 = nn.BatchNorm1d(in_dim3)
        self.fc3 = nn.Linear(in_dim3, growth_rate)
        self.act3 = nn.SiLU()
        self.drop3 = nn.Dropout(drop)
        self.se3 = SEChannelBlock(growth_rate, reduction=4)
        total_dense_dim = in_dim3 + growth_rate
        self.bn_trans = nn.BatchNorm1d(total_dense_dim)
        self.fc_trans = nn.Linear(total_dense_dim, 256)
        self.act_trans = nn.SiLU()
        self.drop_trans = nn.Dropout(0.10)
        self.head = nn.Linear(256, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h0 = self.act0(self.bn0(self.in_proj(x)))
        z1 = self.se1(self.drop1(self.act1(self.fc1(self.bn1(h0)))))
        h1 = torch.cat([h0, z1], dim=1)
        z2 = self.se2(self.drop2(self.act2(self.fc2(self.bn2(h1)))))
        h2 = torch.cat([h1, z2], dim=1)
        z3 = self.se3(self.drop3(self.act3(self.fc3(self.bn3(h2)))))
        h3 = torch.cat([h2, z3], dim=1)
        ht = self.drop_trans(self.act_trans(self.fc_trans(self.bn_trans(h3))))
        return self.head(ht)


class OrdinalCrossEntropyLoss(nn.Module):
    """CE + distancia ordinal esperada (scripts/v39_01: OrdinalCrossEntropyLoss)."""

    def __init__(self, num_classes: int = 6, lambda_dist: float = 0.2):
        super().__init__()
        self.ce = nn.CrossEntropyLoss()
        self.lambda_dist = lambda_dist
        clases = torch.arange(num_classes, dtype=torch.float32)
        dist_mat = torch.abs(clases.unsqueeze(0) - clases.unsqueeze(1))
        self.register_buffer("dist_mat", dist_mat)

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        loss_ce = self.ce(logits, targets)
        probs = torch.softmax(logits, dim=1)
        batch_dists = self.dist_mat[targets]
        dist_loss = torch.sum(probs * batch_dists, dim=1).mean()
        return loss_ce + self.lambda_dist * dist_loss


class OrdinalDistanceLoss(nn.Module):
    """CE + distancia ordinal esperada (scripts/v40_01/v41_01)."""

    def __init__(self, num_classes: int = 6, lambda_dist: float = 0.2):
        super().__init__()
        self.ce = nn.CrossEntropyLoss()
        self.lambda_dist = lambda_dist
        c = torch.arange(num_classes).float()
        self.register_buffer("dist_matrix", torch.abs(c.unsqueeze(0) - c.unsqueeze(1)))

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        loss_ce = self.ce(logits, targets)
        probs = torch.softmax(logits, dim=1)
        dists = self.dist_matrix[targets]
        loss_dist = torch.sum(probs * dists, dim=1).mean()
        return loss_ce + self.lambda_dist * loss_dist


class SmoothOrdinalDistanceLoss(nn.Module):
    """CE con suavizado 0.10 + distancia ordinal (scripts/v42_01)."""

    def __init__(self, num_classes: int = 6, lambda_dist: float = 0.2, smoothing: float = 0.10):
        super().__init__()
        self.ce = nn.CrossEntropyLoss(label_smoothing=smoothing)
        self.lambda_dist = lambda_dist
        c = torch.arange(num_classes).float()
        self.register_buffer("dist_matrix", torch.abs(c.unsqueeze(0) - c.unsqueeze(1)))

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        loss_ce = self.ce(logits, targets)
        probs = torch.softmax(logits, dim=1)
        dists = self.dist_matrix[targets]
        loss_dist = torch.sum(probs * dists, dim=1).mean()
        return loss_ce + self.lambda_dist * loss_dist


class CramerDistanceLoss(nn.Module):
    """CE (suavizado 0.10) + Cramer-von Mises sobre la CDF (scripts/v43_01)."""

    def __init__(self, num_classes: int = 6, lambda_cramer: float = 0.50, smoothing: float = 0.10):
        super().__init__()
        self.ce = nn.CrossEntropyLoss(label_smoothing=smoothing)
        self.num_classes = num_classes
        self.lambda_cramer = lambda_cramer

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        loss_ce = self.ce(logits, targets)
        probs = torch.softmax(logits, dim=1)
        cdf_pred = torch.cumsum(probs, dim=1)[:, :-1]
        k_indices = torch.arange(self.num_classes - 1, device=logits.device).unsqueeze(0)
        cdf_target = (targets.unsqueeze(1) <= k_indices).float()
        loss_cramer = torch.mean(torch.sum((cdf_pred - cdf_target) ** 2, dim=1))
        return loss_ce + self.lambda_cramer * loss_cramer


class WeightedCramerLoss(nn.Module):
    """CE (suavizado 0.10) + Cramer ponderada por frontera (scripts/v47_01:96-120)."""

    def __init__(self, num_classes: int = 6, lambda_cramer: float = 0.50, smoothing: float = 0.10,
                 gamma: list[float] | None = None):
        super().__init__()
        self.ce = nn.CrossEntropyLoss(label_smoothing=smoothing)
        self.num_classes = num_classes
        self.lambda_cramer = lambda_cramer
        if gamma is None:
            gamma = [1.2, 1.5, 1.3, 1.1, 1.0]
        self.register_buffer("gamma", torch.as_tensor(gamma, dtype=torch.float32))

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        loss_ce = self.ce(logits, targets)
        probs = torch.softmax(logits, dim=1)
        cdf_pred = torch.cumsum(probs, dim=1)[:, :-1]
        k_indices = torch.arange(self.num_classes - 1, device=logits.device).unsqueeze(0)
        cdf_target = (targets.unsqueeze(1) <= k_indices).float()
        diff_sq = (cdf_pred - cdf_target) ** 2
        weighted_sq = diff_sq * self.gamma.unsqueeze(0)
        loss_cramer = torch.mean(torch.sum(weighted_sq, dim=1))
        return loss_ce + self.lambda_cramer * loss_cramer


class ConsistentSmoothedCramerLoss(nn.Module):
    """CE (suavizado 0.10) + Cramer sobre CDF objetivo suavizada (scripts/v51_01:53-74; v53_01:145-163)."""

    def __init__(self, num_classes: int = 6, lambda_cramer: float = 0.50, smoothing: float = 0.10):
        super().__init__()
        self.ce = nn.CrossEntropyLoss(label_smoothing=smoothing)
        self.num_classes = num_classes
        self.lambda_cramer = lambda_cramer
        self.smoothing = smoothing

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        loss_ce = self.ce(logits, targets)
        probs = torch.softmax(logits, dim=1)
        cdf_pred = torch.cumsum(probs, dim=1)[:, :-1]
        k_indices = torch.arange(self.num_classes - 1, device=logits.device).unsqueeze(0)
        cdf_step = (targets.unsqueeze(1) <= k_indices).float()
        cdf_target = ((1.0 - self.smoothing) * cdf_step
                      + self.smoothing * (k_indices + 1.0) / self.num_classes)
        loss_cramer = torch.mean(torch.sum((cdf_pred - cdf_target) ** 2, dim=1))
        return loss_ce + self.lambda_cramer * loss_cramer


def _construir_modelo(nombre: str, n_features: int, n_clases: int) -> nn.Module:
    """Crea la arquitectura exacta del componente."""
    tipo = CONFIGS[nombre]["modelo"]
    if tipo == "tabular":
        return TabularResNet(in_features=n_features, hidden=256, num_classes=n_clases,
                             drop=0.2)
    if tipo == "dense_se":
        return TabularDenseNet(in_features=n_features, init_dim=192, growth_rate=128,
                               num_classes=n_clases, drop=0.20)
    return SETabularResNet(in_features=n_features, hidden=256, num_classes=n_clases,
                           drop=0.2, reduction=4)


def _construir_loss(nombre: str, n_clases: int) -> nn.Module:
    """Crea la perdida exacta del componente."""
    tipo = CONFIGS[nombre]["loss"]
    if tipo == "ce":
        return nn.CrossEntropyLoss()
    if tipo == "ordinal_ce":
        return OrdinalCrossEntropyLoss(num_classes=n_clases, lambda_dist=0.2)
    if tipo == "ordinal_dist":
        return OrdinalDistanceLoss(num_classes=n_clases, lambda_dist=0.2)
    if tipo == "smooth_ordinal":
        return SmoothOrdinalDistanceLoss(num_classes=n_clases, lambda_dist=0.2, smoothing=0.10)
    if tipo == "cramer":
        return CramerDistanceLoss(num_classes=n_clases, lambda_cramer=0.50, smoothing=0.10)
    if tipo == "weighted_cramer":
        return WeightedCramerLoss(num_classes=n_clases, lambda_cramer=0.50, smoothing=0.10,
                                  gamma=[1.2, 1.5, 1.3, 1.1, 1.0])
    if tipo == "consistent_cramer":
        return ConsistentSmoothedCramerLoss(num_classes=n_clases, lambda_cramer=0.50,
                                            smoothing=0.10)
    raise ValueError(f"loss desconocida: {tipo}")


def _semilla(nombre: str, fold: int) -> int:
    """Semilla exacta por fold de cada componente."""
    return SEED + fold if CONFIGS[nombre]["seed"] == "42+fold" else SEED + fold * 100


def _entrenar_una(dispositivo, cfg: dict, semilla: int, X_tr: np.ndarray, y_tr: np.ndarray,
                   n_features: int, n_clases: int, nombre: str) -> nn.Module:
    """Entrena un modelo con la receta exacta del componente (con o sin SWA)."""
    torch.manual_seed(semilla)
    np.random.seed(semilla)
    base_model = _construir_modelo(nombre, n_features, n_clases).to(dispositivo)
    criterion = _construir_loss(nombre, n_clases).to(dispositivo)
    optimizer = optim.AdamW(base_model.parameters(), lr=cfg["lr"], weight_decay=cfg["wd"])
    scheduler = None
    if cfg["coseno"]:
        scheduler = optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=cfg["epocas"], eta_min=cfg["eta_min"])
    ds = TensorDataset(torch.from_numpy(X_tr).float(), torch.from_numpy(y_tr).long())
    loader = DataLoader(ds, batch_size=cfg["batch"], shuffle=True)
    swa_model = AveragedModel(base_model) if cfg["swa_start"] is not None else None
    for ep in range(cfg["epocas"]):
        base_model.train()
        for bx, by in loader:
            bx, by = bx.to(dispositivo), by.to(dispositivo)
            optimizer.zero_grad()
            out = base_model(bx)
            loss = criterion(out, by)
            loss.backward()
            optimizer.step()
        if scheduler is not None:
            scheduler.step()
        if swa_model is not None and ep >= cfg["swa_start"]:
            swa_model.update_parameters(base_model)
    if swa_model is not None:
        if cfg.get("bn_batch") is None:
            update_bn(loader, swa_model, device=dispositivo)
        else:
            loader_bn = DataLoader(ds, batch_size=cfg["bn_batch"], shuffle=False)
            update_bn(loader_bn, swa_model, device=dispositivo)
        modelo = swa_model
    else:
        modelo = base_model
    modelo.eval()
    return modelo


def _predecir(modelo: nn.Module, X: np.ndarray, dispositivo) -> np.ndarray:
    """Softmax en eval sobre una matriz (sin gradientes)."""
    with torch.no_grad():
        tensor = torch.from_numpy(X).float().to(dispositivo)
        return torch.softmax(modelo(tensor), dim=1).cpu().numpy()


def entrenar_componente(nombre: str, X_train: np.ndarray, X_test: np.ndarray, y: np.ndarray,
                        fold_idx: np.ndarray, dispositivo: str | None = None
                        ) -> tuple[np.ndarray, np.ndarray]:
    """Reentrena un componente V38-V53 y devuelve (oof, test) con las formas originales."""
    cfg = CONFIGS[nombre]
    if dispositivo is None:
        dispositivo = "cuda" if torch.cuda.is_available() else "cpu"
    n, n_features = X_train.shape
    n_test = X_test.shape[0]
    n_clases = len(CLASSES)
    oof = np.zeros((n, n_clases), dtype=np.float32)
    test = np.zeros((n_test, n_clases), dtype=np.float32)
    for fold in range(5):
        tr = fold_idx != fold
        va = fold_idx == fold
        scaler = StandardScaler()
        X_tr = scaler.fit_transform(X_train[tr])
        X_val = scaler.transform(X_train[va])
        X_te = scaler.transform(X_test)
        if cfg.get("clip") is not None:
            X_tr = np.clip(X_tr, -cfg["clip"], cfg["clip"])
            X_val = np.clip(X_val, -cfg["clip"], cfg["clip"])
            X_te = np.clip(X_te, -cfg["clip"], cfg["clip"])
        modelo = _entrenar_una(dispositivo, cfg, _semilla(nombre, fold), X_tr, y[tr],
                               n_features, n_clases, nombre)
        oof[va] = _predecir(modelo, X_val, dispositivo)
        if cfg["test"] == "media":
            test += _predecir(modelo, X_te, dispositivo) / 5.0
    if cfg["test"] == "full":
        scaler = StandardScaler()
        X_full = scaler.fit_transform(X_train)
        X_te = scaler.transform(X_test)
        modelo = _entrenar_una(dispositivo, cfg, SEED, X_full, y, n_features, n_clases,
                               nombre)
        test = _predecir(modelo, X_te, dispositivo)
    return oof, test


def hgb_componente(X_train: np.ndarray, X_test: np.ndarray, y: np.ndarray,
                   fold_idx: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """V39-HGB: HistGradientBoosting por fold (OOF) y full-train (test)."""
    n = len(y)
    oof = np.zeros((n, len(CLASSES)), dtype=np.float64)
    for fold in range(5):
        tr = fold_idx != fold
        va = fold_idx == fold
        clf = HistGradientBoostingClassifier(
            max_iter=200, max_leaf_nodes=31, learning_rate=0.05,
            min_samples_leaf=50, l2_regularization=1.0, random_state=SEED + fold)
        clf.fit(X_train[tr], y[tr])
        oof[va] = clf.predict_proba(X_train[va])
    clf = HistGradientBoostingClassifier(
        max_iter=200, max_leaf_nodes=31, learning_rate=0.05,
        min_samples_leaf=50, l2_regularization=1.0, random_state=SEED)
    clf.fit(X_train, y)
    test = clf.predict_proba(X_test)
    return oof, test

### Módulo `consenso_v2.py`

In [ ]:
"""Consensos Product-of-Experts de V53/V54/V55/V56 (porte fiel de los scripts de construccion).

Product-of-Experts bayesiano en el espacio de log-probabilidades:
    log P_geom(y = k | x) = sum_m w_m log P_m(y = k | x) - log Z(x)

CONSENSOS congela, por version, los miembros EN ORDEN de apilado y los pesos del script
ANTES de la normalizacion final (v53/v54 los derivan de W_V48; v55/v56 son literales):
- v53_02_build_submission.py:51-59,75-76 (7 miembros).
- v54_01_build_submission.py:52-72,89-93 (8 miembros, rama Cramer 0.80 V43 + 0.20 V47).
- v55_01_build_submission.py:53-56,73 (8 miembros, incluye CatBoost V49).
- v56_01_build_submission.py:53-56,74 (9 miembros, incluye V51 y V53).
Los pesos se normalizan al combinarlos (WEIGHTS_GEOM = WEIGHTS_GEOM / WEIGHTS_GEOM.sum()).
ASCII puro. Sin efectos en import.
"""

from __future__ import annotations

import numpy as np

#: Regla plug-in congelada por clase (v53_02_build_submission.py:51; age_umbrales_validacion.json).
PESOS_PLUG_IN = np.array([1.0000, 1.1749, 1.2721, 1.1630, 1.0010, 0.8128])

#: Pesos V48 de origen de las derivaciones de v53/v54 (v53_02:54-55, v54_01:55-56).
W_V48 = [0.5628, 0.0832, 0.1724, 0.0754, 0.0513, 0.0549]

#: Peso de Dense-SE-TabularNet en v53 (v53_02:56) y en v54 (v54_01:58).
W_DENSE_V53 = 0.0250
W_DENSE_V54 = 0.0200

#: Reparto de la rama Cramer de v54: 0.80 V43 + 0.20 V47 (v54_01:64-65).
REPARTO_CRAMER_V54 = (0.80, 0.20)

#: Epsilon del clip de probabilidades (v53_02:81, v54_01:98, v55_01:74, v56_01:75).
EPS = 1e-12

#: Especificaciones congeladas por version. Los valores float64 de v53/v54 son el resultado
#: EXACTO de la derivacion jerarquica (ver pesos_geometricos); v55/v56 son los literales.
CONSENSOS = {
    "v53": {
        "miembros": ["v30", "v43", "v42", "v40", "v39_ordinal", "v39_hgb", "v53_dense"],
        "pesos": [0.54873, 0.08112000000000001, 0.16809000000000002, 0.07351500000000001,
                  0.050017500000000006, 0.053527500000000006, 0.025],
        "fuente": "scripts/v53_02_build_submission.py:54-59,75-76",
    },
    "v54": {
        "miembros": ["v30", "v43", "v47", "v42", "v40", "v39_ordinal", "v39_hgb", "v53_dense"],
        "pesos": [0.551544, 0.06522880000000002, 0.016307200000000004, 0.16895200000000002,
                  0.07389200000000001, 0.050274000000000006, 0.053802, 0.02],
        "fuente": "scripts/v54_01_build_submission.py:55-72,89-93",
    },
    "v55": {
        "miembros": ["v30", "v43", "v47", "v42", "v40", "v39_ordinal", "v39_hgb", "v49_catboost"],
        "pesos": [0.52829, 0.07720, 0.02895, 0.17931, 0.07890, 0.05062, 0.04869, 0.00805],
        "fuente": "scripts/v55_01_build_submission.py:53-56,73",
    },
    "v56": {
        "miembros": ["v30", "v43", "v47", "v42", "v40", "v39_ordinal", "v39_hgb",
                     "v51_consistent", "v53_dense"],
        "pesos": [0.55939, 0.06023, 0.02089, 0.16873, 0.06080, 0.04376, 0.05204, 0.01035,
                  0.02381],
        "fuente": "scripts/v56_01_build_submission.py:53-56,74",
    },
}


def pesos_geometricos(version: str) -> np.ndarray:
    """Vector de pesos pre-normalizacion de una version, derivado como el script.

    v53: W_V48 normalizado * (1 - 0.0250) + [0.0250] (v53_02_build_submission.py:54-58).
    v54: W_V48 * (1 - 0.0200) con rama Cramer 0.80/0.20 y + [0.0200] (v54_01:55-71).
    v55/v56: literales del script (v55_01:55, v56_01:55).
    """
    if version == "v53":
        w48 = np.array(W_V48, dtype=np.float64)
        w48 = w48 / w48.sum()
        rem = 1.0 - W_DENSE_V53
        return np.array(list(w48 * rem) + [W_DENSE_V53])
    if version == "v54":
        w48 = np.array(W_V48, dtype=np.float64)
        w48 = w48 / w48.sum()
        base = w48 * (1.0 - W_DENSE_V54)
        reparto_v43, reparto_v47 = REPARTO_CRAMER_V54
        return np.array([base[0], base[1] * reparto_v43, base[1] * reparto_v47, base[2],
                         base[3], base[4], base[5], W_DENSE_V54])
    return np.array(CONSENSOS[version]["pesos"], dtype=np.float64)


def poe_ponderado(modelos: list[np.ndarray], pesos: list[float]) -> np.ndarray:
    """PoE ponderado en log-espacio con normalizacion final (v55_01:77-84, v56_01:77-85).

    Los pesos se normalizan (WEIGHTS_GEOM / WEIGHTS_GEOM.sum()); las probabilidades se
    recortan a [EPS, 1.0] antes del log; se desplaza por el maximo por fila y se
    renormalizan las filas. Entrada: modelos (n, 6) en orden de apilado.
    """
    w = np.asarray(pesos, dtype=np.float64)
    w = w / w.sum()
    log_geom = sum(wi * np.log(np.clip(m, EPS, 1.0)) for wi, m in zip(w, modelos))
    p_geom = np.exp(log_geom - np.max(log_geom, axis=1, keepdims=True))
    return p_geom / p_geom.sum(axis=1, keepdims=True)


def regla_plug_in(prob: np.ndarray) -> np.ndarray:
    """Decision plug-in por clase: argmax(prob * PESOS_PLUG_IN) (v53_02:88-89)."""
    return np.argmax(prob * PESOS_PLUG_IN, axis=1)

### Módulo `analisis_v57.py`

In [ ]:
"""Analisis matematico y estadistico de V57 (porte fiel de scripts/v57_analisis_matematico_estadistico.py).

Cuatro bloques sobre los OOF de 169.336 filas (miembros de V56 en orden y regla plug-in):
1. sensibilidad_simplex: perturbacion +-20% de cada coordenada de pesos (v57:66-94).
2. curva_power_mean: familia P_alpha con alpha en [-0.15, +0.15]; alpha=0 es el PoE (v57:101-119).
3. deciles_confiabilidad: deciles de confianza maxima vs exactitud (v57:125-149).
4. brecha_menciones: autocalibracion con/sin mencion (n_mention de dense_train.parquet; v57:151-152,165-178).

Los estados se escriben en ASCII (el original usa tildes: OPTIMO/SUB-OPTIMO). Sin efectos en import.
"""

from __future__ import annotations

import numpy as np
import pandas as pd
from sklearn.metrics import f1_score


#: Grilla de alpha del bloque Power-Mean (v57_analisis_matematico_estadistico.py:104).
ALPHAS = [-0.15, -0.10, -0.05, -0.01, 0.0, 0.01, 0.05, 0.10, 0.15]

#: Epsilon del clip de probabilidades (v57:46).
EPS = 1e-12

#: Nombres de los 9 miembros de V56 en orden de apilado (v57:48-52).
NOMBRES_V56 = ["V30 LightGBM Optuna", "V43 Cramer-SWA", "V47 Asym-Cramer", "V42 Smooth-SWA",
               "V40 SE-TabOrd", "V39 TabOrd", "V39 HistGB", "V51 Consistent-Cramer",
               "V53 Dense-SE"]


def macro_f1(y_true: np.ndarray, pred: np.ndarray) -> float:
    """Macro-F1 canonico sobre indices 0-5 (v57:62)."""
    return float(f1_score(y_true, pred, average="macro"))


def f1_por_folds(y_true: np.ndarray, pred: np.ndarray, fold_idx: np.ndarray) -> list[float]:
    """Macro-F1 por fold 0-4 (v57:63)."""
    return [float(f1_score(y_true[fold_idx == f], pred[fold_idx == f], average="macro"))
            for f in range(5)]


def sensibilidad_simplex(modelos: list[np.ndarray], y_true: np.ndarray, pesos: list[float],
                         nombres: list[str] | None = None) -> list[dict]:
    """Bloque 1: sensibilidad coordenual +-20% en el simplex (v57:66-94)."""
    w_base = np.asarray(pesos, dtype=np.float64)
    w_base = w_base / w_base.sum()
    p_base = poe_ponderado(modelos, w_base)
    f1_base = macro_f1(y_true, regla_plug_in(p_base))
    if nombres is None:
        nombres = [f"miembro_{i}" for i in range(len(modelos))]
    registros = []
    for i, nombre in enumerate(nombres):
        w_low = w_base.copy()
        w_low[i] *= 0.80
        w_low /= w_low.sum()
        f1_low = macro_f1(y_true, regla_plug_in(poe_ponderado(modelos, w_low)))

        w_high = w_base.copy()
        w_high[i] *= 1.20
        w_high /= w_high.sum()
        f1_high = macro_f1(y_true, regla_plug_in(poe_ponderado(modelos, w_high)))

        registros.append({
            "modelo": nombre,
            "peso_base": float(w_base[i]),
            "f1_minus_20pct": f1_low,
            "delta_minus_20pct": f1_low - f1_base,
            "f1_plus_20pct": f1_high,
            "delta_plus_20pct": f1_high - f1_base,
            "estado": "ESTACIONARIO" if (f1_low < f1_base and f1_high < f1_base)
                      else "NO_ESTACIONARIO",
        })
    return registros


def curva_power_mean(modelos: list[np.ndarray], y_true: np.ndarray,
                     pesos: list[float]) -> list[dict]:
    """Bloque 2: Macro-F1(alpha) de la familia Power-Mean; alpha=0 es el PoE (v57:101-119)."""
    w_base = np.asarray(pesos, dtype=np.float64)
    w_base = w_base / w_base.sum()
    p_poe = poe_ponderado(modelos, w_base)
    f1_base = macro_f1(y_true, regla_plug_in(p_poe))
    registros = []
    for alpha in ALPHAS:
        if alpha == 0.0:
            p = p_poe
        else:
            p_alpha = sum(wi * (np.clip(m, EPS, 1.0) ** alpha)
                          for wi, m in zip(w_base, modelos))
            p = p_alpha ** (1.0 / alpha)
            p = p / p.sum(axis=1, keepdims=True)
        f1 = macro_f1(y_true, regla_plug_in(p))
        registros.append({
            "alpha": alpha,
            "macro_f1": f1,
            "delta_vs_poe": f1 - f1_base,
            "estado": "OPTIMO GLOBAL ESTACIONARIO (PoE)" if alpha == 0.0 else "SUB-OPTIMO",
        })
    return registros


def deciles_confiabilidad(prob: np.ndarray, y_true: np.ndarray) -> list[dict]:
    """Bloque 3: deciles de confianza maxima vs exactitud (v57:125-149)."""
    pred = regla_plug_in(prob)
    conf = np.max(prob, axis=1)
    correct = (pred == y_true)
    deciles = pd.qcut(conf, 10, labels=False)
    registros = []
    for d in range(10):
        mask = (deciles == d)
        registros.append({
            "decil": int(d),
            "confianza_min": float(conf[mask].min()),
            "confianza_max": float(conf[mask].max()),
            "confianza_media": float(conf[mask].mean()),
            "exactitud": float(correct[mask].mean()),
            "conteo": int(mask.sum()),
        })
    return registros


def brecha_menciones(prob: np.ndarray, y_true: np.ndarray, n_mention: np.ndarray) -> dict:
    """Bloque 4: brecha confianza-exactitud con/sin mencion (v57:151-152,165-178)."""
    pred = regla_plug_in(prob)
    conf = np.max(prob, axis=1)
    correct = (pred == y_true)
    has_men = (n_mention > 0)
    return {
        "tweets_con_mencion": {
            "proporcion": float(has_men.mean()),
            "exactitud": float(correct[has_men].mean()),
            "confianza_media": float(conf[has_men].mean()),
            "gap": float(conf[has_men].mean() - correct[has_men].mean()),
        },
        "tweets_sin_mencion": {
            "proporcion": float((~has_men).mean()),
            "exactitud": float(correct[~has_men].mean()),
            "confianza_media": float(conf[~has_men].mean()),
            "gap": float(conf[~has_men].mean() - correct[~has_men].mean()),
        },
    }


def analisis_completo(modelos: list[np.ndarray], y_true: np.ndarray, pesos: list[float],
                      n_mention: np.ndarray, nombres: list[str] | None = None,
                      fold_idx: np.ndarray | None = None) -> dict:
    """Los cuatro bloques de V57 en un resumen con el esquema del JSON original."""
    prob = poe_ponderado(modelos, pesos)
    pred = regla_plug_in(prob)
    return {
        "f1_base": macro_f1(y_true, pred),
        "folds_base": None if fold_idx is None else f1_por_folds(y_true, pred, fold_idx),
        "sensibilidad_simplex": sensibilidad_simplex(modelos, y_true, pesos, nombres),
        "curva_power_mean": curva_power_mean(modelos, y_true, pesos),
        "deciles_confiabilidad": deciles_confiabilidad(prob, y_true),
        "brecha_menciones": brecha_menciones(prob, y_true, n_mention),
    }

### Módulo `test_side.py`

In [ ]:
"""Lado TEST de la cadena del campeon: re-ajuste de la cadena sobre el train completo.

Reproduce el camino de test del pipeline v12/V30 (matriz 279) sin dependencias del proyecto
original (prohibido importar `src`/`scripts`; solo `cadena. y `cadena.nb_svm`):

    - representaciones dispersas: porte de ``src/pipeline.py:157-198``
      (``build_representations``), ajustadas solo con train y transformando test.
    - 13 bloques de scores de test: porte de ``src/pipeline.py:199-288``
      (``fit_base_models``) en el orden de ``OOF_FILES`` (``src/pipeline.py:59-65``), con la
      transformacion de ``MEMBER_FILES`` (``scripts/phase2_ponderacion.py:49-55``): ``log``
      si la fuente es probabilistica, crudo si no.
    - NB-SVM de palabras (ADV07): porte de ``scripts/phase25_submission_fresca.py:92-103``.
    - ensamblado (m, 279): porte de ``scripts/phase25_submission_fresca.py:108-111``.

Reglas del porte: ASCII puro, sin efectos en import, sin aleatoriedad anadida (semilla 42).
"""

from __future__ import annotations

import numpy as np
import pandas as pd
from scipy.sparse import hstack
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, Ridge, SGDClassifier
from sklearn.multiclass import OneVsRestClassifier
from sklearn.naive_bayes import ComplementNB
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import LinearSVC


#: src/config.py:59 -- TEXT_COL = "text".
TEXT_COL = "text"

#: Componentes de la SVD del miembro E019b (src/pipeline.py:52).
SVD_COMPONENTS = 64


def adjacent_pair_margins(x_train, y_train: np.ndarray, x_valid) -> np.ndarray:
    """Margenes de 5 SVM entrenados **solo** con cada par de rangos contiguos (E-028a).

    Porte de ``src/pipeline.py:101-115``. Concentra la capacidad del modelo en las fronteras
    donde ocurre el error (la confusion del ensamble es casi toda entre rangos adyacentes).
    """
    margins = np.zeros((x_valid.shape[0], len(ADJACENT_PAIRS)), dtype=np.float32)
    for position, (lower, upper) in enumerate(ADJACENT_PAIRS):
        mask = (y_train == lower) | (y_train == upper)
        binary_target = (y_train[mask] == upper).astype(int)
        model = LinearSVC(C=0.2, max_iter=3000)
        model.fit(x_train[mask], binary_target)
        margins[:, position] = model.decision_function(x_valid)
    return margins


def ordinal_cumulative_probabilities(x_train, y_index: np.ndarray, x_valid) -> np.ndarray:
    """Modelo ordinal acumulativo (Frank & Hall 2001) sobre la representacion dispersa.

    Porte de ``src/pipeline.py:118-136``. Entrena K-1 clasificadores binarios ``P(y > k)`` y
    los combina en una distribucion: ``P(y = k) = P(y > k-1) - P(y > k)`` (E-021a: 0,5527).
    """
    thresholds = np.arange(1, len(CLASSES))
    cumulative = np.ones((x_valid.shape[0], len(thresholds) + 1), dtype=np.float64)
    for position, threshold in enumerate(thresholds):
        binary_target = (y_index >= threshold).astype(int)
        model = LogisticRegression(C=1.0, max_iter=1000, solver="liblinear",
                                   random_state=SEED)
        model.fit(x_train, binary_target)
        cumulative[:, position + 1] = model.predict_proba(x_valid)[:, 1]
    cumulative = np.clip(np.minimum.accumulate(cumulative, axis=1), 0.0, 1.0)
    probabilities = np.column_stack([-np.diff(cumulative, axis=1), cumulative[:, -1]])
    probabilities = np.clip(probabilities, 1e-12, None)
    return probabilities / probabilities.sum(axis=1, keepdims=True)


def rocchio_probabilities(x_train, y: np.ndarray, x_valid) -> np.ndarray:
    """Clasificador de centroides (Rocchio) con similitud coseno (E-021b: 0,4017).

    Porte de ``src/pipeline.py:139-154``. Miembro deliberadamente debil pero **diverso**.
    """
    centroids = []
    for label in CLASSES:
        mask = y == label
        centroid = np.asarray(x_train[mask].mean(axis=0)).ravel()
        centroids.append(centroid / (np.linalg.norm(centroid) + 1e-12))
    centroid_matrix = np.vstack(centroids)
    valid_norms = np.sqrt(np.asarray(x_valid.multiply(x_valid).sum(axis=1)).ravel()) + 1e-12
    similarity = np.asarray(x_valid @ centroid_matrix.T) / valid_norms[:, None]
    shifted = similarity - similarity.max(axis=1, keepdims=True)
    exponentials = np.exp(shifted * 8.0)
    return exponentials / exponentials.sum(axis=1, keepdims=True)


def representaciones_test(train: pd.DataFrame, test: pd.DataFrame) -> dict:
    """Ajusta los vectorizadores con train y transforma train y test.

    Porte de ``src/pipeline.py:157-198`` (``build_representations``). Configs congeladas
    ``CHAR_300K``/``CHAR_700K``/``MENTION``/``HASHTAG``/``WORD`` de ``cadena.`
    (= ``src/pipeline.py:42-51``). Los tokens de menciones y hashtags se extraen con
    ``tokens_of`` (``src/pipeline.py:75-78``) y patrones ``r"@\\w+"`` / ``r"#\\w+"``.

    Returns
    -------
    dict
        Matrices dispersas de cada canal mas el detalle de dimensiones (mismas claves que el
        original: x300/x700/xh/xw de train y test, y "dimensiones").
    """
    char300 = TfidfVectorizer(**CHAR_300K)
    mention300 = TfidfVectorizer(**MENTION)
    x300_train = hstack([char300.fit_transform(train[TEXT_COL]),
                         mention300.fit_transform(tokens_of(train[TEXT_COL], r"@\w+"))]).tocsr()
    x300_test = hstack([char300.transform(test[TEXT_COL]),
                        mention300.transform(tokens_of(test[TEXT_COL], r"@\w+"))]).tocsr()

    char700 = TfidfVectorizer(**CHAR_700K)
    mention700 = TfidfVectorizer(**MENTION)
    x700_train = hstack([char700.fit_transform(train[TEXT_COL]),
                         mention700.fit_transform(tokens_of(train[TEXT_COL], r"@\w+"))]).tocsr()
    x700_test = hstack([char700.transform(test[TEXT_COL]),
                        mention700.transform(tokens_of(test[TEXT_COL], r"@\w+"))]).tocsr()

    hashtag = TfidfVectorizer(**HASHTAG)
    xh_train = hstack([x300_train,
                       hashtag.fit_transform(tokens_of(train[TEXT_COL], r"#\w+"))]).tocsr()
    xh_test = hstack([x300_test,
                      hashtag.transform(tokens_of(test[TEXT_COL], r"#\w+"))]).tocsr()

    word = TfidfVectorizer(**WORD)
    xw_train = word.fit_transform(train[TEXT_COL])
    xw_test = word.transform(test[TEXT_COL])

    return {"x300_train": x300_train, "x300_test": x300_test,
            "x700_train": x700_train, "x700_test": x700_test,
            "xh_train": xh_train, "xh_test": xh_test,
            "xw_train": xw_train, "xw_test": xw_test,
            "dimensiones": {"char300+menciones": int(x300_train.shape[1]),
                            "char700+menciones": int(x700_train.shape[1]),
                            "char300+menciones+hashtags": int(xh_train.shape[1]),
                            "word12": int(xw_train.shape[1])}}


def base_models_test(reps: dict, y: np.ndarray, dense_train: np.ndarray, dense_test: np.ndarray,
                     textos_train: pd.Series, textos_test: pd.Series) -> list[np.ndarray]:
    """Ajusta los 13 modelos base sobre todo el train y devuelve sus scores de test.

    Porte de ``src/pipeline.py:199-288`` (``fit_base_models``). Los 13 bloques van en el orden
    de ``OOF_FILES`` (``src/pipeline.py:59-65``) con la transformacion ``log`` de
    ``MEMBER_FILES`` (``scripts/phase2_ponderacion.py:49-55``) cuando la fuente es
    probabilistica; el resto va crudo:

        1.  E013  LinearSVC(C=0.2, max_iter=3000) sobre x300  -> margenes (raw, 6)
        2.  E016  LinearSVC(C=0.2, max_iter=3000) sobre x700  -> margenes (raw, 6)
        3.  E-009 OneVsRest(LogReg C=4.0) sobre x300          -> proba log (6)
        4.  E-011 SGDClassifier(log_loss) sobre x300          -> proba log (6)
        5.  E-010 ComplementNB(alpha=0.3) sobre x300          -> proba log (6)
        6.  E014  LinearSVC(C=0.2, max_iter=3000) sobre xh    -> margenes (raw, 6)
        7.  E015_ordinal  Ridge(alpha=1.0, sparse_cg) sobre x300 -> continuo (raw, 1)
        8.  E015_softmax  log(softmax_ordinal(continuo))      -> log (6)
        9.  E017  OneVsRest(LogReg C=4.0) sobre xw            -> proba log (6)
        10. E019b MLP(256,128) sobre [35 densas + SVD64 de x300] -> proba log (6)
        11. E021a ordinal acumulativo sobre x300              -> log (6)
        12. E021b centroides Rocchio sobre x300               -> log (6)
        13. E028a margenes de 5 SVM de pares adyacentes       -> raw (5)

    Total 72 columnas (11 bloques de 6 + 1 de 1 + 1 de 5). ``textos_train``/``textos_test``
    forman parte de la interfaz congelada del porte; el cuerpo literal del original no los
    consume (toda la senal de texto entra por ``reps``). El ``detail`` con la varianza
    explicada por la SVD del original no forma parte de la interfaz (se retorna solo la lista).

    Parameters
    ----------
    reps : dict
        Salida de ``representaciones_test``.
    y : np.ndarray
        Etiquetas de train (strings de ``CLASSES``).
    dense_train, dense_test : np.ndarray
        35 features densas (``DENSE_FEATURES``, sin las 3 de emoji).
    textos_train, textos_test : pd.Series
        Textos crudos (no consumidos por el cuerpo portado).

    Returns
    -------
    list[np.ndarray]
        13 bloques de scores de test en el orden de ``OOF_FILES``.
    """
    scores: list[np.ndarray] = []

    model = LinearSVC(C=0.2, max_iter=3000).fit(reps["x300_train"], y)
    scores.append(to_scores(model.decision_function(reps["x300_test"]), model.classes_, False))
    model = LinearSVC(C=0.2, max_iter=3000).fit(reps["x700_train"], y)
    scores.append(to_scores(model.decision_function(reps["x700_test"]), model.classes_, False))

    model = OneVsRestClassifier(LogisticRegression(C=4.0, max_iter=1000, solver="liblinear",
                                                   random_state=SEED))
    model.fit(reps["x300_train"], y)
    scores.append(to_scores(model.predict_proba(reps["x300_test"]), model.classes_, True))

    model = SGDClassifier(loss="log_loss", alpha=1e-6, max_iter=30, tol=1e-4,
                          random_state=SEED).fit(reps["x300_train"], y)
    scores.append(to_scores(model.predict_proba(reps["x300_test"]), model.classes_, True))
    model = ComplementNB(alpha=0.3).fit(reps["x300_train"], y)
    scores.append(to_scores(model.predict_proba(reps["x300_test"]), model.classes_, True))

    model = LinearSVC(C=0.2, max_iter=3000).fit(reps["xh_train"], y)
    scores.append(to_scores(model.decision_function(reps["xh_test"]), model.classes_, False))

    ordinal_target = np.array([CLASS_MIDPOINTS[label] for label in y])
    ridge = Ridge(alpha=1.0, solver="sparse_cg", max_iter=2000, random_state=SEED)
    ridge.fit(reps["x300_train"], ordinal_target)
    continuous_test = ridge.predict(reps["x300_test"])
    scores.append(continuous_test.reshape(-1, 1))
    scores.append(np.log(np.clip(softmax_ordinal(continuous_test), 1e-12, None)))

    model = OneVsRestClassifier(LogisticRegression(C=4.0, max_iter=1000, solver="liblinear",
                                                   random_state=SEED))
    model.fit(reps["xw_train"], y)
    scores.append(to_scores(model.predict_proba(reps["xw_test"]), model.classes_, True))

    # MLP sobre densas + SVD del espacio de caracteres (miembro diverso, E019b).
    svd = TruncatedSVD(n_components=SVD_COMPONENTS, random_state=SEED)
    z_train = svd.fit_transform(reps["x300_train"])
    z_test = svd.transform(reps["x300_test"])
    scaler = StandardScaler().fit(np.hstack([dense_train, z_train]))
    encoder = LabelEncoder().fit(y)
    mlp = MLPClassifier(hidden_layer_sizes=(256, 128), alpha=1e-3, batch_size=512,
                        learning_rate_init=1e-3, max_iter=60, early_stopping=True,
                        n_iter_no_change=6, random_state=SEED)
    mlp.fit(scaler.transform(np.hstack([dense_train, z_train])), encoder.transform(y))
    probabilities = mlp.predict_proba(scaler.transform(np.hstack([dense_test, z_test])))
    scores.append(np.log(np.clip(probabilities, 1e-12, None)))

    # Miembro ordinal acumulativo (Frank & Hall): 5 clasificadores binarios sobre x300.
    y_index = np.array([CLASSES.index(label) for label in y])
    ordinal_probabilities = ordinal_cumulative_probabilities(reps["x300_train"], y_index,
                                                             reps["x300_test"])
    scores.append(np.log(np.clip(ordinal_probabilities, 1e-12, None)))

    # Miembro de centroides (Rocchio): debil pero diverso.
    centroids = rocchio_probabilities(reps["x300_train"], y, reps["x300_test"])
    scores.append(np.log(np.clip(centroids, 1e-12, None)))

    # Supervision ordinal fina: margenes entre pares de rangos contiguos.
    pair_margins = adjacent_pair_margins(reps["x300_train"], y, reps["x300_test"])
    scores.append(pair_margins)

    return scores


def nb_word_test(textos_train: pd.Series, y: np.ndarray,
                 textos_test: pd.Series) -> np.ndarray:
    """NB-SVM de palabras (ADV07) re-ajustado sobre el train completo.

    Porte de ``scripts/phase25_submission_fresca.py:92-103`` (identico en
    ``scripts/phase8_holdout_v12.py:153-164``). Mismo vectorizador (``WORD`` de 
    analyzer word, ngramas 1-2, min_df=3, sublinear_tf, 300k, lowercase) ajustado sobre
    **todo el train**; log-count ratio por clase sobre todo el train (``nb_log_count_ratio``,
    ``scripts/phase7_nbsvm_canal_meta.py:57-61``); ``LinearSVC(C=0.2, max_iter=3000)``
    uno-vs-rest por clase y ``decision_function`` sobre el test.

    Returns
    -------
    np.ndarray
        (m, 6) float32, margenes crudos en el orden de ``CLASSES`` (entran sin transformar
        como ultimas 6 columnas de la matriz 279).
    """
    word = TfidfVectorizer(**WORD)
    x_train_w = word.fit_transform(textos_train)
    x_test_w = word.transform(textos_test)
    margenes = np.zeros((len(textos_test), len(CLASSES)), dtype=np.float32)
    for posicion, etiqueta in enumerate(CLASSES):
        binario = (y == etiqueta).astype(int)
        ratio = nb_log_count_ratio(x_train_w, binario)
        modelo = LinearSVC(C=0.2, max_iter=3000)
        modelo.fit(x_train_w.multiply(ratio).tocsr(), binario)
        margenes[:, posicion] = modelo.decision_function(x_test_w.multiply(ratio).tocsr())
    return margenes


def matriz_test_fresca(bloques_72: list[np.ndarray], nb_test: np.ndarray,
                       densas_test: np.ndarray, estructuradas_test: np.ndarray,
                       supervisadas_test: np.ndarray, nb_word: np.ndarray) -> np.ndarray:
    """Ensambla el lado test (m, 279) en el orden EXACTO del campeon.

    Porte de ``scripts/phase25_submission_fresca.py:108-111``. Orden de columnas:

        [72 scores del pipeline (13 bloques)] + [nb_test 6] + [densas 35] +
        [estructuradas 87] + [supervisadas 73] + [nb_word 6]  = 279

    Parameters
    ----------
    bloques_72 : list[np.ndarray]
        13 bloques de ``base_models_test`` (6x11 + ordinal 1 + pares 5).
    nb_test : np.ndarray
        (m, 6) NB-SVM de char300+menciones (ADV05 lado test; ver ``cadena.nb_svm``).
    densas_test, estructuradas_test, supervisadas_test : np.ndarray
        (m, 35), (m, 87) y (m, 73) respectivamente.
    nb_word : np.ndarray
        (m, 6) salida de ``nb_word_test`` (ADV07 lado test).

    Returns
    -------
    np.ndarray
        (m, 279); falla con ``AssertionError`` si el ancho no es 279.
    """
    matriz = np.concatenate(bloques_72 + [nb_test, densas_test, estructuradas_test,
                                          supervisadas_test, nb_word], axis=1)
    assert matriz.shape[1] == 279, f"lado test inesperado: {matriz.shape}"
    return matriz

## 2.4 La fuga por near-duplicados y los folds canónicos

El corpus contiene **near-duplicados** (Kaufman, Rosset, Perlich y Stitelman, «Leakage in Data Mining: Formulation, Detection, and Avoidance», ACM Transactions on Knowledge Discovery from Data, 2012): textos idénticos o casi idénticos (retuits, copias con un emoji
distinto, el mismo chiste repetido) que aparecen tanto en entrenamiento como en prueba. Si la validación
cruzada se hace con folds aleatorios, un texto de validación puede tener su gemelo en entrenamiento: el
modelo lo memoriza y la validación **sobreestima** el desempeño real. Este fenómeno se midió y quedó
documentado como la decisión de protocolo más importante del proyecto: **la validación agrupada por
clústeres de near-duplicados**, donde un clúster entero cae en un solo fold. Así, ningún fold de
validación contiene un texto cuyo near-duplicado esté en el fold de entrenamiento.

Los clústeres se construyen con un procedimiento determinista (normalización del texto, comparación por
similitud de Jaccard (Jaccard, «The Distribution of the Flora in the Alpine Zone», New Phytologist, 1912) de tokens con *blocking* por tokens raros, unión de pares similares). La partición de
5 folds agrupados es el **protocolo congelado** del proyecto: todos los experimentos de este notebook se
evalúan con exactamente esos folds, que se **generan en la propia ejecución** con la regla documentada
(`StratifiedGroupKFold` con `shuffle=True` y semilla 42 sobre los clústeres) y se verifican por
propiedades: tamaños equilibrados, proporciones de clase estables y cero clústeres compartidos entre
folds. En versiones de scikit-learn distintas de la 1.8.0 la asignación puede variar y, con ella, los
valores absolutos (véase 6.2).

In [ ]:
# Figura 2.4: reconstruccion de los clusteres de near-duplicados (protocolo congelado).
t0 = time.time()
textos_todo = pd.concat([train["text"], test["text"]], ignore_index=True)
clusters = construir_clusters(textos_todo)
print(f"clusters: {clusters['cluster'].nunique():,} en {len(clusters):,} textos "
      f"({time.time() - t0:.1f}s)")

cluster_de = clusters["cluster"].to_numpy()
n_train = len(train)
pares = pd.Series(cluster_de[:n_train])
repetidos = pares.duplicated(keep=False)
exactos_test = float(test["text"].isin(set(train["text"])).mean())
compartidos_test = float(pd.Series(cluster_de[n_train:]).isin(
    set(cluster_de[:n_train])).mean())

fig, ejes = plt.subplots(1, 2, figsize=(11, 3.4))
tamanos = pd.Series(cluster_de).value_counts().value_counts().sort_index()
ejes[0].bar(tamanos.index.astype(str), tamanos.values, color="#2b6cb0")
ejes[0].set_title("Tamano de los clusters de near-duplicados")
ejes[0].set_xlabel("textos por cluster")
ejes[0].set_ylabel("numero de clusters")
ejes[0].tick_params(labelsize=8)
ejes[1].bar(["exactos test~train", "comparten cluster", "sin gemelo"],
            [exactos_test * 100, compartidos_test * 100, (1 - compartidos_test) * 100],
            color=["#c53030", "#ed8936", "#2b6cb0"])
ejes[1].set_title("Test con gemelo en train (%)")
ejes[1].set_ylabel("porcentaje")
plt.tight_layout()
plt.show()
print(f"train en clusters de tamano > 1: {float(repetidos.mean()) * 100:.1f}%")
print(f"test con duplicado exacto en train: {exactos_test * 100:.2f}%")
print(f"test que comparte cluster con train: {compartidos_test * 100:.2f}%")

In [ ]:
# Folds canonicos: se generan con la regla documentada (StratifiedGroupKFold con shuffle y semilla 42
# sobre los clusters de near-duplicados) y se verifican por propiedades, sin constantes opacas.
from sklearn.model_selection import StratifiedGroupKFold


def generar_folds(etiquetas, grupos, semilla=42, n_folds=5):
    """Asigna cada fila a un fold agrupado por cluster: ningun cluster cae en dos folds."""
    divisor = StratifiedGroupKFold(n_splits=n_folds, shuffle=True, random_state=semilla)
    asignacion = np.zeros(len(etiquetas), dtype=np.int64)
    for numero, (_, validacion) in enumerate(
            divisor.split(np.zeros(len(etiquetas)), etiquetas, groups=grupos)):
        asignacion[validacion] = numero
    return asignacion


fold_idx = generar_folds(y, cluster_de[:n_train], semilla=42)
print("folds canonicos (semilla 42):", np.bincount(fold_idx).tolist())
for numero in range(5):
    en_fold = fold_idx == numero
    proporcion = np.bincount(clase[en_fold], minlength=len(CLASSES)) / en_fold.sum()
    print(f"  fold {numero}: {en_fold.sum():,} filas | clases "
          f"{np.round(proporcion * 100, 1).tolist()} %")

# Verificacion por propiedades: cobertura completa, tamano equilibrado y cero fuga entre folds.
assert sorted(np.unique(fold_idx).tolist()) == list(range(5))
tamanos = np.bincount(fold_idx)
diferencia = int(tamanos.max() - tamanos.min())
print(f"tamanos por fold: min {tamanos.min():,} | max {tamanos.max():,} | "
      f"diferencia {diferencia:,}")
assert diferencia <= 0.05 * len(fold_idx) / 5  # cota: 5% del fold medio
grupos = cluster_de[:n_train]
fugados = set()
for numero in range(5):
    fugados |= set(grupos[fold_idx == numero]) & set(grupos[fold_idx != numero])
print(f"clusters compartidos entre folds (debe ser 0): {len(fugados)}")
assert not fugados, "fuga de near-duplicados entre folds"
print("protocolo verificado: 5 folds agrupados sin fuga de near-duplicados.")

## 2.5 Construcción del conjunto de datos de modelado

El corpus crudo no se usa directamente: el modelado consume una **matriz de 279 columnas** construida en
siete pasos, que se resumen aquí porque son el contrato que comparten todos los experimentos del notebook.

1. **Corpus y limpieza.** Se cargan `age_train.csv` (169.336 filas) y `age_test.csv` (112.892) con
   separador `;`; se recorta el relleno de ancho fijo (`strip`), se conserva la transliteración ASCII y se
   tratan los textos vacíos como cadenas vacías. La etiqueta es `age_range`, con seis clases.
2. **Particiones congeladas.** Los cinco folds agrupados por clúster de near-duplicados (sección 2.4) se
   embeben como constante: ninguna fila de validación comparte clúster con el entrenamiento.
3. **Representación.** TF-IDF de palabras (1-2) y de carácter `char_wb` (3-6), ajustado dentro de cada
   fold con el vocabulario, `min_df` y `max_features` de la receta documentada.
4. **Miembros (78 columnas).** Catorce clasificadores individuales (tabla de la sección 3.4) generan
   probabilidades o márgenes fuera de fold; se aplica `log` a las columnas indicadas.
5. **Features densas (35).** Textometría estilométrica por documento: longitudes, emoticonos,
   alargamientos, puntuación repetida, mayúsculas, dígitos, entropías y ratios de compresión.
6. **Features estructuradas (87).** Seis familias: distribución de longitudes, distribución de caracteres,
   riqueza léxica, forma de palabra, posición y entropías de puntuación.
7. **Features supervisadas (73) y ADV-07 (6).** Perfil de clase por entidad (menciones, hashtags, fuente
   de retuit), rareza léxica y léxico log-odds construidos **dentro de cada fold**, más seis márgenes
   NB-SVM de palabra.

La tabla fija el inventario de variables; la construcción efectiva y sus aserciones de forma se ejecutan en
la Fase 3 (secciones 3.3 y 3.4), donde la matriz se ensambla una sola vez
(`assert matriz.shape == (169336, 279)`) y todos los modelos posteriores la consumen.

| Bloque | Columnas | Contenido y ejemplos |
|---|---:|---|
| Miembros | 78 | Probabilidades o márgenes fuera de fold de 14 clasificadores (log si corresponde) |
| Densas | 35 | Longitudes, emoticonos, alargamientos, puntuación repetida, mayúsculas, dígitos, entropías, ratios de compresión |
| Estructuradas | 87 | Distribución de longitudes, de caracteres, riqueza léxica, forma de palabra, posición, entropías de puntuación |
| Supervisadas | 73 | Perfil de clase por entidad, rareza léxica y log-odds por clase (in-fold) |
| ADV-07 | 6 | Márgenes NB-SVM uno-vs-rest de palabra |
| **Total** | **279** | 273 del meta + 6 del canal de palabra |

Este resultado cierra la fase de comprensión de datos con el protocolo listo: corpus auditado, EDA hecho,
estructura ordinal entendida y partición honesta de validación verificada. Todo lo que sigue se evalúa con
estos folds.

## 3.3 Features densas, estructuradas y supervisadas

Sobre la representación base, el proyecto construyó tres bloques de features que después alimentan al
meta-modelo (y que en el campeón ocupan 195 de las 279 columnas). Este bloque de trabajo se adoptó como
`AGE_ADVANCED_V01` tras un gate estricto: **+0.0031 de CV (positivo en 5/5 folds), +0.0052 en el holdout
congelado y +0.0043 en el leaderboard** — una de las pocas adopciones del proyecto, y de las más sólidas.

- **Densas (35 columnas):** textometría y estilometría del texto (Schler, Koppel, Argamon y Pennebaker, «Effects of Age and Gender on Blogging», AAAI Spring Symposium 2006; Argamon, Koppel, Pennebaker y Schler, «Mining the Blogosphere: Age, gender and the varieties of self-expression», First Monday 2007) — longitudes, riqueza léxica (TTR,
  hapax, entropía), proporción de stopwords, emoticonos, mayúsculas, puntuación múltiple. Son locales al
  documento (no dependen del corpus), totalmente deterministas y sin parámetros aprendidos.
- **Estructuradas (87 columnas):** seis familias de forma — distribución de longitudes de palabra,
  distribución de caracteres, riqueza léxica avanzada (MATTR, Yule, Maas), forma de las palabras,
  posición de entidades (menciones, hashtags, URLs; Nguyen, Gravel, Trieschnigg y Meder, «How Old Do You Think I Am? A Study of Language and Age in Twitter», ICWSM 2013) y entropía de estructura (puntuación, compresión).
- **Supervisadas (73 columnas):** rasgos que **usan las etiquetas del fold de entrenamiento** y por eso
  se calculan estrictamente in-fold (nunca ven la validación): probabilidad de cada rango dada una
  mención/hashtag/fuente de retuit (con suavizado sobre el prior), rareza léxica contra el vocabulario
  del fold (con centinela para palabras fuera de vocabulario) y puntuaciones léxicas por clase con
  log-odds de Monroe (Monroe, Colaresi y Quinn, «Fightin' Words: Lexical Feature Selection and Evaluation for Identifying the Content of Political Conflict», Political Analysis, 2008).

In [ ]:
# Construccion de los tres bloques de features (densas, estructuradas, supervisadas in-fold).
t0 = time.time()
dense = extraer_textometria(train)
dense_test = extraer_textometria(test)
print(f"densas: {dense.shape} | {time.time() - t0:.0f}s")

t0 = time.time()
estructuradas = extraer_estructuradas(train)
estructuradas_test = extraer_estructuradas(test)
print(f"estructuradas: {estructuradas.shape} | {time.time() - t0:.0f}s")

t0 = time.time()
supervisadas = supervisadas_infold(train["text"], y, fold_idx)
supervisadas_te = supervisadas_test(train["text"], y, test["text"])
print(f"supervisadas (in-fold): {supervisadas.shape} | {time.time() - t0:.0f}s")

print("\nreferencia: 35 densas + 87 estructuradas + 73 supervisadas = 195 columnas del meta")

In [ ]:
# Figura 3.2: tamano de los bloques y varianza explicada por familia (resumen visual).
fig, ejes = plt.subplots(1, 2, figsize=(11, 3.4))
ejes[0].bar(["densas", "estructuradas", "supervisadas"], [35, 87, 73],
            color=["#2b6cb0", "#2f855a", "#ed8936"])
ejes[0].set_title("Tamano de cada bloque de features")
ejes[0].set_ylabel("columnas")
varianza = pd.DataFrame({"densas": dense.std(), "estructuradas": estructuradas.std(),
                         "supervisadas": supervisadas.std()})
nulos = float((varianza == 0).to_numpy().mean()) * 100
ejes[1].bar(["densas", "estructuradas", "supervisadas"],
            [float(dense.std().mean()), float(estructuradas.std().mean()),
             float(supervisadas.std().mean())], color=["#2b6cb0", "#2f855a", "#ed8936"])
ejes[1].set_title(f"Desviacion tipica media (columnas nulas: {nulos:.1f}%)")
plt.tight_layout()
plt.show()

## 3.4 Las fases PH y el meta-modelo (V5–V12)

Con las representaciones y features listas, la pregunta cambió de «¿cómo representar?» a «¿cómo
combinar?». El proyecto construyó **14 miembros diversos** — cada uno con una representación, un modelo o
una formulación distinta — y los combinó en un meta-modelo:

| # | Miembro | Representación | Modelo | CV documentada |
|---|---|---|---|---|
| 1 | E-013 | char_wb(3-6) 300k + menciones | LinearSVC C=0.2 (márgenes) | 0.5628 |
| 2 | E-016 | char_wb(3-6) 700k + menciones | LinearSVC C=0.2 (márgenes) | 0.5658 |
| 3 | E-009 | char300 + menciones | LogReg OvR C=4.0 | 0.5603 |
| 4 | E-011 | char300 + menciones | SGD log-loss | 0.5579 |
| 5 | E-010 | char300 + menciones | ComplementNB a=0.3 | 0.5370 |
| 6 | E-014 | char300 + menciones + hashtags | LinearSVC C=0.2 | 0.5644 |
| 7 | E-015o | char300 + menciones | Ridge ordinal (puntos medios) | 0.3973 |
| 8 | E-015s | ídem + softmax(-\|d\|/5) | distribución suave | — |
| 9 | E-017 | word(1-2) 300k | LogReg OvR C=4.0 | 0.5015 |
| 10 | E-019b | 35 densas + SVD(64) de char300 | MLP(256,128) | 0.4071 |
| 11 | E-021a | char300 + menciones | Frank & Hall (5 binarios) | 0.5527 |
| 12 | E-021b | char300 + menciones | Centroides Rocchio + softmax | 0.4017 |
| 13 | E-028a | char300 + menciones | 5 SVM por par adyacente | 0.7428 (fronteras) |
| 14 | ADV-05 | char300 + menciones | NB-SVM OvR | 0.5641 |

El meta-modelo final (V12) toma esas 14 salidas (78 columnas tras las transformaciones log), les añade
las 195 features de la sección 3.3 y las 6 columnas del NB-SVM de palabra (ADV-07; Wang y Manning, «Baselines and Bigrams: Simple, Good Sentiment and Topic Classification», ACL 2012; Mukherjee y Liu, «Improving Gender Classification of Blog Authors», EMNLP 2010), formando la **matriz
de 279 columnas**, y combina dos cabezas — una regresión logística y un MLP(64) — promediando sus
probabilidades. V12 alcanzó **CV 0.6005** (0.6077 en el holdout congelado) y **0.62182 en el
leaderboard**, el primer salto grande del proyecto.

In [ ]:
# Reconstruccion de los 14 miembros (OOF con los folds canonicos de la seccion 2.4).
t0 = time.time()
oofs_miembros = construir_miembros_oof(train["text"], y, fold_idx, dense)
print(f"14 miembros reconstruidos en {time.time() - t0:.0f}s")

documentados = {"e013": 0.5628, "e016": 0.5658, "e009": 0.5603, "e011": 0.5579, "e010": 0.5370,
                "e014": 0.5644, "e015_ordinal": 0.3973, "e015_softmax": None, "e017": 0.5015,
                "e019b": 0.4071, "e021a": 0.5527, "e021b": 0.4017, "e028a": None, "adv05": 0.5641}
print(f"\n{'miembro':14} {'CV reconstruida':>15} {'documentada':>12}")
for nombre, oof in oofs_miembros.items():
    if oof.ndim == 1:
        pred = np.argmax(softmax_ordinal(oof), axis=1)
    else:
        pred = np.argmax(oof, axis=1)
    cv_miembro = float(f1_score(clase, pred, average="macro"))
    referencia = documentados.get(nombre)
    texto_ref = f"{referencia:.4f}" if referencia else "(n/d)"
    print(f"{nombre:14} {cv_miembro:15.4f} {texto_ref:>12}")

In [ ]:
# ADV-07 (NB-SVM de palabra) y la matriz 279 del campeon.
t0 = time.time()
adv07 = np.zeros((len(y), 6), dtype=np.float32)
for fold in range(5):
    tr = np.flatnonzero(fold_idx != fold)
    va = np.flatnonzero(fold_idx == fold)
    adv07[va] = nb_svm_margins(train["text"], y, tr, va, WORD)
print(f"ADV-07 reconstruido en {time.time() - t0:.0f}s")

meta273 = build_features_273(oofs_miembros, densas35(dense),
                             estructuradas.to_numpy(float), supervisadas)
matriz = matriz_279(meta273, adv07)
print("meta 273:", meta273.shape, "| matriz 279:", matriz.shape, matriz.dtype)
assert matriz.shape == (169336, 279)

# FASE 4 — Modelado

Esta fase recorre el modelado desde V18 hasta la frontera de 2026. Primero se documenta la búsqueda de
frontera, ordinal y selección de features (V18–V29), que dejó lecciones y cerró caminos; después se
re-ejecuta el corazón del campeón: los cuatro cuartos de V20, el ensemble V30 y la regla plug-in por
clase (Lipton, Elkan y Narayanaswamy, «Thresholding Classifiers to Maximize F1 Score», arXiv, 2014); a continuación se documentan los fracasos seleccionados; luego se re-ejecutan los componentes
neuronales V38–V43 (la superación del techo lineal); se documentan las falsaciones V44–V52, que
delimitan con precisión qué técnicas no aportan y por qué; y se cierra con el arco 2026 (V53–V57):
los componentes nuevos, el consenso de nueve modelos que alcanzó el récord histórico de validación
local y la auditoría matemática del techo.

## 4.2 V30: los cuatro cuartos, el ensemble y la regla plug-in

Con la matriz de 279 columnas fija, la pregunta pasó a ser cómo exprimirla. La respuesta llegó en cuatro
pasos (V19->V20, V29, V30 y la regla plug-in), cada uno con su acta y su verificación:

**V19 → V20 — «cuatro cuartos».** V19 añadió un árbol de boosting (LightGBM con configuración
pre-registrada; Ke et al., «LightGBM: A Highly Efficient Gradient Boosting Decision Tree», NeurIPS 2017) como tercer miembro del meta-modelo. V20 fue más allá y definió un candidato con cuatro
componentes de peso igual (0.25 cada uno): la regresión logística, el MLP(64), el árbol y una **cadena de
cinco umbrales acumulativos** (cinco clasificadores binarios «¿el rango es ≤ k?» que se corrigen para ser
monótonos y se convierten en distribución; Frank y Hall, «A Simple Approach to Ordinal Classification», ECML 2001). La cadena aporta la estructura ordinal de forma explícita.
V20 alcanzó **CV 0.6097782** y **0.62689** en el leaderboard.

**V29 — búsqueda con Optuna.** Una exploración sistemática de LightGBM (Ke et al., 2017) con Optuna (Akiba et al., «Optuna: A Next-generation Hyperparameter Optimization Framework», KDD 2019) encontró una
configuración distinta por fold (el mejor *trial* individual logró **0.6121698** de CV). Importante: las
configuraciones quedaron **congeladas** en el artefacto de la búsqueda — nada se re-optimiza después.

**V30 — el ensemble.** El campeón lineal del proyecto mezcla tres miembros — V20 (cuartos intactos), el
LightGBM de V29 y un LightGBM refinado por Optuna (Akiba et al., 2019) — con pesos que son la **media de los vectores óptimos
por fold, renormalizada** (`0.3548 / 0.3354 / 0.3098`, congelados). El resultado: **CV 0.6134471** y
**0.63302** en el leaderboard.

**La regla plug-in por clase.** La última pieza fue una regla de decisión (Lipton, Elkan y Narayanaswamy, «Thresholding Classifiers to Maximize F1 Score», arXiv, 2014) que reemplaza el `argmax`
plano por un `argmax` con **pesos por clase** (`[1.0000, 1.1749, 1.2721, 1.1630, 1.0010, 0.8128]`): los
pesos se ajustaron por ascenso por coordenadas en espacio logit, **dentro de cada fold** (usando los
otros cuatro como validación interna), y el vector final es la media de los cinco, congelada en un
pre-registro. La intuición es que la macro-F1 no trata igual a todas las clases: mover la frontera de una
clase mayoritaria puede costar más de lo que ayuda, y el plug-in re-calibra esa decisión sin tocar el
modelo. El sanity check del proyecto exige reproducir exactamente la mejora documentada
(**0.6134471 → 0.6159732487657944** en el OOF canónico, sin ningún cambio en filas de alta confianza).

In [ ]:
# V19/V20: los tres miembros de V19 y la cadena de cinco umbrales (OOF, folds canonicos).
t0 = time.time()
p19_logreg, p19_mlp, p19_arbol = v19_oofs(matriz, y, clase, fold_idx)
seis_oof = v20_cadena_oof(matriz, clase, fold_idx)
v20_oof = 0.25 * p19_logreg + 0.25 * p19_mlp + 0.25 * p19_arbol + 0.25 * seis_oof
print(f"reconstruccion V19/V20: {time.time() - t0:.0f}s\n")
for nombre, probabilidad in [("V19 logreg", p19_logreg), ("V19 MLP", p19_mlp),
                             ("V19 arbol", p19_arbol), ("cadena 5 umbrales", seis_oof),
                             ("V20 (cuartos)", v20_oof)]:
    print(f"  {nombre:20} macro-F1 {macro_f1(clase, np.argmax(probabilidad, axis=1)):.6f}")
print("\n  V20 documentado: CV 0.6097782")

In [ ]:
# V29/V30: los dos LightGBM con configuracion congelada por fold, la mezcla y la regla plug-in.
t0 = time.time()
v29_oof = lgbm_refit_oof(matriz, clase, fold_idx, V29_CONFIGS,
                         base={"random_state": SEED})
v30_oof = lgbm_refit_oof(matriz, clase, fold_idx, V30_CONFIGS)
oof_v30 = mezcla(v20_oof, v29_oof, v30_oof)
print(f"reconstruccion V29/V30: {time.time() - t0:.0f}s\n")
print(f"  V29 individual   : {macro_f1(clase, np.argmax(v29_oof, axis=1)):.7f} (doc 0.6121698)")
print(f"  V30 individual   : {macro_f1(clase, np.argmax(v30_oof, axis=1)):.7f} (doc 0.6122387)")
cv_v30_oof = macro_f1(clase, np.argmax(oof_v30, axis=1))
cv_plugin_oof = macro_f1(clase, regla_plug_in(oof_v30))
print(f"  V30 mezcla       : {cv_v30_oof:.7f} (doc 0.6134471)")
print(f"  V30 + plug-in    : {cv_plugin_oof:.10f} (doc 0.6159732487657944)")

In [ ]:
# Verificacion del orden de clases (diagonal) y figura de la progresion hasta el plug-in.
medias_condicionales = np.zeros((6, 6))
for fila in range(6):
    medias_condicionales[fila] = oof_v30[clase == fila].mean(axis=0)
diagonal = [int(np.argmax(medias_condicionales[fila])) for fila in range(6)]
assert diagonal == list(range(6)), f"orden de clases roto: {diagonal}"
print("diagonal de orden de clases verificada:", diagonal)

fig, ejes = plt.subplots(1, 2, figsize=(11.5, 3.8))
imagen = ejes[0].imshow(medias_condicionales, cmap="Blues")
ejes[0].set_title("E[P(clase) | rango real] sobre el OOF de V30")
ejes[0].set_xticks(range(6))
ejes[0].set_xticklabels(CLASSES, rotation=45, fontsize=8)
ejes[0].set_yticks(range(6))
ejes[0].set_yticklabels(CLASSES, fontsize=8)
plt.colorbar(imagen, ax=ejes[0], fraction=0.046)
hitos = ["V12", "V20", "V30", "V30\n+plug-in"]
valores = [0.6005, 0.6097782, cv_v30_oof, cv_plugin_oof]
ejes[1].bar(hitos, valores, color=["#a0aec0", "#90cdf4", "#4299e1", "#2b6cb0"])
ejes[1].set_ylim(0.59, 0.62)
ejes[1].set_ylabel("macro-F1 (CV agrupada)")
ejes[1].set_title("Progresion hasta la regla plug-in")
for posicion, valor in enumerate(valores):
    ejes[1].text(posicion, valor + 0.0004, f"{valor:.4f}", ha="center", fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Lado TEST del campeon: la matriz de 279 de test (representaciones + 13 modelos + features + NB-SVM).
def adv05_test(textos_train, etiquetas, textos_test):
    """ADV-05 lado test: NB-SVM char300+menciones ajustado sobre todo el train."""
    textos_todo = pd.concat([textos_train, textos_test], ignore_index=True)
    etiquetas_todo = np.concatenate([etiquetas, np.array(["13-17"] * len(textos_test))])
    tr = np.arange(len(textos_train))
    va = np.arange(len(textos_train), len(textos_train) + len(textos_test))
    config = {**CHAR_300K, "mentions": {"config": MENTION, "pattern": r"@\w+"}}
    return nb_svm_margins(textos_todo, etiquetas_todo, tr, va, config)


t0 = time.time()
reps_test = representaciones_test(train, test)
bloques_72 = base_models_test(reps_test, y, densas35(dense), densas35(dense_test),
                              train["text"], test["text"])
nb_test = adv05_test(train["text"], y, test["text"])
nb_word_test_matrix = nb_word_test(train["text"], y, test["text"])
matriz_test = matriz_test_fresca(bloques_72, nb_test, densas35(dense_test),
                                 estructuradas_test.to_numpy(float), supervisadas_te, nb_word_test_matrix)
print(f"lado test completo: {time.time() - t0:.0f}s | matriz test {matriz_test.shape}")
assert matriz_test.shape == (112892, 279)

In [ ]:
# Probabilidades de test del candidato V30 (sin plug-in): cuartos + los dos LightGBM reajustados.
t0 = time.time()
p20_test = v20_test(matriz, matriz_test, y, clase)
p29_test = lgbm_refit_test(matriz, clase, matriz_test, V29_CONFIGS)
p30_test = lgbm_refit_test(matriz, clase, matriz_test, V30_CONFIGS)
prob_v30_test = mezcla(p20_test, p29_test, p30_test)
print(f"test V30: {time.time() - t0:.0f}s | forma {prob_v30_test.shape}")
print("decisiones V30+plug-in sobre test:")
pred_v30_test = regla_plug_in(prob_v30_test)
for posicion, etiqueta in enumerate(CLASSES):
    print(f"  {etiqueta}: {int((pred_v30_test == posicion).sum()):,}")

### La segunda partición del campeón (semilla 987654)

El proyecto verificó el linaje V30 también sobre una **segunda partición agrupada**: misma
agrupación de clústeres, semilla 987654, con solape parcial con la canónica — es una
re-ejecución en otra partición, no una partición independiente, y así quedó declarado. Esta
segunda evaluación cumple dos papeles: confirma el linaje fuera de la partición canónica
(0.6144120 de base y 0.6164700 con el plug-in, los valores documentados) y es, además, la que el
proyecto usa como miembro V30 en la **evaluación publicada del campeón V48 y de los consensos del
arco 2026 (secciones 5.2 y 4.6)**: su OOF es el que entró en los consensos geométricos de los
récords. La receta es idéntica (cuartos + dos LightGBM con configuraciones y pesos congelados); la
única diferencia es que los reajustes de esta partición usan el techo completo de árboles, sin early
stopping (Ke et al., 2017), tal como los ejecutó el script original de la confirmación. La partición se
genera aquí mismo con la misma función de la sección 2.4 y la semilla 987654, de modo que el código
queda legible y verificable en el punto donde se usa.

In [ ]:
# Segunda particion (semilla 987654): la misma regla sobre los mismos clusters, generada junto a su uso.
fold_idx2 = generar_folds(y, cluster_de[:n_train], semilla=987654)
print("segunda particion (semilla 987654):", np.bincount(fold_idx2).tolist())

In [ ]:
# La segunda particion del campeon (semilla 987654): validacion del linaje V30.
t0 = time.time()
v20_segunda = cuartos_v20_oof(matriz, y, clase, fold_idx2)
v29_segunda = lgbm_refit_oof_sin_es(matriz, clase, y, fold_idx2, V29_CONFIGS)
v30_segunda = lgbm_refit_oof_sin_es(matriz, clase, y, fold_idx2, V30_CONFIGS)
oof_v30_segunda = mezcla(v20_segunda, v29_segunda, v30_segunda)
print(f"segunda particion reconstruida: {time.time() - t0:.0f}s")
print(f"  V30 segunda        : {macro_f1(clase, np.argmax(oof_v30_segunda, axis=1)):.7f} "
      f"(doc 0.6144120)")
print(f"  V30 + plug-in segunda: {macro_f1(clase, regla_plug_in(oof_v30_segunda)):.7f} "
      f"(doc 0.6164700)")

## 4.4 Los componentes neuronales V38–V43: la superación del techo lineal

Tras el plug-in, el diagnóstico del proyecto fue claro: el modelo lineal había llegado a su techo
(oráculo de 0.6935, sección 5.4) y el margen restante exigía **modelos no lineales sobre la misma matriz
de 279 columnas**. Esta sub-fase construyó seis componentes, cada uno añadiendo una técnica sobre el
anterior, todos evaluados con los folds canónicos y las mismas reglas:

- **V38 — TabularResNet:** una red residual para datos tabulares (proyección, dos bloques residuales con
  normalización por lotes (Ioffe y Szegedy, «Batch Normalization: Accelerating Deep Network Training by Reducing Internal Covariate Shift», ICML 2015) y activación SiLU, dropout 0.2; Gorishniy, Rubachev, Khrulkov y Babenko, «Revisiting Deep Learning Models for Tabular Data», NeurIPS 2021) entrenada con entropía cruzada sobre GPU.
  Superó al techo lineal en validación (CV +0.00068 sobre V30) y demostró que había señal no lineal.
- **V39 — pérdida ordinal + trío:** la misma red con una pérdida que penaliza la **distancia ordinal**
  esperada (equivocarse por un rango cuesta menos que por cinco; Frogner, Zhang, Mobahi, Araya y Poggio, «Learning with a Wasserstein Loss», NeurIPS 2015) y un trío con HistGradientBoosting; la
  combinación con V30 (0.84/0.12/0.04) pasó el gate.
- **V40 — atención Squeeze-and-Excitation:** compuertas que re-ponderan las 256 dimensiones internas de
  cada bloque según su utilidad (mecanismo SE, reducción 4; Hu, Shen y Sun, «Squeeze-and-Excitation Networks», CVPR 2018). La pérdida ordinal se mantiene.
- **V41 — Stochastic Weight Averaging (SWA):** promediar los pesos de las últimas 8 épocas del
  entrenamiento para aterrizar en una zona más plana del espacio de parámetros (mejor generalización; Izmailov, Podoprikhin, Garipov, Vetrov y Wilson, «Averaging Weights Leads to Wider Optima and Better Generalization», UAI 2018).
- **V42 — suavizado de etiquetas + SWA:** el suavizado 0.10 reparte un poco de masa de la clase correcta
  hacia las demás, atenuando la sobreconfianza (Szegedy, Vanhoucke, Ioffe, Shlens y Wojna, «Rethinking the Inception Architecture for Computer Vision», CVPR 2016; Müller, Kornblith y Hinton, «When Does Label Smoothing Help?», NeurIPS 2019), en la formulación de etiquetas suaves ordinales revisada de SORD (Díaz y Marathe, «Soft Labels for Ordinal Regression», CVPR 2019) combinada con la penalización de distancia; con enfriamiento cosenoidal del learning rate (Loshchilov y Hutter, «SGDR: Stochastic Gradient Descent with Warm Restarts», ICLR 2017) y 10
  checkpoints SWA. OOF standalone documentado: 0.613276.
- **V43 — pérdida de Cramér-von Mises:** la pérdida de mayor fundamento matemático de la familia:
  compara la **función de distribución acumulada** de la predicción contra la de la etiqueta
  (distancia cuadrática acumulada, la W₂ de Wasserstein en una dimensión; Cramér, «On the composition of elementary errors: First paper: Mathematical deductions», Scandinavian Actuarial Journal, 1928; von Mises, «Wahrscheinlichkeitsrechnung und ihre Anwendung in der Statistik und theoretischen Physik», Franz Deuticke, 1931; Székely y Rizzo, «Energy statistics: A class of statistics based on energy distances», Journal of Statistical Planning and Inference, 2013), lo que premia predicciones
  «cerca» en el orden incluso cuando fallan. OOF standalone documentado: 0.612840.

Cuatro de los seis componentes (V39-ordinal, V40, V42 y V43), más el HGB de V39 y el ensemble V30,
alimentan al campeón V48. La reproducibilidad es honesta: las
redes se entrenan con semillas fijas por fold y su salida de test es el promedio de los cinco modelos
(promedio que actúa como un mini-ensamble); el HGB y las redes de V38/V39 usan un modelo final entrenado
sobre todo el train. Entre entornos distintos (por ejemplo, GPU AMD vs NVIDIA) puede haber diferencias
numéricas del orden de 1e-6 —inherentes al hardware—, pero las decisiones y las métricas se mantienen
estables; el notebook lo declara explícitamente en su protocolo de reproducibilidad.

In [ ]:
# Componentes neuronales V38-V43: reconstruccion 5-fold completa (GPU si esta disponible).
componentes_nn = {}
referencias_nn = {"v38": None, "v39_ordinal": None, "v40": None, "v41": None,
                  "v42": 0.613276, "v43": 0.612840}
for nombre in ["v38", "v39_ordinal", "v40", "v41", "v42", "v43"]:
    t0 = time.time()
    oof_nn, test_nn = entrenar_componente(nombre, matriz, matriz_test, clase, fold_idx)
    componentes_nn[nombre] = (oof_nn, test_nn)
    cv_nn = macro_f1(clase, np.argmax(oof_nn * PESOS_PLUG_IN, axis=1))
    referencia = referencias_nn.get(nombre)
    texto_ref = f" (doc {referencia:.6f})" if referencia else ""
    print(f"  {nombre:12} OOF standalone {cv_nn:.6f}{texto_ref} | {time.time() - t0:.0f}s")

In [ ]:
# El HGB de V39 (miembro de arboles) y la figura de la familia neuronal completa.
t0 = time.time()
hgb_oof, hgb_test = hgb_componente(matriz, matriz_test, clase, fold_idx)
cv_hgb = macro_f1(clase, np.argmax(hgb_oof * PESOS_PLUG_IN, axis=1))
print(f"  v39_hgb      OOF standalone {cv_hgb:.6f} | {time.time() - t0:.0f}s")

nombres = list(componentes_nn) + ["v39_hgb"]
valores = [macro_f1(clase, np.argmax(componentes_nn[n][0] * PESOS_PLUG_IN, axis=1))
           for n in componentes_nn] + [cv_hgb]
fig, eje = plt.subplots(figsize=(9.5, 3.4))
eje.bar(nombres, valores, color="#2b6cb0", alpha=0.9)
eje.axhline(cv_v30_oof, color="#c53030", linestyle="--", linewidth=1,
            label=f"V30 lineal ({cv_v30_oof:.4f})")
for posicion, valor in enumerate(valores):
    eje.text(posicion, valor + 0.0004, f"{valor:.4f}", ha="center", fontsize=8)
eje.set_ylim(0.58, 0.63)
eje.set_ylabel("macro-F1 OOF standalone")
eje.set_title("Familia neuronal V38-V43 + HGB (cada uno por separado)")
eje.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 4.6 V53-V57: la frontera 2026, el récord de CV y el techo estacionario [RE/DOC]

*Sección mixta: re-ejecuta los componentes y consensos del arco 2026 (V47/V51/V53, los PoE de V53/V54/V56 y la auditoría V57) y documenta lo que exige artefactos congelados o entrenamientos largos (V55 con CatBoost). Fuentes: `RUNLOG.md` (entradas V53-V57), `DECISIONS.md` (D-106 a D-110), `experiments.csv` y los JSON `experiments/v53_*.json` a `v57_*.json`. La procedencia de cada mecanismo en el estado del arte está en `reports/state_of_art.csv` (filas C13-C17), `reports/age_literature_methods_registry.csv` y `references.bib`.*

Tras el récord de V48 (CV 0.6186451; LB 0.63724), el proyecto siguió empujando la frontera con tres piezas nuevas —una arquitectura densa, dos variantes del transporte ordinal de Cramér y un árbol simétrico— más una auditoría matemática del consenso. La pregunta era si el espacio logarítmico admitía miembros aún más diversos sin degradar las cohortes minoritarias; la respuesta quedó medida: el techo de validación local subió a **0.6189929** (V56, 5/5 folds positivos, bootstrap pareado p = 0.0010 frente a V48) y la **selección dual V48/V56** quedó ratificada para Kaggle (V48 conserva el mejor LB, 0.63724; V56 aporta la mejor validación local, LB 0.63717).

### 4.6.1 V53 — Dense-SE-TabularNet y el consenso de siete modelos

La red densa conecta cada bloque con todos los siguientes (192 -> 320 -> 448 -> 576) para preservar las representaciones tempranas sin dilución aditiva (Huang, Liu, van der Maaten y Weinberger, «Densely Connected Convolutional Networks», CVPR 2017), con compuertas Squeeze-and-Excitation (Hu, Shen y Sun, «Squeeze-and-Excitation Networks», CVPR 2018) y la pérdida de Cramér consistente-suavizada (Cramér, 1928; Székely y Rizzo, 2013; véanse las secciones 4.4 y 4.5). Como clasificador individual rinde 0.6111343; integrada al consenso geométrico de siete modelos (V30, V43, V42, V40, V39-ordinal, V39-HGB y la red densa) con la regla plug-in congelada alcanza **CV 0.6188259** (bootstrap p = 0.0280 vs V48) y **LB 0.63706**.

### 4.6.2 V54 — el consenso jerárquico multi-pérdida

La rama de transporte ordinal se unificó en un ensamble jerárquico: 0.80 de Cramér simétrico (V43) más 0.20 de Cramér asimétrico ponderado con recorte a [-6, +6] (V47, documentado en la sección 4.5) más la red densa y los demás miembros, todo en espacio logarítmico (Hinton, «Training Products of Experts by Minimizing Contrastive Divergence», Neural Computation, 2002; Heskes, «Selecting Weighting Factors in Logarithmic Opinion Pools», NeurIPS, 1998). El diagnóstico de V53 —una red densa sin amortiguar contraía la cohorte 13-17— se corrigió con el peso jerárquico. Resultado: **CV 0.6188887** (p = 0.0050 vs V48; 180 filas modificadas vs V48, 0.16%) y **LB 0.63712**.

### 4.6.3 V55 — el octa-modelo limpio y el blindaje del 30% privado

Al constatar que las redes densas rinden por debajo de las residuales en aislamiento (0.6111 vs ~0.6128), V55 excluyó formalmente las arquitecturas subóptimas e inyectó un árbol simétrico CatBoost con peso 0.00805 (Prokhorenkova, Gusev, Vorobev, Dorogush y Gulin, «CatBoost: unbiased boosting with categorical features», NeurIPS 2018) en un consenso PoE de ocho modelos de máxima calidad. Resultado: CV 0.6182436 (4/5 folds positivos vs V48) y **LB 0.63707**, adoptado como protocolo de blindaje para la evaluación privada.

### 4.6.4 V56 — el consenso no lineal de nueve modelos (récord histórico de CV)

La integración armónica definitiva: nueve modelos de sesgos ortogonales en espacio logarítmico —V30 (LightGBM), V43 (Cramér-SWA), V47 (Cramér asimétrico winsorizado), V42 (suavizado-SWA), V40 (SE ordinal), V39-ordinal, V39-HGB, V51 (Cramér consistente) y V53 (red densa)— con la regla plug-in (Lipton, Elkan y Narayanaswamy, 2014). Resultado: **CV 0.6189929 — máximo histórico absoluto del proyecto** (delta +0.0003478 vs V48), con 5/5 folds positivos frente a V54, bootstrap pareado p = 0.0010 vs V48, 194 filas modificadas vs V48 (0.17%) y **LB 0.63717** confirmado en Kaggle (puesto #1).

### 4.6.5 V57 — geometría del símplex y concavidad Power-Mean

La auditoría matemática de cierre caracterizó la cuenca de generalización del consenso: sensibilidad coordenual en el símplex de 9 pesos (Delta^8), concavidad de la familia continua Power-Mean para alpha en [-0.15, +0.15], desalineamiento de pérdidas sustitutas continuas en validación anidada y confiabilidad decil a decil sobre las 169.336 predicciones OOF. La conclusión formal: V56 habita un **óptimo estacionario absoluto** inmune al sobreajuste del leaderboard público, lo que ratificó la selección dual V48/V56 y el blindaje del 30% privado (D-110).

| Versión | Mecanismo | CV local | Kaggle LB | Cierre |
|---|---|---|---|---|
| V53 | Dense-SE-TabularNet + PoE de 7 modelos | 0.6188259 | 0.63706 | Ex-récord CV (p = 0.0280 vs V48) |
| V54 | PoE jerárquico de 8 (Cramér simétrico + asimétrico) | 0.6188887 | 0.63712 | Ex-récord CV (p = 0.0050) |
| V55 | Octa-modelo limpio + CatBoost simétrico | 0.6182436 | 0.63707 | Protocolo de blindaje (4/5 vs V48) |
| V56 | PoE no lineal de 9 modelos + plug-in | **0.6189929** | 0.63717 | **Récord histórico de CV** (5/5; p = 0.0010) |
| V57 | Auditoría matemática del símplex 9D | 0.6189929 | 0.63717 | Techo estacionario confirmado |

El estado final del proyecto queda así: **V48 es el campeón oficial de leaderboard** (0.63724, re-ejecutado en la sección 5.2 de este notebook) y **V56 el récord de validación local** (0.6189929), con la selección dual ratificada por la auditoría de V57. Las celdas siguientes re-ejecutan el arco completo —los tres componentes nuevos, los consensos publicados y la auditoría matemática—; V55 queda documentado (su miembro CatBoost cuesta ~19 minutos de entrenamiento y no es récord).

### 4.6.6 Re-ejecución: los tres componentes nuevos

Los componentes del arco 2026 se reentrenan con la receta congelada de sus scripts (semillas 42 + fold·100, 16 épocas, enfriamiento cosenoidal, SWA desde la época 6 y test como media de los cinco folds): V47 (Cramér asimétrico ponderado con recorte a [-6, +6]), V51 (Cramér consistente-suavizada; standalone documentado 0.6128371, D-104) y V53 (red densa multi-escala con compuertas SE; standalone documentado 0.6111343, D-106).

In [ ]:
# Los tres componentes nuevos del arco 2026, con la receta congelada de sus scripts.
t0 = time.time()
componentes_2026 = {}
for nombre in ("v47", "v51", "v53"):
    oof_nuevo, test_nuevo = entrenar_componente(nombre, matriz, matriz_test, clase, fold_idx)
    componentes_2026[nombre] = (oof_nuevo, test_nuevo)
    cv_nuevo = macro_f1(clase, np.argmax(oof_nuevo, axis=1))
    print(f"  {nombre:4} OOF standalone {cv_nuevo:.6f} | acumulado {time.time() - t0:.0f}s")

### 4.6.7 Re-ejecución: los consensos del arco 2026

Los tres consensos publicados se reconstruyen con el esquema exacto del proyecto: el miembro V30 usa su OOF de la segunda partición (la misma convención de la evaluación del campeón, sección 5.2), las probabilidades se combinan en espacio logarítmico con los pesos congelados de cada versión y sobre el consenso se aplica la regla plug-in. Documentados: V53 0.6188259 (D-106), V54 0.6188887 (D-107) y V56 0.6189929 (D-109, récord histórico).

In [ ]:
# Los consensos del arco 2026 (V53, V54 y V56) en el esquema publicado del proyecto.
oofs_consenso = {
    "v30": oof_v30_segunda, "v43": componentes_nn["v43"][0], "v42": componentes_nn["v42"][0],
    "v40": componentes_nn["v40"][0], "v39_ordinal": componentes_nn["v39_ordinal"][0],
    "v39_hgb": hgb_oof, "v47": componentes_2026["v47"][0],
    "v51_consistent": componentes_2026["v51"][0], "v53_dense": componentes_2026["v53"][0],
}
documentados_consenso = {"v53": 0.6188259, "v54": 0.6188887, "v56": 0.6189929}
for version in ("v53", "v54", "v56"):
    espec = CONSENSOS[version]
    prob_consenso = poe_ponderado([oofs_consenso[m] for m in espec["miembros"]], espec["pesos"])
    cv_consenso = macro_f1(clase, regla_plug_in(prob_consenso))
    print(f"  {version.upper()} CV {cv_consenso:.7f} (doc {documentados_consenso[version]:.7f})")

In [ ]:
# La submission del record V56 (nueve modelos) y su diferencia con la sellada del campeon V48.
test_consenso = {
    "v30": prob_v30_test, "v43": componentes_nn["v43"][1], "v42": componentes_nn["v42"][1],
    "v40": componentes_nn["v40"][1], "v39_ordinal": componentes_nn["v39_ordinal"][1],
    "v39_hgb": hgb_test, "v47": componentes_2026["v47"][1],
    "v51_consistent": componentes_2026["v51"][1], "v53_dense": componentes_2026["v53"][1],
}
prob_v56_test = poe_ponderado([test_consenso[m] for m in CONSENSOS["v56"]["miembros"]],
                              CONSENSOS["v56"]["pesos"])
pred_v56 = regla_plug_in(prob_v56_test)
pesos_v48 = [0.5628, 0.0832, 0.1724, 0.0754, 0.0513, 0.0549]
p_v48_test = poe_ponderado([test_consenso[m] for m in
                            ("v30", "v43", "v42", "v40", "v39_ordinal", "v39_hgb")], pesos_v48)
pred_v48 = regla_plug_in(p_v48_test)
filas_v56 = int((pred_v56 != pred_v48).sum())
print(f"submission V56 reconstruida: {filas_v56} filas distintas de la V48 (documentado 194, 0.17%)")

### 4.6.8 Re-ejecución: la auditoría V57

La auditoría matemática se repite sobre el OOF del consenso V56: sensibilidad coordenual en el símplex de nueve pesos (perturbación ±20% por coordenada), la curva de la familia Power-Mean alrededor del PoE, la confiabilidad decil a decil y la brecha de menciones. La conclusión documentada (D-110): el consenso habita un óptimo estacionario absoluto —9/9 modelos estacionarios en la corrida original, superficie cóncava con vértice en el PoE y calibración monótona—; la reconstrucción del notebook clasifica 6/9 por el umbral de estado bajo la deriva de entorno, manteniendo el vértice en alpha = 0. Ese resultado blindó la selección dual frente al 30% privado.

In [ ]:
# La auditoria V57 sobre el consenso V56: simplex, Power-Mean, deciles y menciones.
t0 = time.time()
n_mention = train["text"].str.count(r"@\w+").to_numpy()
resumen_v57 = analisis_completo(
    [oofs_consenso[m] for m in CONSENSOS["v56"]["miembros"]], clase,
    CONSENSOS["v56"]["pesos"], n_mention=n_mention,
    nombres=CONSENSOS["v56"]["miembros"], fold_idx=fold_idx)
print(f"V57: f1 base {resumen_v57['f1_base']:.10f} (doc 0.6189929489711685) | "
      f"{time.time() - t0:.1f}s")
estacionarios = sum(1 for r in resumen_v57["sensibilidad_simplex"] if r["estado"] == "ESTACIONARIO")
print(f"  simplex: {estacionarios}/{len(resumen_v57['sensibilidad_simplex'])} modelos "
      f"estacionarios ante perturbaciones de +-20%")
optimo = [r for r in resumen_v57["curva_power_mean"] if not r["estado"].startswith("SUB")]
print(f"  Power-Mean: optimo en alpha={optimo[0]['alpha']:.2f} (PoE) sobre "
      f"{len(resumen_v57['curva_power_mean'])} puntos")

In [ ]:
# Figura: la concavidad Power-Mean del consenso (alpha=0 es el PoE).
alphas = [r["alpha"] for r in resumen_v57["curva_power_mean"]]
f1s = [r["macro_f1"] for r in resumen_v57["curva_power_mean"]]
fig, eje = plt.subplots(figsize=(8.5, 3.2))
eje.plot(alphas, f1s, marker="o", color="#2b6cb0")
eje.axvline(0.0, color="#2f855a", linestyle="--", linewidth=1, label="PoE (alpha = 0)")
eje.set_xlabel("alpha (familia Power-Mean)")
eje.set_ylabel("macro-F1 (OOF, 169.336 filas)")
eje.set_title("Concavidad Power-Mean del consenso V56 (auditoria V57)")
eje.legend(fontsize=8)
plt.tight_layout()
plt.show()

# FASE 5 — Evaluación

## 5.1 El protocolo de validación y su reproducibilidad

Toda la evaluación del proyecto se apoya en el mismo protocolo, definido en la Fase 2 y usado sin
excepciones: **validación cruzada de 5 folds agrupados por clústeres de near-duplicados** (folds
generados con la regla documentada de la sección 2.4; Kaufman, Rosset, Perlich y Stitelman, «Leakage in Data Mining: Formulation, Detection, and Avoidance», ACM Transactions on Knowledge Discovery from Data, 2012), con la métrica **macro-F1 sin ponderar** calculada de dos maneras
complementarias — *pooled* (todas las predicciones juntas) y media por folds — más pruebas de
significancia cuando una decisión estaba en juego (bootstrap (Efron, «Bootstrap Methods: Another Look at the Jackknife», The Annals of Statistics, 1979) pareado con re-muestreo de clústeres, B=1000,
intervalo de confianza al 95 % y fracción de folds positivos). Las decisiones importantes se confirmaron
además en una **segunda partición** con semilla distinta.

Este diseño tiene una propiedad que conviene explicitar: el CV agrupado es **conservador** respecto del
leaderboard (sección 5.5). Al eliminar los near-duplicados entre folds, mide la capacidad de generalizar
a textos nuevos; el leaderboard, en cambio, contiene near-duplicados del train y los «cobra», de modo que
el puntaje real queda sistemáticamente por encima del CV local. El proyecto verificó esa brecha en cada
envío y la usó como calibración, nunca como objetivo.

## 5.2 El campeón V48: consenso geométrico de seis modelos

El modelo entregado combina los seis componentes de la Fase 4 mediante un **Product-of-Experts**
(consenso bayesiano multiplicativo; Hinton, «Training Products of Experts by Minimizing Contrastive Divergence», Neural Computation, 2002): las probabilidades de cada modelo se combinan en **espacio
logarítmico** con pesos congelados (Heskes, «Selecting Weighting Factors in Logarithmic Opinion Pools», NeurIPS 1998),

$$\log P_{\text{geom}}(y = k \mid x) \propto \sum_{m=1}^{6} w_m \log P_m(y = k \mid x)$$

con $w = [0.5628, 0.0832, 0.1724, 0.0754, 0.0513, 0.0549]$ (V30, V43, V42, V40, V39-ordinal,
V39-HGB), pesos óptimos por validación cruzada anidada. La intuición del PoE: si varios modelos
independientes asignan probabilidad baja a una clase, el producto la hunde más rápido que el promedio;
el consenso es más agresivo con el desacuerdo y más estable con el acuerdo. La ablación de operadores
(sección 4.5) confirmó que el geométrico es el óptimo de su familia. El miembro V30 del consenso usa
su OOF de la segunda partición —el esquema con el que el proyecto selló el récord—; la celda reporta
además la variante estricta con el V30 canónico, para que el detalle sea verificable. Sobre el
consenso se aplica la misma **regla plug-in** (Lipton, Elkan y Narayanaswamy, «Thresholding Classifiers to Maximize F1 Score», arXiv, 2014) del campeón lineal, y el resultado se envía al
leaderboard: **CV 0.6186451** y **0.63724** en Kaggle (puesto #1).

In [ ]:
# El campeon V48: PoE geometrico sobre los seis componentes, version OOF y version test.
PESOS_GEOM = np.array([0.5628, 0.0832, 0.1724, 0.0754, 0.0513, 0.0549])
PESOS_GEOM = PESOS_GEOM / PESOS_GEOM.sum()


def poe_log_espacio(modelos):
    """Consenso Product-of-Experts: suma ponderada de log-probabilidades, normalizada."""
    eps = 1e-12
    log_geom = sum(w * np.log(np.clip(modelo, eps, 1.0))
                   for w, modelo in zip(PESOS_GEOM, modelos))
    p_geom = np.exp(log_geom - np.max(log_geom, axis=1, keepdims=True))
    return p_geom / p_geom.sum(axis=1, keepdims=True)


# Protocolo publicado del proyecto: el miembro V30 del consenso usa su OOF de la segunda
# particion, tal como lo evaluo el acta D-101 y la seccion 47 del notebook consolidado.
modelos_oof = [oof_v30_segunda, componentes_nn["v43"][0], componentes_nn["v42"][0],
               componentes_nn["v40"][0], componentes_nn["v39_ordinal"][0], hgb_oof]
p_geom_oof = poe_log_espacio(modelos_oof)
cv_v48 = macro_f1(clase, regla_plug_in(p_geom_oof))
print(f"V48 reconstruido (OOF, con plug-in): {cv_v48:.7f} (documentado 0.6186451)")

# Variante estricta con el V30 canonico (se declara en vez de esconderse).
modelos_oof_canonica = [oof_v30, componentes_nn["v43"][0], componentes_nn["v42"][0],
                        componentes_nn["v40"][0], componentes_nn["v39_ordinal"][0], hgb_oof]
cv_v48_canonica = macro_f1(clase, regla_plug_in(poe_log_espacio(modelos_oof_canonica)))
print(f"variante con el V30 canonico      : {cv_v48_canonica:.7f}")

modelos_test = [prob_v30_test, componentes_nn["v43"][1], componentes_nn["v42"][1],
                componentes_nn["v40"][1], componentes_nn["v39_ordinal"][1], hgb_test]
p_geom_test = poe_log_espacio(modelos_test)
pred_final = regla_plug_in(p_geom_test)
print("consenso test:", p_geom_test.shape,
      "| filas normalizadas:", bool(np.allclose(p_geom_test.sum(axis=1), 1.0)))

## 5.3 Análisis de error

El análisis de error del campeón se concentra en la estructura ordinal del problema. La matriz de
confusión sobre el OOF muestra el patrón esperado y deseable: los errores viven en las **fronteras
adyacentes** (confundir `18-24` con `25-34`), mientras que los errores lejanos son marginales. Esto no
es un accidente: es exactamente lo que la pérdida de Cramér (V43; Cramér, «On the composition of elementary errors: First paper: Mathematical deductions», Scandinavian Actuarial Journal, 1928; von Mises, «Wahrscheinlichkeitsrechnung und ihre Anwendung in der Statistik und theoretischen Physik», Franz Deuticke, 1931) y la cadena de umbrales (V20; Frank y Hall, «A Simple Approach to Ordinal Classification», ECML 2001) están
diseñadas para favorecer. La tabla por clase revela además que los extremos son los mejor clasificados
(65-xx 0.7446; 13-17 0.6845) y que los rangos centrales (25-34 0.5051; 18-24 0.5306) concentran el
error, por el solapamiento de vocabulario entre cohortes vecinas.

In [ ]:
# Matriz de confusion y metricas por clase del campeon (sobre el OOF).
pred_oof_final = regla_plug_in(p_geom_oof)
confusion = np.zeros((6, 6), dtype=int)
for real, predicho in zip(clase, pred_oof_final):
    confusion[real, predicho] += 1

fig, ejes = plt.subplots(1, 2, figsize=(12, 4.2))
imagen = ejes[0].imshow(confusion, cmap="Blues")
for fila in range(6):
    for columna in range(6):
        ejes[0].text(columna, fila, f"{confusion[fila, columna]:,}", ha="center", va="center",
                     fontsize=7, color="black" if confusion[fila, columna] < confusion.max() * 0.7
                     else "white")
ejes[0].set_xticks(range(6))
ejes[0].set_xticklabels(CLASSES, rotation=45, fontsize=8)
ejes[0].set_yticks(range(6))
ejes[0].set_yticklabels(CLASSES, fontsize=8)
ejes[0].set_xlabel("predicho")
ejes[0].set_ylabel("real")
ejes[0].set_title("Matriz de confusion (OOF del campeon)")
plt.colorbar(imagen, ax=ejes[0], fraction=0.046)

from sklearn.metrics import f1_score as f1_por_clase

f1_clases = [float(f1_por_clase(clase, pred_oof_final, average=None,
                                labels=list(range(6)))[posicion]) for posicion in range(6)]
ejes[1].bar(CLASSES, f1_clases, color="#2b6cb0")
for posicion, valor in enumerate(f1_clases):
    ejes[1].text(posicion, valor + 0.005, f"{valor:.3f}", ha="center", fontsize=8)
ejes[1].set_ylim(0, 0.75)
ejes[1].set_title(f"F1 por rango (macro-F1 = {cv_v48:.4f})")
plt.xticks(rotation=30, fontsize=8)
plt.tight_layout()
plt.show()

vecinos = sum(int(confusion[k, k + 1] + confusion[k + 1, k]) for k in range(5))
lejanos = int(confusion.sum() - confusion.trace() - vecinos)
print(f"aciertos: {int(confusion.trace()):,} | confusiones adyacentes: {vecinos:,} "
      f"| confusiones lejanas: {lejanos:,}")
print(f"proporcion de errores que son de frontera: {vecinos / (vecinos + lejanos) * 100:.1f}%")

# FASE 6 — Despliegue

## 6.1 La submission final

El pipeline de despliegue es deliberadamente simple y verificado: se toma el consenso geométrico de los
seis componentes sobre el test, se aplica la regla plug-in congelada, y se escribe el archivo de envío
con el formato exacto de la competición (`id, age_range`). El proyecto sometió esa construcción a un
protocolo de integridad de ocho comprobaciones (número de filas, columnas exactas, identificadores únicos
y en el mismo orden del test, ausencia de nulos, clases válidas, todas las clases presentes y sin
duplicados); cualquier fallo detiene la generación. El notebook repite ese protocolo y verifica la
integridad del archivo resultante con las ocho comprobaciones y su huella SHA-256, sin depender de
artefactos externos. La comparación con la **submission sellada del proyecto** (el archivo exacto que
alcanzó 0.63724 en Kaggle) quedó registrada en el control de calidad de la entrega: 112.142 de 112.892
filas idénticas (99,3356%), con las 750 diferencias concentradas en las dos columnas sensibles a zlib y
en la deriva de los componentes neuronales (§6.2).

In [ ]:
# Construccion de la submission final y verificacion de integridad (8 comprobaciones).
submission = pd.DataFrame({"id": test["id"].to_numpy(),
                           "age_range": [CLASSES[i] for i in pred_final]})

comprobaciones = {
    "filas == 112892": len(submission) == 112892,
    "columnas exactas [id, age_range]": list(submission.columns) == ["id", "age_range"],
    "ids unicos": bool(submission["id"].is_unique),
    "ids == test y mismo orden": bool((submission["id"].to_numpy() == test["id"].to_numpy()).all()),
    "sin valores nulos": bool(submission.notna().all().all()),
    "clases validas": bool(submission["age_range"].isin(CLASSES).all()),
    "todas las clases presentes": bool(submission["age_range"].nunique() == 6),
    "sin filas duplicadas": bool(not submission.duplicated().any()),
}
for nombre, resultado in comprobaciones.items():
    print(f"  [{'OK' if resultado else 'FALLA'}] {nombre}")
assert all(comprobaciones.values()), "la submission no paso las comprobaciones"

submission.to_csv("submission.csv", index=False)
if Path("/kaggle/working").exists():
    submission.to_csv("/kaggle/working/submission.csv", index=False)
sha_reconstruida = hashlib.sha256(Path("submission.csv").read_bytes()).hexdigest()
print("\nSHA256 reconstruida :", sha_reconstruida)
print("SHA256 del proyecto :", "bf2554571ca6676195af66a053e28411fc1cfc9e3946fd95bac1cae0ce71df62")

In [ ]:
# Figura 6.1: distribucion final de las decisiones sobre el test.
fig, eje = plt.subplots(figsize=(8.5, 3.2))
conteo_final = pd.Series([CLASSES[i] for i in pred_final]).value_counts().reindex(CLASSES)
eje.bar(conteo_final.index.astype(str), conteo_final.values, color="#2b6cb0")
for posicion, valor in enumerate(conteo_final.values):
    eje.text(posicion, valor + 150, f"{valor:,}", ha="center", fontsize=8)
eje.set_title("Distribucion de la submission final (V48 + plug-in)")
eje.set_ylabel("tuits")
plt.tight_layout()
plt.show()

## 6.2 Reproducibilidad e idempotencia

La reproducibilidad del notebook descansa en cuatro piezas: (1) el **protocolo de folds generado con
la regla documentada** (clústeres de near-duplicados + `StratifiedGroupKFold` con semilla 42; Pedregosa et al., «Scikit-learn: Machine Learning in Python», Journal of Machine Learning Research, 2011),
verificado por propiedades en la propia ejecución —en la versión 1.8.0 reproduce exactamente la
partición canónica del proyecto; en otras versiones la asignación y los valores absolutos pueden
desplazarse—; (2) las **configuraciones congeladas** de todos los
modelos (semillas, hiperparámetros y pesos — nada se busca durante la ejecución); (3) la **cadena
embebida** (ninguna dependencia de scripts o artefactos externos); y (4) el **corpus autónomo** (se
localiza o descarga al inicio). Ejecutar el notebook dos veces produce exactamente los mismos valores.

Tres tolerancias declaradas, todas verificadas sobre la cadena reconstruida (mismo servidor, versiones
exactas). Primera: `compression_ratio` y `compression_gain` — 2 de las 87 columnas estructuradas —
dependen del tamaño de salida de `zlib.compress` (nivel 6) de la librería del sistema; entre builds de
zlib el tamaño comprimido varía en unos bytes y, en tuits cortos, el ratio cambia hasta 0.115 (máximo
medido; media 5.4e-5 sobre las 169.336 filas). Cualquier modelo que use la matriz de 279 columnas
hereda esa sensibilidad: las decisiones de los miembros coinciden entre el 99,96% y el 100%; los
reajustes de árboles (V20, V29, V30 y la segunda partición), entre el 93% y el 97%, porque tanto las
hojas como el punto de parada del early stopping dependen de esas columnas. Segunda: las métricas
agregadas quedan dentro de 1e-3 del valor documentado (en las corridas de verificación, las
comparaciones crudas van de 1e-5 a 8e-4 según el componente, y la regla plug-in —un decisor afinado
sobre las probabilidades originales— es del mismo orden); cada celda imprime el valor exacto de esta
ejecución junto al documentado. Tercera: los componentes neuronales son deterministas en un mismo
entorno, pero entre GPUs de proveedores distintos (AMD ROCm vs NVIDIA CUDA) aparecen diferencias del
orden de 1e-6 en las probabilidades —típicamente por debajo del 0.2% de las decisiones, hasta ~1.2% en
la red densa V53—, que no alteran las métricas ni la práctica totalidad de las decisiones. La idempotencia es exacta: los reajustes de LightGBM fijan el determinismo explícito
(`deterministic` y `force_col_wise`), de modo que dos ejecuciones completas en la misma máquina y
versiones reproducen los mismos valores celda a celda. La comparación con la submission sellada quedó registrada en el control de calidad de la entrega
(99,3356% de filas idénticas); el notebook, para mantenerse autónomo, verifica en cada ejecución el
formato y la huella SHA-256 de su propia reconstrucción.

In [ ]:
# Resumen final de la corrida: versiones, formas y valores clave reconstruidos.
print("=== resumen de la corrida ===")
print("python:", sys.version.split()[0], "| numpy:", np.__version__,
      "| pandas:", pd.__version__, "| scikit-learn:", sklearn.__version__)
print(f"matriz train: {matriz.shape} | matriz test: {matriz_test.shape}")
print(f"V20 (OOF)          : {macro_f1(clase, np.argmax(v20_oof, axis=1)):.7f} (doc 0.6097782)")
print(f"V30 (OOF)          : {cv_v30_oof:.7f} (doc 0.6134471)")
print(f"V30 + plug-in (OOF): {cv_plugin_oof:.10f} (doc 0.6159732487657944)")
print(f"V48 + plug-in (OOF): {cv_v48:.7f} (doc 0.6186451)")
print(f"leaderboard del proyecto: 0.63724 (puesto #1; +0.00196 sobre el segundo lugar al cierre, 0.63528)")